<a href="https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Block 1 — Load 19-column final dataset**

In [1]:
# ============================================================
# ML-05 / W05 — BLOCK 1
# LOAD FINAL 19-COLUMN FEATURE DATASET
# ============================================================

import pandas as pd
import numpy as np

DATA_PATH = "/content/final_features_clean.parquet"

df = pd.read_parquet(DATA_PATH)

print("=" * 70)
print("FINAL FEATURE DATASET")
print("=" * 70)

print("Rows    :", f"{len(df):,}")
print("Columns :", len(df.columns))

print("\nColumns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2}. {col}")

print("\nShape:", df.shape)

FINAL FEATURE DATASET
Rows    : 2,871,202
Columns : 19

Columns:
 1. client_hash_id
 2. content_hash_id
 3. month
 4. gsc_clicks
 5. gsc_impressions
 6. gsc_avg_position
 7. ga4_total_engagement_sec
 8. sessions_organic
 9. sessions_ai
10. missing_count
11. gsc_avg_position_missing
12. ga4_total_engagement_sec_missing
13. sessions_organic_missing
14. sessions_ai_missing
15. ai_other_missing
16. ctr
17. sec_per_click
18. ai_share
19. engagement_per_organic_session

Shape: (2871202, 19)


**Block 2 — Missing indicators + 90-day eligibility**

In [2]:
# ============================================================
# BLOCK 2
# MISSING INDICATORS + BASIC DATA QUALITY
# ============================================================

df["month"] = pd.to_datetime(
    df["month"],
    errors="coerce"
)

# ------------------------------------------------------------
# 1. Remove meaningless missing indicator
# ------------------------------------------------------------

DROP_INDICATOR = "ai_other_missing"

if DROP_INDICATOR in df.columns:
    df = df.drop(columns=[DROP_INDICATOR])

print("=" * 70)
print("MISSING INDICATOR DECISION")
print("=" * 70)

print("Dropped:", DROP_INDICATOR)

remaining_indicators = [
    c for c in df.columns
    if c.endswith("_missing")
]

print("\nMissing indicators kept:")
for c in remaining_indicators:
    print("KEEP ->", c)

# ------------------------------------------------------------
# 2. Sort page-wise chronologically
# ------------------------------------------------------------

df = (
    df
    .sort_values(
        ["content_hash_id", "month"]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Check page history
# ------------------------------------------------------------

page_history = (
    df
    .groupby("content_hash_id")["month"]
    .nunique()
)

print("\n" + "=" * 70)
print("PAGE HISTORY")
print("=" * 70)

print(
    "Total pages:",
    f"{len(page_history):,}"
)

print(
    "Pages with < 3 months:",
    f"{(page_history < 3).sum():,}"
)

print(
    "Pages with >= 3 months:",
    f"{(page_history >= 3).sum():,}"
)

# ------------------------------------------------------------
# 4. Check monthly continuity
# ------------------------------------------------------------

df["previous_month"] = (
    df
    .groupby("content_hash_id")["month"]
    .shift(1)
)

df["month_gap"] = (
    (
        df["month"].dt.year
        - df["previous_month"].dt.year
    ) * 12
    +
    (
        df["month"].dt.month
        - df["previous_month"].dt.month
    )
)

gap_rows = df[
    df["month_gap"].notna()
    &
    (df["month_gap"] > 1)
]

print("\n" + "=" * 70)
print("MONTHLY CONTINUITY")
print("=" * 70)

print(
    "Rows with a month gap:",
    f"{len(gap_rows):,}"
)

print(
    "Pages affected:",
    f"{gap_rows['content_hash_id'].nunique():,}"
)

# helper columns no longer needed
df = df.drop(
    columns=[
        "previous_month",
        "month_gap"
    ]
)

MISSING INDICATOR DECISION
Dropped: ai_other_missing

Missing indicators kept:
KEEP -> gsc_avg_position_missing
KEEP -> ga4_total_engagement_sec_missing
KEEP -> sessions_organic_missing
KEEP -> sessions_ai_missing

PAGE HISTORY
Total pages: 427,292
Pages with < 3 months: 47,141
Pages with >= 3 months: 380,151

MONTHLY CONTINUITY
Rows with a month gap: 13,014
Pages affected: 11,525


**Block 3 — Feature impact + leakage checks**

In [3]:
# ============================================================
# BLOCK 3
# FEATURE AUDIT + LEAKAGE + VIF
# ============================================================

from sklearn.feature_selection import mutual_info_classif
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ------------------------------------------------------------
# 1. ID columns are NOT model features
# ------------------------------------------------------------

ID_COLUMNS = [
    "client_hash_id",
    "content_hash_id"
]

# ------------------------------------------------------------
# 2. Potential future/label-derived names
# ------------------------------------------------------------

future_keywords = [
    "future",
    "target",
    "label",
    "decay",
    "next_",
    "lead_"
]

suspicious_columns = []

for col in df.columns:

    col_lower = col.lower()

    if any(
        keyword in col_lower
        for keyword in future_keywords
    ):
        suspicious_columns.append(col)

print("=" * 70)
print("LEAKAGE NAME CHECK")
print("=" * 70)

if suspicious_columns:
    print("Potentially suspicious columns:")
    for col in suspicious_columns:
        print("CHECK ->", col)
else:
    print("PASS: No obvious future/target-derived columns.")

# ------------------------------------------------------------
# 3. Constant columns
# ------------------------------------------------------------

feature_candidates = [
    c for c in df.columns
    if c not in ID_COLUMNS + ["month"]
]

constant_columns = [
    c for c in feature_candidates
    if df[c].nunique(dropna=False) <= 1
]

print("\n" + "=" * 70)
print("ZERO-VARIANCE CHECK")
print("=" * 70)

if constant_columns:
    for c in constant_columns:
        print("DROP ->", c)
else:
    print("PASS: No zero-variance columns.")

# ------------------------------------------------------------
# 4. Numeric feature list
# ------------------------------------------------------------

numeric_features = [
    c for c in feature_candidates
    if c not in constant_columns
    and pd.api.types.is_numeric_dtype(df[c])
]

print("\nNumeric model candidates:")
for c in numeric_features:
    print(" -", c)

# ------------------------------------------------------------
# 5. Correlation between current features
# ------------------------------------------------------------

corr_matrix = (
    df[numeric_features]
    .corr()
)

print("\n" + "=" * 70)
print("HIGH FEATURE-TO-FEATURE CORRELATION")
print("=" * 70)

high_corr_pairs = []

for i in range(len(numeric_features)):

    for j in range(i + 1, len(numeric_features)):

        a = numeric_features[i]
        b = numeric_features[j]

        corr = corr_matrix.loc[a, b]

        if abs(corr) >= 0.90:

            high_corr_pairs.append(
                (a, b, round(corr, 3))
            )

if high_corr_pairs:

    for pair in high_corr_pairs:
        print(pair)

else:

    print("No feature pairs with |correlation| >= 0.90")

# ------------------------------------------------------------
# 6. VIF sample
# ------------------------------------------------------------
# VIF on millions of rows is unnecessarily expensive.
# We use a reproducible sample for diagnostic purposes.

vif_sample = (
    df[numeric_features]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .sample(
        n=min(100_000, len(df)),
        random_state=42
    )
)

# Remove columns with zero variance inside sample
vif_features = [
    c for c in numeric_features
    if vif_sample[c].nunique() > 1
]

X_vif = vif_sample[vif_features]

vif_table = pd.DataFrame({
    "feature": vif_features,
    "VIF": [
        variance_inflation_factor(
            X_vif.values,
            i
        )
        for i in range(X_vif.shape[1])
    ]
})

vif_table = (
    vif_table
    .sort_values("VIF", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("VIF TEST")
print("=" * 70)

display(vif_table)

LEAKAGE NAME CHECK
PASS: No obvious future/target-derived columns.

ZERO-VARIANCE CHECK
PASS: No zero-variance columns.

Numeric model candidates:
 - gsc_clicks
 - gsc_impressions
 - gsc_avg_position
 - ga4_total_engagement_sec
 - sessions_organic
 - sessions_ai
 - missing_count
 - gsc_avg_position_missing
 - ga4_total_engagement_sec_missing
 - sessions_organic_missing
 - sessions_ai_missing
 - ctr
 - sec_per_click
 - ai_share
 - engagement_per_organic_session

HIGH FEATURE-TO-FEATURE CORRELATION
('gsc_clicks', 'sessions_organic', np.float64(0.978))
('missing_count', 'ga4_total_engagement_sec_missing', np.float64(0.968))
('missing_count', 'sessions_organic_missing', np.float64(0.968))
('missing_count', 'sessions_ai_missing', np.float64(0.968))
('ga4_total_engagement_sec_missing', 'sessions_organic_missing', np.float64(1.0))
('ga4_total_engagement_sec_missing', 'sessions_ai_missing', np.float64(1.0))
('sessions_organic_missing', 'sessions_ai_missing', np.float64(1.0))


/usr/local/lib/python3.12/dist-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)



VIF TEST


,feature,VIF
0,missing_count,inf
1,ga4_total_engagement_sec_missing,inf
2,gsc_avg_position_missing,inf
3,sessions_ai_missing,inf
4,sessions_organic_missing,inf
5,sessions_organic,4.518367
6,gsc_clicks,3.863913
7,gsc_impressions,2.184852
8,ga4_total_engagement_sec,2.176906
9,sessions_ai,1.924955


**Block 3.5: keep selected impactful  features after VIF and correlation test**

In [7]:
# ============================================================
# ML-05 → ML-06/07 PREPARATION
# FINAL FEATURE SET + 90-DAY ELIGIBILITY CHECK
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("FINAL FEATURE DATASET PREPARATION")
print("=" * 70)

# ============================================================
# 1. FIND THE CURRENT 19-COLUMN DATAFRAME
# ============================================================

# Try known dataframe names first
candidate_names = [
    "df_baseline",
    "df_model_base",
    "df_features_clean",
    "df_clean",
    "df"
]

source_df = None
source_name = None

for name in candidate_names:
    if name in globals():
        obj = globals()[name]

        if isinstance(obj, pd.DataFrame):
            source_df = obj.copy()
            source_name = name
            break

if source_df is None:
    raise NameError(
        "No suitable dataframe found. "
        "Please load your 19-column Parquet dataset first."
    )

print(f"Source dataframe used: {source_name}")
print(f"Rows before cleaning: {len(source_df):,}")
print(f"Columns before cleaning: {source_df.shape[1]}")

# ============================================================
# 2. START FROM SOURCE DATA
# ============================================================

df_model_base = source_df.copy()

# Convert month safely
df_model_base["month"] = pd.to_datetime(
    df_model_base["month"],
    errors="coerce"
)

# ============================================================
# 3. REMOVE REDUNDANT MISSINGNESS FEATURES
# ============================================================

remove_columns = [
    "missing_count",
    "ga4_total_engagement_sec_missing",
    "sessions_organic_missing",
    "sessions_ai_missing"
]

# Only remove columns that actually exist
remove_columns = [
    col
    for col in remove_columns
    if col in df_model_base.columns
]

df_model_base = df_model_base.drop(
    columns=remove_columns
)

print("\n" + "=" * 70)
print("REMOVED REDUNDANT FEATURES")
print("=" * 70)

if remove_columns:
    for col in remove_columns:
        print(" -", col)
else:
    print("None")

# ============================================================
# 4. REQUIRED COLUMN CHECK
# ============================================================

required_columns = [
    "content_hash_id",
    "month"
]

missing_required = [
    col
    for col in required_columns
    if col not in df_model_base.columns
]

if missing_required:
    raise ValueError(
        f"Required columns missing: {missing_required}"
    )

# ============================================================
# 5. REMOVE INVALID MONTH ROWS
# ============================================================

before_month_filter = len(df_model_base)

df_model_base = df_model_base[
    df_model_base["month"].notna()
].copy()

removed_invalid_months = (
    before_month_filter -
    len(df_model_base)
)

print(
    f"\nRows removed because month was invalid: "
    f"{removed_invalid_months:,}"
)

# ============================================================
# 6. REMOVE DUPLICATE PAGE-MONTH RECORDS IF ANY
# ============================================================

duplicate_count = (
    df_model_base
    .duplicated(
        subset=["content_hash_id", "month"]
    )
    .sum()
)

print(
    f"Duplicate page-month rows found: "
    f"{duplicate_count:,}"
)

if duplicate_count > 0:
    df_model_base = (
        df_model_base
        .drop_duplicates(
            subset=["content_hash_id", "month"],
            keep="first"
        )
        .copy()
    )

# ============================================================
# 7. SORT PAGE HISTORIES CHRONOLOGICALLY
# ============================================================

df_model_base = (
    df_model_base
    .sort_values(
        ["content_hash_id", "month"]
    )
    .reset_index(drop=True)
)

# ============================================================
# 8. CALCULATE OBSERVED HISTORY
# ============================================================

first_month = (
    df_model_base
    .groupby("content_hash_id")["month"]
    .transform("min")
)

last_month = (
    df_model_base
    .groupby("content_hash_id")["month"]
    .transform("max")
)

df_model_base["content_age_days"] = (
    last_month - first_month
).dt.days

# ============================================================
# 9. 90-DAY ELIGIBILITY
# ============================================================

eligible_mask = (
    df_model_base["content_age_days"] >= 90
)

df_model_eligible = (
    df_model_base.loc[eligible_mask]
    .copy()
)

removed_rows = (
    len(df_model_base) -
    len(df_model_eligible)
)

print("\n" + "=" * 70)
print("90-DAY ELIGIBILITY")
print("=" * 70)

print(
    f"Rows before 90-day filter : "
    f"{len(df_model_base):,}"
)

print(
    f"Rows after 90-day filter  : "
    f"{len(df_model_eligible):,}"
)

print(
    f"Rows removed              : "
    f"{removed_rows:,}"
)

# ============================================================
# 10. PAGE-LEVEL ELIGIBILITY CHECK
# ============================================================

page_age = (
    df_model_eligible
    .groupby("content_hash_id")["content_age_days"]
    .max()
)

if len(page_age) > 0:

    print(
        f"\nEligible pages: "
        f"{len(page_age):,}"
    )

    print(
        f"Minimum eligible history: "
        f"{page_age.min()} days"
    )

    print(
        f"Pages with <90 days remaining: "
        f"{(page_age < 90).sum():,}"
    )

    # Safety check
    assert (
        page_age >= 90
    ).all(), (
        "ERROR: A page below 90 days remains."
    )

else:
    raise ValueError(
        "No pages remain after the 90-day eligibility filter."
    )

# ============================================================
# 11. FINAL CHRONOLOGICAL ORDER
# ============================================================

df_model_eligible = (
    df_model_eligible
    .sort_values(
        ["content_hash_id", "month"]
    )
    .reset_index(drop=True)
)

# ============================================================
# 12. FINAL DATASET INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATASET")
print("=" * 70)

print(
    f"Final rows    : "
    f"{len(df_model_eligible):,}"
)

print(
    f"Final columns : "
    f"{df_model_eligible.shape[1]}"
)

print(
    f"Final shape   : "
    f"{df_model_eligible.shape}"
)

print("\nRemaining columns:")

for i, col in enumerate(
    df_model_eligible.columns,
    start=1
):
    print(f"{i:2}. {col}")

# ============================================================
# 13. FINAL 5 ROWS
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATASET — FIRST 5 ROWS")
print("=" * 70)

display(
    df_model_eligible.head(5)
)

# ============================================================
# 14. FINAL 90-DAY SAFETY CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL SAFETY CHECK")
print("=" * 70)

print(
    "Minimum content_age_days:",
    df_model_eligible["content_age_days"].min()
)

print(
    "Rows below 90 days:",
    (
        df_model_eligible["content_age_days"] < 90
    ).sum()
)

assert (
    df_model_eligible["content_age_days"] >= 90
).all()

print("PASS: No page below the 90-day requirement.")

# ============================================================
# 15. SAVE FINAL INTERMEDIATE DATASET
# ============================================================

output_path = (
    "/content/final_model_eligible.parquet"
)

df_model_eligible.to_parquet(
    output_path,
    index=False
)

print("\n" + "=" * 70)
print("PARQUET SAVED")
print("=" * 70)

print(output_path)

print("\n" + "=" * 70)
print("PREPARATION COMPLETE")
print("=" * 70)

FINAL FEATURE DATASET PREPARATION
Source dataframe used: df_model_base
Rows before cleaning: 2,871,202
Columns before cleaning: 15

REMOVED REDUNDANT FEATURES
None

Rows removed because month was invalid: 0
Duplicate page-month rows found: 0

90-DAY ELIGIBILITY
Rows before 90-day filter : 2,871,202
Rows after 90-day filter  : 2,705,303
Rows removed              : 165,899

Eligible pages: 349,557
Minimum eligible history: 92 days
Pages with <90 days remaining: 0

FINAL DATASET
Final rows    : 2,705,303
Final columns : 15
Final shape   : (2705303, 15)

Remaining columns:
 1. client_hash_id
 2. content_hash_id
 3. month
 4. gsc_clicks
 5. gsc_impressions
 6. gsc_avg_position
 7. ga4_total_engagement_sec
 8. sessions_organic
 9. sessions_ai
10. gsc_avg_position_missing
11. ctr
12. sec_per_click
13. ai_share
14. engagement_per_organic_session
15. content_age_days

FINAL DATASET — FIRST 5 ROWS


,client_hash_id,content_hash_id,month,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_total_engagement_sec,sessions_organic,sessions_ai,gsc_avg_position_missing,ctr,sec_per_click,ai_share,engagement_per_organic_session,content_age_days
0,client_9958f0a7ae1df715,content_000005d4ced12088,2025-03-01,0.0,7.0,9.333333,0.0,0.0,0.0,0,0.000000,0.0,0.0,0.0,457
1,client_9958f0a7ae1df715,content_000005d4ced12088,2025-04-01,1.0,146.0,35.762918,0.0,0.0,0.0,0,0.006849,0.0,0.0,0.0,457
2,client_9958f0a7ae1df715,content_000005d4ced12088,2025-05-01,0.0,257.0,38.982641,0.0,0.0,0.0,0,0.000000,0.0,0.0,0.0,457
3,client_9958f0a7ae1df715,content_000005d4ced12088,2025-06-01,0.0,139.0,37.522978,0.0,0.0,0.0,0,0.000000,0.0,0.0,0.0,457
4,client_9958f0a7ae1df715,content_000005d4ced12088,2025-07-01,0.0,254.0,35.843110,0.0,0.0,0.0,0,0.000000,0.0,0.0,0.0,457



FINAL SAFETY CHECK
Minimum content_age_days: 92
Rows below 90 days: 0
PASS: No page below the 90-day requirement.

PARQUET SAVED
/content/final_model_eligible.parquet

PREPARATION COMPLETE


**BLOCK 4 — ROLLING 90-DAY FEATURES + 30-DAY BASELINE SIGNAL**

In [8]:
# ============================================================
# BLOCK 4 — ROLLING 90-DAY FEATURES
#
# CURRENT 3 MONTHS = MODEL INPUT HISTORY
# NEXT 3 MONTHS    = TARGET GENERATION DATA ONLY
#
# ALSO PRESERVED:
#   gsc_impressions_prev_30d
#   gsc_impressions_last_30d
#
# These two columns are required later for the
# Early Drop Baseline comparison.
#
# IMPORTANT:
# NO target / target_label is created in this block.
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("BLOCK 4 — ROLLING 90-DAY FEATURES + EARLY DROP SIGNAL")
print("=" * 80)

# ============================================================
# 1. SOURCE DATA
# ============================================================

if "df_model_eligible" not in globals():
    raise NameError(
        "df_model_eligible not found. "
        "Run the 90-day eligibility preparation block first."
    )

df_roll = df_model_eligible.copy()

print(f"Source rows : {len(df_roll):,}")
print(f"Source cols : {df_roll.shape[1]}")

# ============================================================
# 2. DATE CLEANING
# ============================================================

df_roll["month"] = pd.to_datetime(
    df_roll["month"],
    errors="coerce"
)

df_roll = df_roll.dropna(
    subset=["content_hash_id", "month"]
).copy()

# ============================================================
# 3. DUPLICATE PAGE-MONTH CLEANUP
# ============================================================

before_dup = len(df_roll)

df_roll = (
    df_roll
    .drop_duplicates(
        subset=["content_hash_id", "month"],
        keep="first"
    )
    .sort_values(
        ["content_hash_id", "month"]
    )
    .reset_index(drop=True)
)

print(
    f"Duplicate rows removed: "
    f"{before_dup - len(df_roll):,}"
)

# ============================================================
# 4. MONTH PERIOD
# ============================================================

df_roll["month_period"] = (
    df_roll["month"].dt.to_period("M")
)

# ============================================================
# 5. REQUIRED RAW FEATURES
# ============================================================

required_features = [
    "gsc_clicks",
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_ai",
    "gsc_avg_position_missing",
    "ctr",
    "sec_per_click",
    "ai_share",
    "engagement_per_organic_session"
]

missing_features = [
    col
    for col in required_features
    if col not in df_roll.columns
]

if missing_features:
    raise KeyError(
        "Required raw feature columns are missing:\n"
        + "\n".join(missing_features)
    )

# ============================================================
# 6. VERIFY 30-DAY BASELINE SOURCE
#
# We need monthly impressions.
#
# For each rolling row:
#
# prev_30d = first month of current 3-month window
# last_30d = latest month of current 3-month window
#
# This keeps the Early Drop signal aligned with
# the same rolling-window observation.
# ============================================================

print("\n" + "=" * 80)
print("30-DAY EARLY DROP BASELINE")
print("=" * 80)

print(
    "Baseline definition:"
)
print(
    "gsc_impressions_last_30d < gsc_impressions_prev_30d"
)

# ============================================================
# 7. CONSECUTIVE 6-MONTH WINDOWS
#
# Current:
#   i     i+1     i+2
#
# Future:
#   i+3   i+4     i+5
#
# All six months must be consecutive.
# ============================================================

page = df_roll["content_hash_id"]
period = df_roll["month_period"]

valid_6m = (
    page.eq(page.shift(-1))
    & page.eq(page.shift(-2))
    & page.eq(page.shift(-3))
    & page.eq(page.shift(-4))
    & page.eq(page.shift(-5))

    & period.add(1).eq(period.shift(-1))
    & period.add(2).eq(period.shift(-2))
    & period.add(3).eq(period.shift(-3))
    & period.add(4).eq(period.shift(-4))
    & period.add(5).eq(period.shift(-5))
)

valid_idx = np.flatnonzero(
    valid_6m.to_numpy()
)

print(
    f"Valid 6-month windows: "
    f"{len(valid_idx):,}"
)

if len(valid_idx) == 0:
    raise ValueError(
        "No valid 6-month rolling windows found."
    )

# ============================================================
# 8. OUTPUT DATAFRAME
# ============================================================

out = pd.DataFrame(
    index=np.arange(len(valid_idx))
)

# ============================================================
# 9. METADATA
# ============================================================

out["content_hash_id"] = (
    df_roll[
        "content_hash_id"
    ]
    .iloc[valid_idx]
    .to_numpy()
)

if "client_hash_id" in df_roll.columns:

    out["client_hash_id"] = (
        df_roll[
            "client_hash_id"
        ]
        .iloc[valid_idx]
        .to_numpy()
    )

# Current 90-day window
out["window_start"] = (
    df_roll[
        "month"
    ]
    .iloc[valid_idx]
    .to_numpy()
)

out["window_end"] = (
    df_roll[
        "month"
    ]
    .iloc[valid_idx + 2]
    .to_numpy()
)

# Future 90-day window
out["future_start"] = (
    df_roll[
        "month"
    ]
    .iloc[valid_idx + 3]
    .to_numpy()
)

out["future_end"] = (
    df_roll[
        "month"
    ]
    .iloc[valid_idx + 5]
    .to_numpy()
)

# ============================================================
# 10. CURRENT 3-MONTH MODEL FEATURES
# ============================================================

print("\n" + "=" * 80)
print("CREATING CURRENT 90-DAY FEATURES")
print("=" * 80)

for col in required_features:

    values = pd.to_numeric(
        df_roll[col],
        errors="coerce"
    ).to_numpy()

    v0 = values[valid_idx]
    v1 = values[valid_idx + 1]
    v2 = values[valid_idx + 2]

    # 3-month mean
    out[
        f"{col}_mean_3m"
    ] = (
        v0 + v1 + v2
    ) / 3

    # Latest/current month
    out[
        f"{col}_last"
    ] = v2

# ============================================================
# 11. CURRENT 90-DAY IMPRESSIONS
# ============================================================

imp = pd.to_numeric(
    df_roll["gsc_impressions"],
    errors="coerce"
).to_numpy()

current_imp_3m = (
    imp[valid_idx]
    + imp[valid_idx + 1]
    + imp[valid_idx + 2]
) / 3

out["current_imp_3m"] = (
    current_imp_3m
)

# ============================================================
# 12. EARLY DROP BASELINE COLUMNS
#
# IMPORTANT:
#
# prev_30d = first month of current 90-day window
# last_30d = last month of current 90-day window
#
# This reproduces the intended:
#
# last_30d < prev_30d
# ============================================================

out["gsc_impressions_prev_30d"] = (
    imp[valid_idx]
)

out["gsc_impressions_last_30d"] = (
    imp[valid_idx + 2]
)

# ============================================================
# 13. EARLY DROP SIGNAL
# ============================================================

prev_30d = pd.to_numeric(
    out[
        "gsc_impressions_prev_30d"
    ],
    errors="coerce"
)

last_30d = pd.to_numeric(
    out[
        "gsc_impressions_last_30d"
    ],
    errors="coerce"
)

out["early_drop_signal"] = (
    last_30d < prev_30d
)

# ============================================================
# 14. FUTURE IMPRESSIONS
#
# Target-generation information only.
# ============================================================

future_imp_3m = (
    imp[valid_idx + 3]
    + imp[valid_idx + 4]
    + imp[valid_idx + 5]
) / 3

out["future_imp_3m"] = (
    future_imp_3m
)

# ============================================================
# 15. FUTURE IMPRESSION CHANGE %
# ============================================================

valid_change = (
    np.isfinite(current_imp_3m)
    & np.isfinite(future_imp_3m)
    & (current_imp_3m > 0)
)

out["future_impression_change_pct"] = (
    np.nan
)

out.loc[
    valid_change,
    "future_impression_change_pct"
] = (
    (
        future_imp_3m[valid_change]
        - current_imp_3m[valid_change]
    )
    / current_imp_3m[valid_change]
) * 100

# ============================================================
# 16. REMOVE INVALID FUTURE CHANGE ROWS
# ============================================================

before_target_validity = len(out)

out = out[
    out[
        "future_impression_change_pct"
    ].notna()
].copy()

print(
    f"Rows removed due to invalid "
    f"future change: "
    f"{before_target_validity - len(out):,}"
)

# ============================================================
# 17. SAFETY CHECK
# ============================================================

assert (
    out[
        "future_impression_change_pct"
    ].notna().all()
)

assert (
    out[
        "gsc_impressions_prev_30d"
    ].notna().all()
)

assert (
    out[
        "gsc_impressions_last_30d"
    ].notna().all()
)

# ============================================================
# 18. COLUMN SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("ROLLING WINDOW DATASET")
print("=" * 80)

print(
    f"Rows    : {len(out):,}"
)

print(
    f"Columns : {out.shape[1]}"
)

print("\nColumns:")

for i, col in enumerate(
    out.columns,
    start=1
):
    print(
        f"{i:2}. {col}"
    )

# ============================================================
# 19. EARLY DROP BASELINE SUMMARY
# ============================================================

early_drop_count = int(
    out["early_drop_signal"].sum()
)

early_drop_pct = (
    early_drop_count
    / len(out)
    * 100
)

print("\n" + "=" * 80)
print("EARLY DROP BASELINE SIGNAL")
print("=" * 80)

print(
    f"Early Drop pages : "
    f"{early_drop_count:,}"
)

print(
    f"Early Drop rate  : "
    f"{early_drop_pct:.2f}%"
)

print(
    "\nRule:"
)

print(
    "gsc_impressions_last_30d "
    "< gsc_impressions_prev_30d"
)

# ============================================================
# 20. PREVIEW
# ============================================================

print("\n" + "=" * 80)
print("ROLLING 90-DAY DATASET PREVIEW")
print("=" * 80)

display(
    out.head(10)
)

# ============================================================
# 21. SAVE BLOCK 4 DATASET
# ============================================================

rolling90_features_path = (
    "/content/rolling90_features.parquet"
)

out.to_parquet(
    rolling90_features_path,
    index=False
)

print("\n" + "=" * 80)
print("BLOCK 4 SAVED")
print("=" * 80)

print(
    rolling90_features_path
)

print("\n✓ Current 90-day features created.")
print("✓ Future 90-day information retained for target generation.")
print("✓ Early Drop baseline columns retained.")
print("✓ No target labels created yet.")

BLOCK 4 — ROLLING 90-DAY FEATURES + EARLY DROP SIGNAL
Source rows : 2,705,303
Source cols : 15
Duplicate rows removed: 0

30-DAY EARLY DROP BASELINE
Baseline definition:
gsc_impressions_last_30d < gsc_impressions_prev_30d
Valid 6-month windows: 978,801

CREATING CURRENT 90-DAY FEATURES
Rows removed due to invalid future change: 351,965

ROLLING WINDOW DATASET
Rows    : 626,836
Columns : 34

Columns:
 1. content_hash_id
 2. client_hash_id
 3. window_start
 4. window_end
 5. future_start
 6. future_end
 7. gsc_clicks_mean_3m
 8. gsc_clicks_last
 9. gsc_impressions_mean_3m
10. gsc_impressions_last
11. gsc_avg_position_mean_3m
12. gsc_avg_position_last
13. ga4_total_engagement_sec_mean_3m
14. ga4_total_engagement_sec_last
15. sessions_organic_mean_3m
16. sessions_organic_last
17. sessions_ai_mean_3m
18. sessions_ai_last
19. gsc_avg_position_missing_mean_3m
20. gsc_avg_position_missing_last
21. ctr_mean_3m
22. ctr_last
23. sec_per_click_mean_3m
24. sec_per_click_last
25. ai_share_mean_3m
26

,content_hash_id,client_hash_id,window_start,window_end,future_start,future_end,gsc_clicks_mean_3m,gsc_clicks_last,gsc_impressions_mean_3m,gsc_impressions_last,...,ai_share_mean_3m,ai_share_last,engagement_per_organic_session_mean_3m,engagement_per_organic_session_last,current_imp_3m,gsc_impressions_prev_30d,gsc_impressions_last_30d,early_drop_signal,future_imp_3m,future_impression_change_pct
0,content_000005d4ced12088,client_9958f0a7ae1df715,2025-03-01,2025-05-01,2025-06-01,2025-08-01,0.333333,0.0,136.666667,257.0,...,0.0,0.0,0.0,0.0,136.666667,7.0,257.0,False,378.333333,176.829268
1,content_000005d4ced12088,client_9958f0a7ae1df715,2025-04-01,2025-06-01,2025-07-01,2025-09-01,0.333333,0.0,180.666667,139.0,...,0.0,0.0,0.0,0.0,180.666667,146.0,139.0,True,506.666667,180.442804
2,content_000005d4ced12088,client_9958f0a7ae1df715,2025-05-01,2025-07-01,2025-08-01,2025-10-01,0.000000,0.0,216.666667,254.0,...,0.0,0.0,0.0,0.0,216.666667,257.0,254.0,True,487.666667,125.076923
3,content_000005d4ced12088,client_9958f0a7ae1df715,2025-06-01,2025-08-01,2025-09-01,2025-11-01,0.333333,1.0,378.333333,742.0,...,0.0,0.0,0.0,0.0,378.333333,139.0,742.0,False,280.333333,-25.903084
4,content_000005d4ced12088,client_9958f0a7ae1df715,2025-07-01,2025-09-01,2025-10-01,2025-12-01,0.666667,1.0,506.666667,524.0,...,0.0,0.0,0.0,0.0,506.666667,254.0,524.0,False,167.000000,-67.039474
5,content_000005d4ced12088,client_9958f0a7ae1df715,2025-08-01,2025-10-01,2025-11-01,2026-01-01,0.666667,0.0,487.666667,197.0,...,0.0,0.0,0.0,0.0,487.666667,742.0,197.0,True,106.333333,-78.195489
6,content_000005d4ced12088,client_9958f0a7ae1df715,2025-09-01,2025-11-01,2025-12-01,2026-02-01,0.333333,0.0,280.333333,120.0,...,0.0,0.0,0.0,0.0,280.333333,524.0,120.0,True,74.333333,-73.483948
7,content_000005d4ced12088,client_9958f0a7ae1df715,2025-10-01,2025-12-01,2026-01-01,2026-03-01,0.000000,0.0,167.000000,184.0,...,0.0,0.0,0.0,0.0,167.000000,197.0,184.0,True,41.666667,-75.049900
8,content_000005d4ced12088,client_9958f0a7ae1df715,2025-11-01,2026-01-01,2026-02-01,2026-04-01,0.000000,0.0,106.333333,15.0,...,0.0,0.0,0.0,0.0,106.333333,120.0,15.0,True,63.666667,-40.125392
9,content_000005d4ced12088,client_9958f0a7ae1df715,2025-12-01,2026-02-01,2026-03-01,2026-05-01,0.000000,0.0,74.333333,24.0,...,0.0,0.0,0.0,0.0,74.333333,184.0,24.0,True,82.666667,11.210762



BLOCK 4 SAVED
/content/rolling90_features.parquet

✓ Current 90-day features created.
✓ Future 90-day information retained for target generation.
✓ Early Drop baseline columns retained.
✓ No target labels created yet.


**BLOCK 4.5 — TARGET THRESHOLD VALIDATION**

Where Target does not apply only see future_impression_change_pct and compare candidate thresholds  


In [9]:
# ============================================================
# BLOCK 4.5 — TARGET THRESHOLD VALIDATION
#
# NO TARGET LABEL IS CREATED HERE.
#
# Purpose:
# Select a meaningful threshold ONLY on the
# rolling-window observations.
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("BLOCK 4.5 — TARGET THRESHOLD VALIDATION")
print("=" * 80)

if "out" not in globals():
    raise NameError(
        "Rolling-window dataframe 'out' not found. "
        "Run Block 4 first."
    )

df_threshold = out.copy()

change = pd.to_numeric(
    df_threshold[
        "future_impression_change_pct"
    ],
    errors="coerce"
).dropna()

print(
    f"\nValid rolling windows: "
    f"{len(change):,}"
)

# ============================================================
# 1. DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("FUTURE IMPRESSION CHANGE DISTRIBUTION")
print("=" * 80)

distribution = pd.DataFrame({
    "statistic": [
        "Minimum",
        "5th percentile",
        "10th percentile",
        "25th percentile",
        "Median",
        "75th percentile",
        "90th percentile",
        "95th percentile",
        "Maximum"
    ],
    "change_pct": [
        change.min(),
        change.quantile(.05),
        change.quantile(.10),
        change.quantile(.25),
        change.median(),
        change.quantile(.75),
        change.quantile(.90),
        change.quantile(.95),
        change.max()
    ]
})

distribution["change_pct"] = (
    distribution["change_pct"]
    .round(2)
)

display(distribution)

# ============================================================
# 2. CANDIDATE THRESHOLDS
# ============================================================

symmetric_rules = [
    (-10, 10),
    (-15, 15),
    (-20, 20),
    (-25, 25),
    (-30, 30),
    (-40, 40),
    (-50, 50)
]

asymmetric_rules = [
    (-20, 50),
    (-25, 50),
    (-30, 50),
    (-40, 50),
    (-50, 50),

    (-30, 60),
    (-40, 60),
    (-50, 60),

    (-30, 75),
    (-40, 75),
    (-50, 75),

    (-50, 100)
]

all_rules = (
    symmetric_rules
    + asymmetric_rules
)

# Remove duplicate rules
all_rules = list(
    dict.fromkeys(all_rules)
)

# ============================================================
# 3. BALANCE SCORE
#
# Higher score = more balanced classes.
#
# This is ONLY a screening metric.
# We do NOT automatically select the highest score.
# ============================================================

results = []

for down_threshold, up_threshold in all_rules:

    down_mask = (
        change <= down_threshold
    )

    flat_mask = (
        (change > down_threshold)
        & (change < up_threshold)
    )

    up_mask = (
        change >= up_threshold
    )

    down_pct = (
        down_mask.mean() * 100
    )

    flat_pct = (
        flat_mask.mean() * 100
    )

    up_pct = (
        up_mask.mean() * 100
    )

    # Simple balance score:
    # maximum possible when all three are ~33.33%
    balance_score = (
        100
        - (
            abs(down_pct - 33.33)
            + abs(flat_pct - 33.33)
            + abs(up_pct - 33.33)
        )
    )

    results.append({
        "rule": (
            f"{down_threshold}% / "
            f"+{up_threshold}%"
        ),
        "down_pct": round(down_pct, 2),
        "flat_pct": round(flat_pct, 2),
        "up_pct": round(up_pct, 2),
        "balance_score": round(
            balance_score,
            2
        )
    })

threshold_results = pd.DataFrame(
    results
)

# ============================================================
# 4. SYMMETRIC RULES
# ============================================================

print("\n" + "=" * 80)
print("SYMMETRIC THRESHOLD COMPARISON")
print("=" * 80)

symmetric_labels = [
    f"±{x}%"
    for x in [10, 15, 20, 25, 30, 40, 50]
]

symmetric_table = threshold_results[
    threshold_results["rule"].isin(
        symmetric_labels
    )
].copy()

display(
    symmetric_table.reset_index(drop=True)
)

# ============================================================
# 5. ASYMMETRIC RULES
# ============================================================

print("\n" + "=" * 80)
print("ASYMMETRIC THRESHOLD COMPARISON")
print("=" * 80)

asymmetric_table = threshold_results[
    ~threshold_results["rule"].isin(
        symmetric_labels
    )
].copy()

display(
    asymmetric_table
    .sort_values(
        "balance_score",
        ascending=False
    )
    .reset_index(drop=True)
)

# ============================================================
# 6. PROJECT WORKING RULE
#
# Based on your validated rule:
#
# <= -30%  = DOWN
# > -30% and < +50% = FLAT
# >= +50% = UP
# ============================================================

recommended_down = -30
recommended_up = 50

recommended_mask = (
    change <= recommended_down
)

recommended_flat = (
    (change > recommended_down)
    & (change < recommended_up)
)

recommended_up_mask = (
    change >= recommended_up
)

recommended_result = pd.DataFrame([{
    "rule": "PROJECT RULE: -30% / +50%",
    "down_pct": round(
        recommended_mask.mean() * 100,
        2
    ),
    "flat_pct": round(
        recommended_flat.mean() * 100,
        2
    ),
    "up_pct": round(
        recommended_up_mask.mean() * 100,
        2
    )
}])

print("\n" + "=" * 80)
print("PROJECT TARGETING RULE")
print("=" * 80)

display(
    recommended_result
)

print(
    "\nRecommended working rule:"
)

print(
    "DOWN : change <= -30%"
)

print(
    "FLAT : -30% < change < +50%"
)

print(
    "UP   : change >= +50%"
)

# ============================================================
# 7. TARGET ENCODING VERIFICATION
# ============================================================

print("\n" + "=" * 80)
print("TARGET ENCODING PLAN")
print("=" * 80)

print("0 = DOWN")
print("1 = FLAT")
print("2 = UP")

print(
    "\n✓ Threshold validation complete."
)

print(
    "✓ No target column has been created."
)

print(
    "✓ The rolling-window dataframe remains unchanged."
)

BLOCK 4.5 — TARGET THRESHOLD VALIDATION

Valid rolling windows: 626,836

FUTURE IMPRESSION CHANGE DISTRIBUTION


,statistic,change_pct
0,Minimum,-100.00
1,5th percentile,-100.00
2,10th percentile,-90.00
3,25th percentile,-50.16
4,Median,19.57
5,75th percentile,150.00
6,90th percentile,442.13
7,95th percentile,909.09
8,Maximum,2306500.00



SYMMETRIC THRESHOLD COMPARISON


,rule,down_pct,flat_pct,up_pct,balance_score



ASYMMETRIC THRESHOLD COMPARISON


,rule,down_pct,flat_pct,up_pct,balance_score
0,-30% / +75%,32.93,30.48,36.59,93.48
1,-40% / +75%,29.37,34.03,36.59,92.07
2,-40% / +60%,29.37,30.96,39.66,87.34
3,-30% / +60%,32.93,27.41,39.66,87.34
4,-50% / +60%,25.98,34.35,39.66,85.30
5,-50% / +75%,25.98,37.42,36.59,85.30
6,-50% / +100%,25.98,41.66,32.36,83.35
7,-40% / +50%,29.37,28.55,42.08,82.50
8,-50% / +50%,25.98,31.93,42.08,82.50
9,-30% / +50%,32.93,24.99,42.08,82.50



PROJECT TARGETING RULE


,rule,down_pct,flat_pct,up_pct
0,PROJECT RULE: -30% / +50%,32.93,24.99,42.08



Recommended working rule:
DOWN : change <= -30%
FLAT : -30% < change < +50%
UP   : change >= +50%

TARGET ENCODING PLAN
0 = DOWN
1 = FLAT
2 = UP

✓ Threshold validation complete.
✓ No target column has been created.
✓ The rolling-window dataframe remains unchanged.


**BLOCK 4.6 — APPLY FROZEN TARGET + SAVE FINAL ROLLING90WINDOW**

In [10]:
# ============================================================
# BLOCK 4.6 — APPLY FROZEN TARGET
#
# FINAL ROLLING 90-DAY DATASET
#
# Target:
#   0 = DOWN
#   1 = FLAT
#   2 = UP
#
# Frozen thresholds:
#   <= -30%          -> DOWN
#   > -30% & < +50% -> FLAT
#   >= +50%         -> UP
#
# IMPORTANT:
# All metadata
# All current-window features
# All future target-generation columns
# Target
# Target label
# are retained.
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("BLOCK 4.6 — FINAL ROLLING 90-DAY DATASET")
print("=" * 80)

if "out" not in globals():
    raise NameError(
        "Rolling-window dataframe 'out' not found. "
        "Run Block 4 first."
    )

finalrolling90 = out.copy()

# ============================================================
# 1. FROZEN TARGET THRESHOLDS
# ============================================================

DOWN_THRESHOLD = -30
UP_THRESHOLD = 50

print("\n" + "=" * 80)
print("FROZEN TARGET RULE")
print("=" * 80)

print(
    f"DOWN : change <= {DOWN_THRESHOLD}%"
)

print(
    f"FLAT : {DOWN_THRESHOLD}% < change < "
    f"+{UP_THRESHOLD}%"
)

print(
    f"UP   : change >= +{UP_THRESHOLD}%"
)

# ============================================================
# 2. CREATE NUMERIC TARGET
# ============================================================

change = pd.to_numeric(
    finalrolling90[
        "future_impression_change_pct"
    ],
    errors="coerce"
)

if change.isna().any():
    raise ValueError(
        "NaN values found in future_impression_change_pct."
    )

finalrolling90["target"] = np.select(
    [
        change <= DOWN_THRESHOLD,
        change >= UP_THRESHOLD
    ],
    [
        0,
        2
    ],
    default=1
).astype("int8")

# ============================================================
# 3. CREATE HUMAN-READABLE TARGET LABEL
# ============================================================

label_map = {
    0: "DOWN",
    1: "FLAT",
    2: "UP"
}

finalrolling90["target_label"] = (
    finalrolling90["target"]
    .map(label_map)
)

# ============================================================
# 4. TARGET ENCODING SAFETY CHECK
# ============================================================

print("\n" + "=" * 80)
print("TARGET ENCODING")
print("=" * 80)

print("0 = DOWN")
print("1 = FLAT")
print("2 = UP")

assert set(
    finalrolling90["target"].unique()
).issubset({0, 1, 2})

assert (
    finalrolling90["target_label"]
    .notna()
    .all()
)

assert (
    finalrolling90.loc[
        finalrolling90["target"] == 0,
        "target_label"
    ].eq("DOWN").all()
)

assert (
    finalrolling90.loc[
        finalrolling90["target"] == 1,
        "target_label"
    ].eq("FLAT").all()
)

assert (
    finalrolling90.loc[
        finalrolling90["target"] == 2,
        "target_label"
    ].eq("UP").all()
)

print("✓ Encoding confirmed.")

# ============================================================
# 5. BOUNDARY SANITY CHECK
# ============================================================

down_boundary_ok = (
    finalrolling90.loc[
        finalrolling90["target"] == 0,
        "future_impression_change_pct"
    ] <= -30
).all()

flat_values = finalrolling90.loc[
    finalrolling90["target"] == 1,
    "future_impression_change_pct"
]

flat_boundary_ok = (
    (flat_values > -30)
    & (flat_values < 50)
).all()

up_boundary_ok = (
    finalrolling90.loc[
        finalrolling90["target"] == 2,
        "future_impression_change_pct"
    ] >= 50
).all()

assert down_boundary_ok
assert flat_boundary_ok
assert up_boundary_ok

print("\n" + "=" * 80)
print("BOUNDARY SANITY CHECK")
print("=" * 80)

print(
    "✓ DOWN contains only change <= -30%"
)

print(
    "✓ FLAT contains only -30% < change < +50%"
)

print(
    "✓ UP contains only change >= +50%"
)

# ============================================================
# 6. FINAL TARGET DISTRIBUTION
# ============================================================

target_summary = (
    finalrolling90
    .groupby(
        ["target", "target_label"],
        sort=True
    )
    .size()
    .reset_index(
        name="count"
    )
)

target_summary["percentage"] = (
    target_summary["count"]
    / len(finalrolling90)
    * 100
).round(2)

print("\n" + "=" * 80)
print("FINAL TARGET DISTRIBUTION")
print("=" * 80)

display(
    target_summary
)

# ============================================================
# 7. EARLY DROP BASELINE SUMMARY
# ============================================================

early_drop_count = int(
    finalrolling90[
        "early_drop_signal"
    ].sum()
)

early_drop_pct = (
    early_drop_count
    / len(finalrolling90)
    * 100
)

print("\n" + "=" * 80)
print("EARLY DROP BASELINE")
print("=" * 80)

print(
    f"Early Drop signals : "
    f"{early_drop_count:,}"
)

print(
    f"Early Drop rate    : "
    f"{early_drop_pct:.2f}%"
)

print(
    "\nRule:"
)

print(
    "gsc_impressions_last_30d "
    "< gsc_impressions_prev_30d"
)

# ============================================================
# 8. FINAL COLUMN INVENTORY
# ============================================================

print("\n" + "=" * 80)
print("FINAL ROLLING90WINDOW DATASET")
print("=" * 80)

print(
    f"Rows    : "
    f"{len(finalrolling90):,}"
)

print(
    f"Columns : "
    f"{finalrolling90.shape[1]}"
)

print("\nAll columns:")

for i, col in enumerate(
    finalrolling90.columns,
    start=1
):
    print(
        f"{i:2}. {col}"
    )

# ============================================================
# 9. PREVIEW
# ============================================================

print("\n" + "=" * 80)
print("FINAL DATASET PREVIEW")
print("=" * 80)

display(
    finalrolling90.head(10)
)

# ============================================================
# 10. FINAL SAFETY CHECKS
# ============================================================

assert (
    finalrolling90[
        "target"
    ].notna().all()
)

assert (
    finalrolling90[
        "target_label"
    ].notna().all()
)

assert (
    finalrolling90[
        "future_imp_3m"
    ].notna().all()
)

assert (
    finalrolling90[
        "future_impression_change_pct"
    ].notna().all()
)

assert (
    "future_start"
    in finalrolling90.columns
)

assert (
    "future_end"
    in finalrolling90.columns
)

assert (
    "gsc_impressions_prev_30d"
    in finalrolling90.columns
)

assert (
    "gsc_impressions_last_30d"
    in finalrolling90.columns
)

assert (
    "early_drop_signal"
    in finalrolling90.columns
)

# ============================================================
# 11. SAVE FINAL DATASET
# ============================================================

finalrolling90_path = (
    "/content/finalrolling90window.parquet"
)

finalrolling90.to_parquet(
    finalrolling90_path,
    index=False
)

print("\n" + "=" * 80)
print("FINAL ROLLING90WINDOW SAVED")
print("=" * 80)

print(
    finalrolling90_path
)

print("\n" + "=" * 80)
print("BLOCK 4.6 COMPLETE")
print("=" * 80)

print(
    "✓ 90-day rolling features retained"
)

print(
    "✓ Future 90-day target-generation data retained"
)

print(
    "✓ Previous 30-day impressions retained"
)

print(
    "✓ Last 30-day impressions retained"
)

print(
    "✓ Early Drop baseline signal retained"
)

print(
    "✓ Target created: 0=DOWN, 1=FLAT, 2=UP"
)

print(
    "✓ Target labels created"
)

print(
    "✓ Final parquet saved"
)

BLOCK 4.6 — FINAL ROLLING 90-DAY DATASET

FROZEN TARGET RULE
DOWN : change <= -30%
FLAT : -30% < change < +50%
UP   : change >= +50%

TARGET ENCODING
0 = DOWN
1 = FLAT
2 = UP
✓ Encoding confirmed.

BOUNDARY SANITY CHECK
✓ DOWN contains only change <= -30%
✓ FLAT contains only -30% < change < +50%
✓ UP contains only change >= +50%

FINAL TARGET DISTRIBUTION


,target,target_label,count,percentage
0,0,DOWN,206390,32.93
1,1,FLAT,156653,24.99
2,2,UP,263793,42.08



EARLY DROP BASELINE
Early Drop signals : 194,851
Early Drop rate    : 31.08%

Rule:
gsc_impressions_last_30d < gsc_impressions_prev_30d

FINAL ROLLING90WINDOW DATASET
Rows    : 626,836
Columns : 36

All columns:
 1. content_hash_id
 2. client_hash_id
 3. window_start
 4. window_end
 5. future_start
 6. future_end
 7. gsc_clicks_mean_3m
 8. gsc_clicks_last
 9. gsc_impressions_mean_3m
10. gsc_impressions_last
11. gsc_avg_position_mean_3m
12. gsc_avg_position_last
13. ga4_total_engagement_sec_mean_3m
14. ga4_total_engagement_sec_last
15. sessions_organic_mean_3m
16. sessions_organic_last
17. sessions_ai_mean_3m
18. sessions_ai_last
19. gsc_avg_position_missing_mean_3m
20. gsc_avg_position_missing_last
21. ctr_mean_3m
22. ctr_last
23. sec_per_click_mean_3m
24. sec_per_click_last
25. ai_share_mean_3m
26. ai_share_last
27. engagement_per_organic_session_mean_3m
28. engagement_per_organic_session_last
29. current_imp_3m
30. gsc_impressions_prev_30d
31. gsc_impressions_last_30d
32. early_drop

,content_hash_id,client_hash_id,window_start,window_end,future_start,future_end,gsc_clicks_mean_3m,gsc_clicks_last,gsc_impressions_mean_3m,gsc_impressions_last,...,engagement_per_organic_session_mean_3m,engagement_per_organic_session_last,current_imp_3m,gsc_impressions_prev_30d,gsc_impressions_last_30d,early_drop_signal,future_imp_3m,future_impression_change_pct,target,target_label
0,content_000005d4ced12088,client_9958f0a7ae1df715,2025-03-01,2025-05-01,2025-06-01,2025-08-01,0.333333,0.0,136.666667,257.0,...,0.0,0.0,136.666667,7.0,257.0,False,378.333333,176.829268,2,UP
1,content_000005d4ced12088,client_9958f0a7ae1df715,2025-04-01,2025-06-01,2025-07-01,2025-09-01,0.333333,0.0,180.666667,139.0,...,0.0,0.0,180.666667,146.0,139.0,True,506.666667,180.442804,2,UP
2,content_000005d4ced12088,client_9958f0a7ae1df715,2025-05-01,2025-07-01,2025-08-01,2025-10-01,0.000000,0.0,216.666667,254.0,...,0.0,0.0,216.666667,257.0,254.0,True,487.666667,125.076923,2,UP
3,content_000005d4ced12088,client_9958f0a7ae1df715,2025-06-01,2025-08-01,2025-09-01,2025-11-01,0.333333,1.0,378.333333,742.0,...,0.0,0.0,378.333333,139.0,742.0,False,280.333333,-25.903084,1,FLAT
4,content_000005d4ced12088,client_9958f0a7ae1df715,2025-07-01,2025-09-01,2025-10-01,2025-12-01,0.666667,1.0,506.666667,524.0,...,0.0,0.0,506.666667,254.0,524.0,False,167.000000,-67.039474,0,DOWN
5,content_000005d4ced12088,client_9958f0a7ae1df715,2025-08-01,2025-10-01,2025-11-01,2026-01-01,0.666667,0.0,487.666667,197.0,...,0.0,0.0,487.666667,742.0,197.0,True,106.333333,-78.195489,0,DOWN
6,content_000005d4ced12088,client_9958f0a7ae1df715,2025-09-01,2025-11-01,2025-12-01,2026-02-01,0.333333,0.0,280.333333,120.0,...,0.0,0.0,280.333333,524.0,120.0,True,74.333333,-73.483948,0,DOWN
7,content_000005d4ced12088,client_9958f0a7ae1df715,2025-10-01,2025-12-01,2026-01-01,2026-03-01,0.000000,0.0,167.000000,184.0,...,0.0,0.0,167.000000,197.0,184.0,True,41.666667,-75.049900,0,DOWN
8,content_000005d4ced12088,client_9958f0a7ae1df715,2025-11-01,2026-01-01,2026-02-01,2026-04-01,0.000000,0.0,106.333333,15.0,...,0.0,0.0,106.333333,120.0,15.0,True,63.666667,-40.125392,0,DOWN
9,content_000005d4ced12088,client_9958f0a7ae1df715,2025-12-01,2026-02-01,2026-03-01,2026-05-01,0.000000,0.0,74.333333,24.0,...,0.0,0.0,74.333333,184.0,24.0,True,82.666667,11.210762,1,FLAT



FINAL ROLLING90WINDOW SAVED
/content/finalrolling90window.parquet

BLOCK 4.6 COMPLETE
✓ 90-day rolling features retained
✓ Future 90-day target-generation data retained
✓ Previous 30-day impressions retained
✓ Last 30-day impressions retained
✓ Early Drop baseline signal retained
✓ Target created: 0=DOWN, 1=FLAT, 2=UP
✓ Target labels created
✓ Final parquet saved


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method Choice and Why

### Strategy Overview
To address the SEO content decay task, we evaluate a combination of linear benchmarks, non-linear tree ensembles, and validation techniques. We select **Random Forest Classifier** as our primary champion model and **Logistic Regression** as our linear benchmark model.

---

### Comparison of Toolkit Methods

| Toolkit Method | Role in Workflow | Key Strengths | Why Included or Excluded for Decay Lane |
| :--- | :--- | :--- | :--- |
| **Correlation & Signal Analysis** | Feature Screening | Identifies multicollinearity and target leakage. | **Included (Pre-processing):** Essential to strictly remove forbidden trend variables (`trend_pct`, `trend_direction`). |
| **Grouped Validation** | Validation Design | Prevents data leakage across same-client pages. | **Included (Validation):** Grouping by `client_id` ensures the model generalizes to unseen domains rather than memorizing domain-specific baseline numbers. |
| **Logistic Regression** | Baseline ML Model | Simple, fast, and directly interpretable linear baseline. | **Included (ML Baseline):** Benchmark model to verify if a learned linear boundary beats our Week 4 rule-based baseline. |
| **Decision Tree** | Interpretable Model | Visualizable if-else logic trees. | **Included (Secondary):** Useful for quick rules extraction, though prone to higher variance on continuous traffic signals compared to ensembles. |
| **Random Forest** | **Primary Champion Model** | Ensemble of decision trees; handles non-linearities, outliers, and feature interactions. | **SELECTED CHAMPION:** Perfectly fits the power-law nature of web traffic and non-linear ranking drops. |
| **Gradient Boosting** | High-Capacity Model | Strong predictive power on structured tabular data. | **Tested with Constraints:** Evaluated cautiously with shallow depth to avoid overfitting noisy month-to-month traffic fluctuations. |
| **Permutation Importance** | Post-Hoc Interpretability | Measures true feature contribution by shuffling values post-training. | **Included (Sanity Check):** Verifies model honesty and guards against hidden proxy data leakage. |
| **Clustering (K-Means)** | Unsupervised Analysis | Segments content items into distinct performance tiers. | **Exploratory:** Used to analyze structural performance clusters prior to classification. |

---

### Why Random Forest Fits Our SEO Content Decay Lane

1. **Captures Non-Linear SEO Ranking Dynamics:**
   SEO ranking drops do not decay linearly. Losing Rank 1 to Rank 4 results in a catastrophic drop ($\approx 50\%+$) in impressions and CTR, whereas dropping from Rank 25 to Rank 28 has negligible impact. Random Forest handles these step-function thresholds naturally without requiring non-linear feature transformations.

2. **Robust to Heavy-Tailed Power-Law Distributions:**
   Search traffic (`gsc_impressions`) follows a steep power-law distribution where a small percentage of high-traffic pages dominate total volume. Random Forest uses threshold-based splits rather than distance metrics, making it scale-invariant and immune to extreme traffic outliers.

3. **Handles Multi-Signal Feature Interactions:**
   Content decay is rarely caused by a single metric. Random Forest automatically captures multi-variable interaction logic (e.g., *low impressions AND dropping position AND low engagement*) without requiring manual feature engineering.

4. **Transparent Feature Importance & Leakage Defense:**
   Combined with Permutation Importance, Random Forest provides clear insight into which features drive predictions. This ensures the model relies on true signals rather than memorizing forbidden trend indicators.

**Audit to check pages coverage in windows:**

In [18]:
import pandas as pd

# 1. Parquet file load karein
file_path = "/content/finalrolling90window.parquet"
df = pd.read_parquet(file_path)

# 2. Actual dataset columns mapping
page_col = 'content_hash_id'   # Page ID
window_col = 'window_start'    # Window Identifier
client_col = 'client_hash_id'  # Client ID

# 3. Overall Dataset Metrics
total_rows = len(df)
total_unique_pages = df[page_col].nunique()
total_unique_clients = df[client_col].nunique()

# 4. Window-wise Audit Table
audit_df = df.groupby(window_col).agg(
    total_rows=(page_col, 'count'),
    unique_pages=(page_col, 'nunique'),
    unique_clients=(client_col, 'nunique')
).reset_index()

# 5. Percentage Calculations
audit_df['page_coverage_pct'] = ((audit_df['unique_pages'] / total_unique_pages) * 100).round(2)
audit_df['row_share_pct'] = ((audit_df['total_rows'] / total_rows) * 100).round(2)

# Output Print
print("=== OVERALL DATASET METRICS ===")
print(f"Total Rows: {total_rows:,}")
print(f"Total Unique Clients: {total_unique_clients:,}")
print(f"Total Unique Pages (Content Hashes): {total_unique_pages:,}\n")

print("=== WINDOW AUDIT REPORT ===")
print(audit_df.to_string(index=False))

=== OVERALL DATASET METRICS ===
Total Rows: 626,836
Total Unique Clients: 36
Total Unique Pages (Content Hashes): 150,997

=== WINDOW AUDIT REPORT ===
window_start  total_rows  unique_pages  unique_clients  page_coverage_pct  row_share_pct
  2025-01-01         173           173               2               0.11           0.03
  2025-02-01        4388          4388               3               2.91           0.70
  2025-03-01        8723          8723               4               5.78           1.39
  2025-04-01       10879         10879               4               7.20           1.74
  2025-05-01       11860         11860               4               7.85           1.89
  2025-06-01       13482         13482               9               8.93           2.15
  2025-07-01       23387         23387              14              15.49           3.73
  2025-08-01       32357         32357              15              21.43           5.16
  2025-09-01       50214         50214          

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [20]:
import pandas as pd
import numpy as np

# 1. Dataset load karein
file_path = "/content/finalrolling90window.parquet"
df = pd.read_parquet(file_path)
df['window_start'] = pd.to_datetime(df['window_start'])

total_rows = len(df)
total_pages = df['content_hash_id'].nunique()
total_clients = df['client_hash_id'].nunique()

print("==================================================")
print("=== APPROACH 1: LATE TEMPORAL SPLIT (2026-01-01) ==")
print("==================================================")

# Threshold at 2026-01-01 (2025 full = Train, Jan 2026 = Test)
temp_threshold = pd.to_datetime('2026-01-01')
train_temp = df[df['window_start'] < temp_threshold]
test_temp = df[df['window_start'] >= temp_threshold]

print(f"TRAIN: {len(train_temp):,} rows ({len(train_temp)/total_rows*100:.2f}%) | {train_temp['content_hash_id'].nunique():,} pages | {train_temp['client_hash_id'].nunique()} clients")
print(f"TEST : {len(test_temp):,} rows ({len(test_temp)/total_rows*100:.2f}%) | {test_temp['content_hash_id'].nunique():,} pages | {test_temp['client_hash_id'].nunique()} clients\n")


print("==================================================")
print("=== APPROACH 2: GROUPED BY CLIENT SPLIT (80/20) ===")
print("==================================================")

# Client basis par 80-20 split (Unseen websites test karne ke liye)
np.random.seed(42)
unique_clients = df['client_hash_id'].unique()
np.random.shuffle(unique_clients)

train_client_count = int(len(unique_clients) * 0.8)
train_clients = unique_clients[:train_client_count]
test_clients = unique_clients[train_client_count:]

train_grp = df[df['client_hash_id'].isin(train_clients)]
test_grp = df[df['client_hash_id'].isin(test_clients)]

print(f"TRAIN: {len(train_grp):,} rows ({len(train_grp)/total_rows*100:.2f}%) | {train_grp['content_hash_id'].nunique():,} pages | {len(train_clients)} clients")
print(f"TEST : {len(test_grp):,} rows ({len(test_grp)/total_rows*100:.2f}%) | {test_grp['content_hash_id'].nunique():,} pages | {len(test_clients)} clients")

=== APPROACH 1: LATE TEMPORAL SPLIT (2026-01-01) ==
TRAIN: 486,894 rows (77.67%) | 136,266 pages | 36 clients
TEST : 139,942 rows (22.33%) | 139,942 pages | 32 clients

=== APPROACH 2: GROUPED BY CLIENT SPLIT (80/20) ===
TRAIN: 571,381 rows (91.15%) | 134,143 pages | 28 clients
TEST : 55,455 rows (8.85%) | 16,854 pages | 8 clients


**Proper leakage audit on this split**

In [21]:
# ============================================================
# FINAL SPLIT + LEAKAGE AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("FINAL TEMPORAL SPLIT + LEAKAGE AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# 1. LOAD FINAL DATASET
# ------------------------------------------------------------

file_path = "/content/finalrolling90window.parquet"

df = pd.read_parquet(file_path)

df["window_start"] = pd.to_datetime(
    df["window_start"],
    errors="coerce"
)

print(f"Rows    : {len(df):,}")
print(f"Columns : {df.shape[1]}")

# ------------------------------------------------------------
# 2. TARGET VERIFICATION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TARGET VERIFICATION")
print("=" * 80)

assert "target" in df.columns
assert "target_label" in df.columns

print("Target unique values:", sorted(df["target"].dropna().unique()))

assert set(df["target"].dropna().unique()).issubset({0, 1, 2})

print("✓ 0 = DOWN")
print("✓ 1 = FLAT")
print("✓ 2 = UP")

# ------------------------------------------------------------
# 3. FUTURE COLUMN AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FUTURE COLUMN AUDIT")
print("=" * 80)

future_columns = [
    c for c in df.columns
    if c.lower().startswith("future_")
]

print("Future columns found:")

for col in future_columns:
    print(" -", col)

print(f"\nTotal future columns: {len(future_columns)}")

# These are allowed to exist in final dataset.
# They MUST NOT enter X.

# ------------------------------------------------------------
# 4. EXPLICIT TARGET / LEAKAGE COLUMNS
# ------------------------------------------------------------

target_columns = {
    "target",
    "target_label",
    "future_impression_change_pct",
    "future_imp_3m"
}

leakage_columns = (
    target_columns
    | set(future_columns)
)

print("\n" + "=" * 80)
print("TARGET / FUTURE LEAKAGE COLUMNS")
print("=" * 80)

for col in sorted(leakage_columns):
    if col in df.columns:
        print("BLOCKED:", col)

# ------------------------------------------------------------
# 5. TEMPORAL SPLIT
# ------------------------------------------------------------

cutoff = pd.Timestamp("2026-01-01")

train = df[
    df["window_start"] < cutoff
].copy()

test = df[
    df["window_start"] >= cutoff
].copy()

print("\n" + "=" * 80)
print("TEMPORAL SPLIT")
print("=" * 80)

print(f"Cutoff : {cutoff.date()}")

print(
    f"TRAIN rows : {len(train):,} "
    f"({len(train)/len(df)*100:.2f}%)"
)

print(
    f"TEST rows  : {len(test):,} "
    f"({len(test)/len(df)*100:.2f}%)"
)

# ------------------------------------------------------------
# 6. TEMPORAL SAFETY
# ------------------------------------------------------------

train_max = train["window_start"].max()
test_min = test["window_start"].min()

print("\nTrain latest window :", train_max.date())
print("Test earliest window:", test_min.date())

assert train_max < cutoff
assert test_min >= cutoff

print("✓ Temporal ordering is correct.")

# ------------------------------------------------------------
# 7. PAGE OVERLAP
# ------------------------------------------------------------

train_pages = set(
    train["content_hash_id"].dropna().unique()
)

test_pages = set(
    test["content_hash_id"].dropna().unique()
)

page_overlap = (
    train_pages &
    test_pages
)

print("\n" + "=" * 80)
print("PAGE OVERLAP AUDIT")
print("=" * 80)

print(f"Train pages : {len(train_pages):,}")
print(f"Test pages  : {len(test_pages):,}")
print(f"Overlap     : {len(page_overlap):,}")

if len(page_overlap) > 0:
    print(
        "\n✓ Page overlap exists — this is expected "
        "for future prediction of existing pages."
    )
else:
    print("✓ No page overlap.")

# ------------------------------------------------------------
# 8. CLIENT OVERLAP
# ------------------------------------------------------------

train_clients = set(
    train["client_hash_id"].dropna().unique()
)

test_clients = set(
    test["client_hash_id"].dropna().unique()
)

client_overlap = (
    train_clients &
    test_clients
)

print("\n" + "=" * 80)
print("CLIENT OVERLAP AUDIT")
print("=" * 80)

print(f"Train clients : {len(train_clients):,}")
print(f"Test clients  : {len(test_clients):,}")
print(f"Overlap       : {len(client_overlap):,}")

# Client overlap is NOT leakage for the primary
# existing-client future prediction objective.

# ------------------------------------------------------------
# 9. MODEL FEATURE CANDIDATES
# ------------------------------------------------------------

excluded = {
    "content_hash_id",
    "client_hash_id",
    "window_start",
    "window_end",
    "month",
    "target",
    "target_label"
}

excluded.update(future_columns)

candidate_features = [
    c for c in df.columns
    if c not in excluded
]

# ------------------------------------------------------------
# 10. SECONDARY NAME-BASED LEAKAGE CHECK
# ------------------------------------------------------------

suspicious_features = []

for col in candidate_features:

    name = col.lower()

    suspicious_words = [
        "future",
        "target",
        "label",
        "next_3m",
        "next_90d",
        "decay_rate"
    ]

    if any(word in name for word in suspicious_words):
        suspicious_features.append(col)

print("\n" + "=" * 80)
print("SUSPICIOUS FEATURE-NAME AUDIT")
print("=" * 80)

if suspicious_features:

    for col in suspicious_features:
        print("REVIEW:", col)

else:
    print("✓ No suspicious feature names found.")

# ------------------------------------------------------------
# 11. BUILD X / y
# ------------------------------------------------------------

X_train = train[candidate_features].copy()
X_test = test[candidate_features].copy()

y_train = train["target"].copy()
y_test = test["target"].copy()

print("\n" + "=" * 80)
print("MODEL INPUT")
print("=" * 80)

print("Number of features:", len(candidate_features))
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

# ------------------------------------------------------------
# 12. FINAL NA CHECK
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MISSING VALUE CHECK")
print("=" * 80)

train_missing = X_train.isna().sum().sum()
test_missing = X_test.isna().sum().sum()

print("Train missing values:", train_missing)
print("Test missing values :", test_missing)

# ------------------------------------------------------------
# 13. TARGET DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TRAIN TARGET DISTRIBUTION")
print("=" * 80)

train_dist = (
    y_train
    .value_counts()
    .sort_index()
    .rename_axis("target")
    .reset_index(name="count")
)

train_dist["percentage"] = (
    train_dist["count"]
    / len(y_train)
    * 100
).round(2)

display(train_dist)

print("\n" + "=" * 80)
print("TEST TARGET DISTRIBUTION")
print("=" * 80)

test_dist = (
    y_test
    .value_counts()
    .sort_index()
    .rename_axis("target")
    .reset_index(name="count")
)

test_dist["percentage"] = (
    test_dist["count"]
    / len(y_test)
    * 100
).round(2)

display(test_dist)

# ------------------------------------------------------------
# 14. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL AUDIT VERDICT")
print("=" * 80)

print("✓ Temporal cutoff applied.")
print("✓ Test contains later windows than training.")
print("✓ Future columns excluded from X.")
print("✓ Target columns excluded from X.")
print("✓ Target encoding verified.")
print("✓ Existing-page overlap is allowed for future prediction.")
print("✓ No random row mixing.")
print("✓ No artificial class balancing.")
print("✓ Candidate features detected from final dataset.")

print("\nFINAL FEATURE COUNT:", len(candidate_features))

FINAL TEMPORAL SPLIT + LEAKAGE AUDIT
Rows    : 626,836
Columns : 36

TARGET VERIFICATION
Target unique values: [np.int8(0), np.int8(1), np.int8(2)]
✓ 0 = DOWN
✓ 1 = FLAT
✓ 2 = UP

FUTURE COLUMN AUDIT
Future columns found:
 - future_start
 - future_end
 - future_imp_3m
 - future_impression_change_pct

Total future columns: 4

TARGET / FUTURE LEAKAGE COLUMNS
BLOCKED: future_end
BLOCKED: future_imp_3m
BLOCKED: future_impression_change_pct
BLOCKED: future_start
BLOCKED: target
BLOCKED: target_label

TEMPORAL SPLIT
Cutoff : 2026-01-01
TRAIN rows : 486,894 (77.67%)
TEST rows  : 139,942 (22.33%)

Train latest window : 2025-12-01
Test earliest window: 2026-01-01
✓ Temporal ordering is correct.

PAGE OVERLAP AUDIT
Train pages : 136,266
Test pages  : 139,942
Overlap     : 125,211

✓ Page overlap exists — this is expected for future prediction of existing pages.

CLIENT OVERLAP AUDIT
Train clients : 36
Test clients  : 32
Overlap       : 32

SUSPICIOUS FEATURE-NAME AUDIT
✓ No suspicious feature na

,target,count,percentage
0,0,132136,27.14
1,1,119430,24.53
2,2,235328,48.33



TEST TARGET DISTRIBUTION


,target,count,percentage
0,0,74254,53.06
1,1,37223,26.60
2,2,28465,20.34



FINAL AUDIT VERDICT
✓ Temporal cutoff applied.
✓ Test contains later windows than training.
✓ Future columns excluded from X.
✓ Target columns excluded from X.
✓ Target encoding verified.
✓ Existing-page overlap is allowed for future prediction.
✓ No random row mixing.
✓ No artificial class balancing.
✓ Candidate features detected from final dataset.

FINAL FEATURE COUNT: 26


Selected Evaluation Strategy
Late Temporal Split (Cutoff Date: 2026-01-01)

Train Set (2025-01-01 to 2025-12-01): 486,894 rows (77.67%) | 136,266 unique pages | 36 clients

Test Set (2026-01-01): 139,942 rows (22.33%) | 139,942 unique pages | 32 clients

💡 Justification & Reasons
Zero Data Leakage: SEO traffic forecasting time-series problem hai. Strictly 2026-01-01 par cut karne se future window data train set me leak hone se bach jata hai.

Production-Like Simulation: Real-world deployment ko simulate karta hai jahan historical trends se future performance forecast ki jati hai.

Ideal Split Ratio (~78/22): Without artificial downsampling, naturally balanced 77.67% Train aur 22.33% Test distribution milti hai.

Feature Generalization: Model static page IDs memorized karne ke bajaye real GSC impressions aur engagement dynamics ke actual signals seekhta hai.

**Block 5.1 — Final Rolling Dataset Load + Structure Check**

In [1]:
# ============================================================
# BLOCK 5.1 — FINAL ROLLING DATASET LOAD + STRUCTURE CHECK
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("BLOCK 5.1 — FINAL ROLLING DATASET LOAD + STRUCTURE CHECK")
print("=" * 80)

# ------------------------------------------------------------
# 1. LOAD FINAL DATASET
# ------------------------------------------------------------

file_path = "/content/finalrolling90window.parquet"

df_final = pd.read_parquet(file_path)

# ------------------------------------------------------------
# 2. BASIC INFORMATION
# ------------------------------------------------------------

print(f"\nSource file : {file_path}")
print(f"Rows        : {len(df_final):,}")
print(f"Columns     : {df_final.shape[1]}")

# ------------------------------------------------------------
# 3. REQUIRED CORE COLUMNS
# ------------------------------------------------------------

required_columns = [
    "content_hash_id",
    "client_hash_id",
    "window_start",
    "window_end",
    "target",
    "target_label"
]

missing_required = [
    col for col in required_columns
    if col not in df_final.columns
]

if missing_required:
    raise KeyError(
        "Required columns missing:\n"
        + "\n".join(missing_required)
    )

# ------------------------------------------------------------
# 4. DATE CONVERSION
# ------------------------------------------------------------

df_final["window_start"] = pd.to_datetime(
    df_final["window_start"],
    errors="coerce"
)

df_final["window_end"] = pd.to_datetime(
    df_final["window_end"],
    errors="coerce"
)

if df_final["window_start"].isna().any():
    raise ValueError(
        "window_start contains invalid/missing dates."
    )

# ------------------------------------------------------------
# 5. TARGET VALIDATION
# ------------------------------------------------------------

target_values = set(
    pd.to_numeric(
        df_final["target"],
        errors="coerce"
    ).dropna().unique()
)

print("\n" + "=" * 80)
print("TARGET VALIDATION")
print("=" * 80)

print("Target values:", sorted(target_values))

if not target_values.issubset({0, 1, 2}):
    raise ValueError(
        f"Unexpected target values: {target_values}"
    )

if df_final["target"].isna().any():
    raise ValueError("Target contains missing values.")

print("✓ 0 = DOWN")
print("✓ 1 = FLAT")
print("✓ 2 = UP")

# ------------------------------------------------------------
# 6. TARGET LABEL VALIDATION
# ------------------------------------------------------------

expected_labels = {
    0: "DOWN",
    1: "FLAT",
    2: "UP"
}

label_check = (
    df_final[["target", "target_label"]]
    .drop_duplicates()
    .sort_values("target")
)

print("\nTarget mapping:")
display(label_check)

for target_value, label in expected_labels.items():

    rows = df_final[
        df_final["target"] == target_value
    ]

    if len(rows) > 0:

        actual_labels = set(
            rows["target_label"]
            .astype(str)
            .str.upper()
            .unique()
        )

        if actual_labels != {label}:
            raise ValueError(
                f"Target mapping incorrect for "
                f"{target_value}: {actual_labels}"
            )




print("✓ Target encoding confirmed.")

# ------------------------------------------------------------
# 7. FUTURE COLUMN AUDIT
# ------------------------------------------------------------

future_columns = [
    col for col in df_final.columns
    if col.startswith("future_")
]

print("\n" + "=" * 80)
print("FUTURE COLUMN AUDIT")
print("=" * 80)

for col in future_columns:
    print(" -", col)

print(f"\nFuture columns found: {len(future_columns)}")

# ------------------------------------------------------------
# 8. FINAL STRUCTURE PREVIEW
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL DATASET PREVIEW")
print("=" * 80)

display(df_final.head(5))


print("\n✓ BLOCK 5.1 COMPLETE")

BLOCK 5.1 — FINAL ROLLING DATASET LOAD + STRUCTURE CHECK

Source file : /content/finalrolling90window.parquet
Rows        : 626,836
Columns     : 36

TARGET VALIDATION
Target values: [np.int8(0), np.int8(1), np.int8(2)]
✓ 0 = DOWN
✓ 1 = FLAT
✓ 2 = UP

Target mapping:


,target,target_label
4,0,DOWN
3,1,FLAT
0,2,UP


✓ Target encoding confirmed.

FUTURE COLUMN AUDIT
 - future_start
 - future_end
 - future_imp_3m
 - future_impression_change_pct

Future columns found: 4

FINAL DATASET PREVIEW


,content_hash_id,client_hash_id,window_start,window_end,future_start,future_end,gsc_clicks_mean_3m,gsc_clicks_last,gsc_impressions_mean_3m,gsc_impressions_last,...,engagement_per_organic_session_mean_3m,engagement_per_organic_session_last,current_imp_3m,gsc_impressions_prev_30d,gsc_impressions_last_30d,early_drop_signal,future_imp_3m,future_impression_change_pct,target,target_label
0,content_000005d4ced12088,client_9958f0a7ae1df715,2025-03-01,2025-05-01,2025-06-01,2025-08-01,0.333333,0.0,136.666667,257.0,...,0.0,0.0,136.666667,7.0,257.0,False,378.333333,176.829268,2,UP
1,content_000005d4ced12088,client_9958f0a7ae1df715,2025-04-01,2025-06-01,2025-07-01,2025-09-01,0.333333,0.0,180.666667,139.0,...,0.0,0.0,180.666667,146.0,139.0,True,506.666667,180.442804,2,UP
2,content_000005d4ced12088,client_9958f0a7ae1df715,2025-05-01,2025-07-01,2025-08-01,2025-10-01,0.000000,0.0,216.666667,254.0,...,0.0,0.0,216.666667,257.0,254.0,True,487.666667,125.076923,2,UP
3,content_000005d4ced12088,client_9958f0a7ae1df715,2025-06-01,2025-08-01,2025-09-01,2025-11-01,0.333333,1.0,378.333333,742.0,...,0.0,0.0,378.333333,139.0,742.0,False,280.333333,-25.903084,1,FLAT
4,content_000005d4ced12088,client_9958f0a7ae1df715,2025-07-01,2025-09-01,2025-10-01,2025-12-01,0.666667,1.0,506.666667,524.0,...,0.0,0.0,506.666667,254.0,524.0,False,167.000000,-67.039474,0,DOWN



✓ BLOCK 5.1 COMPLETE


**BLOCK 5.2 — SEPARATE X/y + LEAKAGE CHECK**

In [2]:
# ============================================================
# BLOCK 5.2 — SEPARATE X / y + LEAKAGE CHECK
# ============================================================

print("=" * 80)
print("BLOCK 5.2 — X / y SEPARATION + LEAKAGE AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# 1. COLUMNS THAT MUST NEVER ENTER X
# ------------------------------------------------------------

blocked_columns = {
    # Metadata
    "content_hash_id",
    "client_hash_id",
    "window_start",
    "window_end",

    # Target
    "target",
    "target_label",

    # Explicit future information
    "future_start",
    "future_end",
    "future_imp_3m",
    "future_impression_change_pct"
}

# ------------------------------------------------------------
# 2. FIND POTENTIAL FUTURE COLUMNS
# ------------------------------------------------------------

future_name_columns = [
    col
    for col in df_final.columns
    if col.lower().startswith("future_")
]

# Add every future_* column to blocked set
blocked_columns.update(future_name_columns)

# ------------------------------------------------------------
# 3. CREATE MODEL FEATURE LIST
# ------------------------------------------------------------

feature_columns = [
    col
    for col in df_final.columns
    if col not in blocked_columns
]

# ------------------------------------------------------------
# 4. SUSPICIOUS NAME AUDIT
# ------------------------------------------------------------

suspicious_keywords = [
    "future",
    "target",
    "label",
    "impression_change",
    "next_",
    "outcome"
]

suspicious_features = [
    col
    for col in feature_columns
    if any(
        keyword in col.lower()
        for keyword in suspicious_keywords
    )
]

print("\n" + "=" * 80)
print("LEAKAGE CHECK")
print("=" * 80)

print(f"Candidate model features: {len(feature_columns)}")

if suspicious_features:
    print("\nWARNING — suspicious feature names:")
    for col in suspicious_features:
        print(" -", col)

    raise ValueError(
        "Potential leakage detected in model features."
    )

print("✓ No suspicious feature names found.")

# ------------------------------------------------------------
# 5. VERIFY NO FUTURE / TARGET COLUMNS IN X
# ------------------------------------------------------------

invalid_x_columns = [
    col
    for col in feature_columns
    if (
        col.startswith("future_")
        or col in {"target", "target_label"}
    )
]

if invalid_x_columns:
    raise ValueError(
        "Leakage columns found in X:\n"
        + "\n".join(invalid_x_columns)
    )

# ------------------------------------------------------------
# 6. CREATE X AND y
# ------------------------------------------------------------

X = df_final[feature_columns].copy()

y = pd.to_numeric(
    df_final["target"],
    errors="coerce"
).astype("int8")

# ------------------------------------------------------------
# 7. NUMERIC FEATURE CHECK
# ------------------------------------------------------------

non_numeric_features = [
    col
    for col in X.columns
    if not pd.api.types.is_numeric_dtype(X[col])
]

if non_numeric_features:
    raise TypeError(
        "Non-numeric model features found:\n"
        + "\n".join(non_numeric_features)
    )

# ------------------------------------------------------------
# 8. MISSING VALUE CHECK
# ------------------------------------------------------------

train_ready_missing = X.isna().sum()

missing_features = (
    train_ready_missing[
        train_ready_missing > 0
    ]
)

if len(missing_features) > 0:

    print("\nMissing values found:")
    print(missing_features)

    raise ValueError(
        "Model features contain missing values."
    )

# ------------------------------------------------------------
# 9. TARGET CHECK
# ------------------------------------------------------------

if not set(y.unique()).issubset({0, 1, 2}):
    raise ValueError(
        "Target contains values outside 0,1,2."
    )

# ------------------------------------------------------------
# 10. FINAL FEATURE LIST
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL MODEL INPUT FEATURES")
print("=" * 80)

for i, col in enumerate(feature_columns, 1):
    print(f"{i:2}. {col}")

print("\n" + "=" * 80)
print("MODEL INPUT SHAPE")
print("=" * 80)

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")

print("\n✓ No future columns in X")
print("✓ No target columns in X")
print("✓ No target_label in X")
print("✓ No metadata in X")
print("✓ No suspicious feature names")
print("✓ No missing feature values")
print("✓ Target = 0 / 1 / 2")

print("\n✓ BLOCK 5.2 COMPLETE")

BLOCK 5.2 — X / y SEPARATION + LEAKAGE AUDIT

LEAKAGE CHECK
Candidate model features: 26
✓ No suspicious feature names found.

FINAL MODEL INPUT FEATURES
 1. gsc_clicks_mean_3m
 2. gsc_clicks_last
 3. gsc_impressions_mean_3m
 4. gsc_impressions_last
 5. gsc_avg_position_mean_3m
 6. gsc_avg_position_last
 7. ga4_total_engagement_sec_mean_3m
 8. ga4_total_engagement_sec_last
 9. sessions_organic_mean_3m
10. sessions_organic_last
11. sessions_ai_mean_3m
12. sessions_ai_last
13. gsc_avg_position_missing_mean_3m
14. gsc_avg_position_missing_last
15. ctr_mean_3m
16. ctr_last
17. sec_per_click_mean_3m
18. sec_per_click_last
19. ai_share_mean_3m
20. ai_share_last
21. engagement_per_organic_session_mean_3m
22. engagement_per_organic_session_last
23. current_imp_3m
24. gsc_impressions_prev_30d
25. gsc_impressions_last_30d
26. early_drop_signal

MODEL INPUT SHAPE
X shape : (626836, 26)
y shape : (626836,)

✓ No future columns in X
✓ No target columns in X
✓ No target_label in X
✓ No metadata in X

**BLOCK 5.3 — LATE TEMPORAL TRAIN / TEST SPLIT**

In [3]:

# ============================================================
# BLOCK 5.3 — LATE TEMPORAL TRAIN / TEST SPLIT
# ============================================================

print("=" * 80)
print("BLOCK 5.3 — LATE TEMPORAL TRAIN / TEST SPLIT")
print("=" * 80)

# ------------------------------------------------------------
# 1. TEMPORAL CUTOFF
# ------------------------------------------------------------

temporal_cutoff = pd.Timestamp("2026-01-01")

# ------------------------------------------------------------
# 2. CREATE MASKS
# ------------------------------------------------------------

train_mask = (
    df_final["window_start"] < temporal_cutoff
)

test_mask = (
    df_final["window_start"] >= temporal_cutoff
)

# ------------------------------------------------------------
# 3. SPLIT X / y
# ------------------------------------------------------------

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

# ------------------------------------------------------------
# 4. BASIC SIZE CHECK
# ------------------------------------------------------------

if len(X_train) == 0:
    raise ValueError("Training set is empty.")

if len(X_test) == 0:
    raise ValueError("Test set is empty.")

# ------------------------------------------------------------
# 5. TEMPORAL ORDER CHECK
# ------------------------------------------------------------

train_latest = df_final.loc[
    train_mask,
    "window_start"
].max()

test_earliest = df_final.loc[
    test_mask,
    "window_start"
].min()

if train_latest >= test_earliest:
    raise ValueError(
        "Temporal leakage: training extends into test period."
    )

# ------------------------------------------------------------
# 6. PAGE OVERLAP AUDIT
# ------------------------------------------------------------

train_pages = set(
    df_final.loc[
        train_mask,
        "content_hash_id"
    ]
)

test_pages = set(
    df_final.loc[
        test_mask,
        "content_hash_id"
    ]
)

page_overlap = (
    len(train_pages.intersection(test_pages))
)

# Existing-page overlap is allowed because this is
# future trajectory prediction.

# ------------------------------------------------------------
# 7. CLIENT OVERLAP AUDIT
# ------------------------------------------------------------

train_clients = set(
    df_final.loc[
        train_mask,
        "client_hash_id"
    ]
)

test_clients = set(
    df_final.loc[
        test_mask,
        "client_hash_id"
    ]
)

client_overlap = (
    len(train_clients.intersection(test_clients))
)

# ------------------------------------------------------------
# 8. DISTRIBUTION FUNCTION
# ------------------------------------------------------------

def distribution(series):

    result = (
        series
        .value_counts()
        .sort_index()
        .rename_axis("target")
        .reset_index(name="count")
    )

    result["percentage"] = (
        result["count"]
        / len(series)
        * 100
    ).round(2)

    result["label"] = result["target"].map({
        0: "DOWN",
        1: "FLAT",
        2: "UP"
    })

    return result[
        ["target", "label", "count", "percentage"]
    ]

# ------------------------------------------------------------
# 9. REPORT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TEMPORAL SPLIT")
print("=" * 80)

print(f"Cutoff : {temporal_cutoff.date()}")

print(
    f"TRAIN : {len(X_train):,} "
    f"({len(X_train)/len(X)*100:.2f}%)"
)

print(
    f"TEST  : {len(X_test):,} "
    f"({len(X_test)/len(X)*100:.2f}%)"
)

print(f"\nTrain latest window : {train_latest.date()}")
print(f"Test earliest window: {test_earliest.date()}")

print("\n✓ Temporal ordering verified.")

# ------------------------------------------------------------
# 10. PAGE AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PAGE OVERLAP AUDIT")
print("=" * 80)

print(f"Train pages : {len(train_pages):,}")
print(f"Test pages  : {len(test_pages):,}")
print(f"Overlap     : {page_overlap:,}")

print(
    "\n✓ Existing-page overlap is allowed "
    "for future trajectory prediction."
)

# ------------------------------------------------------------
# 11. CLIENT AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CLIENT OVERLAP AUDIT")
print("=" * 80)

print(f"Train clients : {len(train_clients):,}")
print(f"Test clients  : {len(test_clients):,}")
print(f"Overlap       : {client_overlap:,}")

print(
    "\nNote: Client overlap is expected because "
    "the model predicts future behavior of existing clients/pages."
)

# ------------------------------------------------------------
# 12. TRAIN DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TRAIN TARGET DISTRIBUTION")
print("=" * 80)

display(
    distribution(y_train)
)

# ------------------------------------------------------------
# 13. TEST DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TEST TARGET DISTRIBUTION")
print("=" * 80)

display(
    distribution(y_test)
)

# ------------------------------------------------------------
# 14. FINAL SHAPES
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL SPLIT SHAPES")
print("=" * 80)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

# ------------------------------------------------------------
# 15. FINAL ASSERTIONS
# ------------------------------------------------------------

assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

assert train_latest < test_earliest

assert not any(
    col.startswith("future_")
    for col in X_train.columns
)

assert "target" not in X_train.columns
assert "target_label" not in X_train.columns

print("\n✓ No temporal leakage.")
print("✓ No future columns in training/testing.")
print("✓ No target columns in training/testing.")
print("✓ Split is strictly time-aware.")

print("\n✓ BLOCK 5.3 COMPLETE")

BLOCK 5.3 — LATE TEMPORAL TRAIN / TEST SPLIT

TEMPORAL SPLIT
Cutoff : 2026-01-01
TRAIN : 486,894 (77.67%)
TEST  : 139,942 (22.33%)

Train latest window : 2025-12-01
Test earliest window: 2026-01-01

✓ Temporal ordering verified.

PAGE OVERLAP AUDIT
Train pages : 136,266
Test pages  : 139,942
Overlap     : 125,211

✓ Existing-page overlap is allowed for future trajectory prediction.

CLIENT OVERLAP AUDIT
Train clients : 36
Test clients  : 32
Overlap       : 32

Note: Client overlap is expected because the model predicts future behavior of existing clients/pages.

TRAIN TARGET DISTRIBUTION


,target,label,count,percentage
0,0,DOWN,132136,27.14
1,1,FLAT,119430,24.53
2,2,UP,235328,48.33



TEST TARGET DISTRIBUTION


,target,label,count,percentage
0,0,DOWN,74254,53.06
1,1,FLAT,37223,26.60
2,2,UP,28465,20.34



FINAL SPLIT SHAPES
X_train : (486894, 26)
X_test  : (139942, 26)
y_train : (486894,)
y_test  : (139942,)

✓ No temporal leakage.
✓ No future columns in training/testing.
✓ No target columns in training/testing.
✓ Split is strictly time-aware.

✓ BLOCK 5.3 COMPLETE


**BLOCK A — Target Quality Audit**

In [4]:
# ================================================================
# AUDIT A — TARGET QUALITY + THRESHOLD SANITY CHECK
# ================================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("AUDIT A — TARGET QUALITY + THRESHOLD SANITY CHECK")
print("=" * 80)

# ------------------------------------------------
# 1. LOAD FINAL DATASET
# ------------------------------------------------

path = "/content/finalrolling90window.parquet"

df_audit = pd.read_parquet(path)

df_audit["window_start"] = pd.to_datetime(
    df_audit["window_start"],
    errors="coerce"
)

df_audit["future_start"] = pd.to_datetime(
    df_audit["future_start"],
    errors="coerce"
)

print(f"\nRows : {len(df_audit):,}")
print(f"Cols : {df_audit.shape[1]}")

# ------------------------------------------------
# 2. REQUIRED COLUMNS
# ------------------------------------------------

required = [
    "window_start",
    "future_start",
    "future_impression_change_pct",
    "target"
]

missing = [
    c for c in required
    if c not in df_audit.columns
]

if missing:
    raise KeyError(
        "Missing required columns:\n"
        + "\n".join(missing)
    )

# ------------------------------------------------
# 3. RE-CALCULATE TARGET INDEPENDENTLY
# ------------------------------------------------

change = pd.to_numeric(
    df_audit["future_impression_change_pct"],
    errors="coerce"
)

recalculated_target = np.select(
    [
        change <= -30,
        change < 50
    ],
    [
        0,
        1
    ],
    default=2
).astype("int8")

stored_target = pd.to_numeric(
    df_audit["target"],
    errors="coerce"
)

# ------------------------------------------------
# 4. TARGET CONSISTENCY
# ------------------------------------------------

mismatch = (
    stored_target != recalculated_target
)

print("\n" + "=" * 80)
print("TARGET CONSISTENCY")
print("=" * 80)

print(
    f"Target mismatches : {mismatch.sum():,}"
)

assert mismatch.sum() == 0, (
    "ERROR: Stored target does not match "
    "the frozen -30 / +50 rule."
)

print("✓ Target exactly matches frozen rule.")

# ------------------------------------------------
# 5. CLASS DISTRIBUTION
# ------------------------------------------------

labels = {
    0: "DOWN",
    1: "FLAT",
    2: "UP"
}

summary = (
    stored_target
    .value_counts()
    .sort_index()
    .rename_axis("target")
    .reset_index(name="count")
)

summary["label"] = summary["target"].map(labels)

summary["percentage"] = (
    summary["count"] /
    len(df_audit) * 100
).round(2)

print("\n" + "=" * 80)
print("OVERALL TARGET DISTRIBUTION")
print("=" * 80)

display(
    summary[
        ["target", "label", "count", "percentage"]
    ]
)

# ------------------------------------------------
# 6. BOUNDARY TEST
# ------------------------------------------------

print("\n" + "=" * 80)
print("BOUNDARY SANITY CHECK")
print("=" * 80)

boundary_checks = {
    "DOWN <= -30": (
        (stored_target == 0)
        & (change > -30)
    ).sum(),

    "FLAT > -30": (
        (stored_target == 1)
        & (change <= -30)
    ).sum(),

    "FLAT < +50": (
        (stored_target == 1)
        & (change >= 50)
    ).sum(),

    "UP >= +50": (
        (stored_target == 2)
        & (change < 50)
    ).sum()
}

for name, count in boundary_checks.items():
    print(f"{name:<20}: {count:,}")

assert all(
    count == 0
    for count in boundary_checks.values()
)

print("\n✓ All target boundaries are correct.")

# ------------------------------------------------
# 7. CHANGE DISTRIBUTION
# ------------------------------------------------

print("\n" + "=" * 80)
print("CHANGE DISTRIBUTION")
print("=" * 80)

stats = pd.Series({
    "Minimum": change.min(),
    "Q01": change.quantile(.01),
    "Q05": change.quantile(.05),
    "Q10": change.quantile(.10),
    "Q25": change.quantile(.25),
    "Median": change.median(),
    "Q75": change.quantile(.75),
    "Q90": change.quantile(.90),
    "Q95": change.quantile(.95),
    "Q99": change.quantile(.99),
    "Maximum": change.max()
})

display(
    stats.round(2).rename("change_pct").to_frame()
)

# ------------------------------------------------
# 8. EXTREME VALUES
# ------------------------------------------------

print("\n" + "=" * 80)
print("EXTREME CHANGE CHECK")
print("=" * 80)

print(
    "Rows <= -90% :",
    (change <= -90).sum()
)

print(
    "Rows >= +500%:",
    (change >= 500).sum()
)

print(
    "Rows >= +1000%:",
    (change >= 1000).sum()
)

# ------------------------------------------------
# 9. CURRENT VS FUTURE IMPRESSIONS
# ------------------------------------------------

for col in [
    "current_imp_3m",
    "future_imp_3m"
]:
    if col in df_audit.columns:
        df_audit[col] = pd.to_numeric(
            df_audit[col],
            errors="coerce"
        )

print("\n" + "=" * 80)
print("ZERO / VERY LOW IMPRESSION CHECK")
print("=" * 80)

if "current_imp_3m" in df_audit.columns:

    print(
        "Current 3M impressions = 0:",
        (df_audit["current_imp_3m"] == 0).sum()
    )

    print(
        "Current 3M impressions < 10:",
        (df_audit["current_imp_3m"] < 10).sum()
    )

if "future_imp_3m" in df_audit.columns:

    print(
        "Future 3M impressions = 0:",
        (df_audit["future_imp_3m"] == 0).sum()
    )

print("\n" + "=" * 80)
print("AUDIT A COMPLETE")
print("=" * 80)

AUDIT A — TARGET QUALITY + THRESHOLD SANITY CHECK

Rows : 626,836
Cols : 36

TARGET CONSISTENCY
Target mismatches : 0
✓ Target exactly matches frozen rule.

OVERALL TARGET DISTRIBUTION


,target,label,count,percentage
0,0,DOWN,206390,32.93
1,1,FLAT,156653,24.99
2,2,UP,263793,42.08



BOUNDARY SANITY CHECK
DOWN <= -30         : 0
FLAT > -30          : 0
FLAT < +50          : 0
UP >= +50           : 0

✓ All target boundaries are correct.

CHANGE DISTRIBUTION


,change_pct
Minimum,-100.00
Q01,-100.00
Q05,-100.00
Q10,-90.00
Q25,-50.16
Median,19.57
Q75,150.00
Q90,442.13
Q95,909.09
Q99,4678.11



EXTREME CHANGE CHECK
Rows <= -90% : 62514
Rows >= +500%: 56140
Rows >= +1000%: 28828

ZERO / VERY LOW IMPRESSION CHECK
Current 3M impressions = 0: 0
Current 3M impressions < 10: 152759
Future 3M impressions = 0: 45688

AUDIT A COMPLETE


**BLOCK B — Temporal Target Drift Audit**

In [5]:
# ================================================================
# AUDIT B — TEMPORAL TARGET DRIFT
# ================================================================

print("=" * 80)
print("AUDIT B — TEMPORAL TARGET DRIFT")
print("=" * 80)

df_b = df_audit.copy()

df_b["year_month"] = (
    df_b["window_start"]
    .dt.to_period("M")
)

# ------------------------------------------------
# MONTHLY TARGET COUNTS
# ------------------------------------------------

monthly_counts = pd.crosstab(
    df_b["year_month"],
    df_b["target"]
)

for cls in [0, 1, 2]:
    if cls not in monthly_counts.columns:
        monthly_counts[cls] = 0

monthly_counts = monthly_counts[
    [0, 1, 2]
]

monthly_pct = (
    monthly_counts
    .div(monthly_counts.sum(axis=1), axis=0)
    * 100
).round(2)

monthly_pct.columns = [
    "DOWN_%",
    "FLAT_%",
    "UP_%"
]

monthly_pct = monthly_pct.reset_index()

print("\n" + "=" * 80)
print("MONTHLY TARGET DISTRIBUTION")
print("=" * 80)

display(monthly_pct)

# ------------------------------------------------
# EARLY VS LATE PERIOD
# ------------------------------------------------

cutoff = pd.Timestamp("2026-01-01")

early = df_b[
    df_b["window_start"] < cutoff
]

late = df_b[
    df_b["window_start"] >= cutoff
]

def distribution(data):

    counts = (
        data["target"]
        .value_counts()
        .reindex([0, 1, 2], fill_value=0)
    )

    return pd.DataFrame({
        "count": counts,
        "percentage": (
            counts / len(data) * 100
        ).round(2)
    }, index=["DOWN", "FLAT", "UP"])

print("\n" + "=" * 80)
print("BEFORE 2026-01-01")
print("=" * 80)

display(distribution(early))

print("\n" + "=" * 80)
print("FROM 2026-01-01")
print("=" * 80)

display(distribution(late))

# ------------------------------------------------
# DRIFT DIFFERENCE
# ------------------------------------------------

early_dist = (
    early["target"]
    .value_counts(normalize=True)
    .reindex([0, 1, 2], fill_value=0)
    * 100
)

late_dist = (
    late["target"]
    .value_counts(normalize=True)
    .reindex([0, 1, 2], fill_value=0)
    * 100
)

drift = pd.DataFrame({
    "early_pct": early_dist.round(2),
    "late_pct": late_dist.round(2),
    "change_pp": (
        late_dist - early_dist
    ).round(2)
})

drift.index = [
    "DOWN",
    "FLAT",
    "UP"
]

print("\n" + "=" * 80)
print("TARGET DISTRIBUTION DRIFT")
print("=" * 80)

display(drift)

print("\n" + "=" * 80)
print("AUDIT B COMPLETE")
print("=" * 80)

AUDIT B — TEMPORAL TARGET DRIFT

MONTHLY TARGET DISTRIBUTION


,year_month,DOWN_%,FLAT_%,UP_%
0,2025-01,3.47,10.40,86.13
1,2025-02,16.52,22.95,60.53
2,2025-03,21.39,30.80,47.80
3,2025-04,27.95,38.44,33.61
4,2025-05,33.54,39.31,27.15
5,2025-06,40.51,34.61,24.88
6,2025-07,31.36,28.10,40.55
7,2025-08,28.58,24.99,46.42
8,2025-09,21.50,20.74,57.77
9,2025-10,19.77,15.14,65.08



BEFORE 2026-01-01


,count,percentage
DOWN,NaN,NaN
FLAT,NaN,NaN
UP,NaN,NaN



FROM 2026-01-01


,count,percentage
DOWN,NaN,NaN
FLAT,NaN,NaN
UP,NaN,NaN



TARGET DISTRIBUTION DRIFT


,early_pct,late_pct,change_pp
DOWN,27.14,53.06,25.92
FLAT,24.53,26.60,2.07
UP,48.33,20.34,-27.99



AUDIT B COMPLETE


**BLOCK C — Feature Signal Audit**

In [6]:
# ================================================================
# AUDIT C — FEATURE → TARGET SIGNAL
# ================================================================

print("=" * 80)
print("AUDIT C — FEATURE PREDICTIVE SIGNAL")
print("=" * 80)

df_c = df_audit.copy()

# ------------------------------------------------
# EXPLICITLY BLOCK NON-FEATURE COLUMNS
# ------------------------------------------------

blocked = {
    "target",
    "target_label",
    "future_start",
    "future_end",
    "future_imp_3m",
    "future_impression_change_pct",
    "content_hash_id",
    "client_hash_id",
    "window_start",
    "window_end"
}

# Current-window numeric columns only
numeric_cols = df_c.select_dtypes(
    include=np.number
).columns.tolist()

feature_candidates = [
    c for c in numeric_cols
    if c not in blocked
]

print(
    f"\nCandidate numeric features: "
    f"{len(feature_candidates)}"
)

# ------------------------------------------------
# TARGET-WISE MEDIANS
# ------------------------------------------------

target_medians = (
    df_c
    .groupby("target")[feature_candidates]
    .median()
    .T
)

target_medians.columns = [
    "DOWN",
    "FLAT",
    "UP"
]

print("\n" + "=" * 80)
print("TARGET-WISE FEATURE MEDIANS")
print("=" * 80)

display(
    target_medians.round(3)
)

# ------------------------------------------------
# CORRELATION WITH TARGET
# ------------------------------------------------

corr_rows = []

for col in feature_candidates:

    x = pd.to_numeric(
        df_c[col],
        errors="coerce"
    )

    if x.nunique(dropna=True) < 2:
        continue

    corr = x.corr(
        df_c["target"],
        method="spearman"
    )

    corr_rows.append({
        "feature": col,
        "spearman_abs": abs(corr),
        "spearman": corr
    })

corr_df = (
    pd.DataFrame(corr_rows)
    .sort_values(
        "spearman_abs",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("TOP FEATURES BY TARGET ASSOCIATION")
print("=" * 80)

display(
    corr_df.head(20).round(4)
)

# ------------------------------------------------
# TARGET-WISE MEAN / MEDIAN DIFFERENCE
# ------------------------------------------------

signal_rows = []

for col in feature_candidates:

    grouped = (
        df_c
        .groupby("target")[col]
        .median()
        .reindex([0, 1, 2])
    )

    if grouped.isna().all():
        continue

    signal_rows.append({
        "feature": col,
        "DOWN_median": grouped.iloc[0],
        "FLAT_median": grouped.iloc[1],
        "UP_median": grouped.iloc[2],
        "max_class_gap": (
            grouped.max() - grouped.min()
        )
    })

signal_df = (
    pd.DataFrame(signal_rows)
    .sort_values(
        "max_class_gap",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("FEATURE CLASS-SEPARATION AUDIT")
print("=" * 80)

display(
    signal_df.head(20).round(3)
)

print("\n" + "=" * 80)
print("AUDIT C COMPLETE")
print("=" * 80)

AUDIT C — FEATURE PREDICTIVE SIGNAL

Candidate numeric features: 25

TARGET-WISE FEATURE MEDIANS


,DOWN,FLAT,UP
gsc_clicks_mean_3m,0.000,0.333,0.000
gsc_clicks_last,0.000,0.000,0.000
gsc_impressions_mean_3m,53.000,325.000,90.667
gsc_impressions_last,36.000,378.000,131.000
gsc_avg_position_mean_3m,8.317,8.185,9.060
gsc_avg_position_last,7.493,7.906,8.570
ga4_total_engagement_sec_mean_3m,0.000,0.000,0.000
ga4_total_engagement_sec_last,0.000,0.000,0.000
sessions_organic_mean_3m,0.000,0.000,0.000
sessions_organic_last,0.000,0.000,0.000



TOP FEATURES BY TARGET ASSOCIATION


,feature,spearman_abs,spearman
0,gsc_avg_position_missing_last,0.1555,-0.1555
1,gsc_impressions_last_30d,0.1193,0.1193
2,gsc_impressions_last,0.1193,0.1193
3,ctr_last,0.1132,0.1132
4,gsc_clicks_last,0.0994,0.0994
5,ctr_mean_3m,0.0881,0.0881
6,gsc_impressions_prev_30d,0.0785,-0.0785
7,gsc_avg_position_last,0.0663,0.0663
8,ga4_total_engagement_sec_mean_3m,0.0647,-0.0647
9,gsc_clicks_mean_3m,0.0618,0.0618



FEATURE CLASS-SEPARATION AUDIT


,feature,DOWN_median,FLAT_median,UP_median,max_class_gap
0,gsc_impressions_last,36.000,378.000,131.000,342.000
1,gsc_impressions_last_30d,36.000,378.000,131.000,342.000
2,gsc_impressions_mean_3m,53.000,325.000,90.667,272.000
3,current_imp_3m,53.000,325.000,90.667,272.000
4,gsc_impressions_prev_30d,37.000,212.000,30.000,182.000
5,gsc_avg_position_last,7.493,7.906,8.570,1.077
6,gsc_avg_position_mean_3m,8.317,8.185,9.060,0.876
7,gsc_clicks_mean_3m,0.000,0.333,0.000,0.333
8,ctr_mean_3m,0.000,0.001,0.000,0.001
9,sessions_organic_mean_3m,0.000,0.000,0.000,0.000



AUDIT C COMPLETE


**BLOCK D — Feature Distribution Drift**

In [7]:
# ================================================================
# AUDIT D — FEATURE DISTRIBUTION DRIFT
# ================================================================

from scipy.stats import ks_2samp

print("=" * 80)
print("AUDIT D — FEATURE DISTRIBUTION DRIFT")
print("=" * 80)

df_d = df_audit.copy()

cutoff = pd.Timestamp("2026-01-01")

train_period = df_d[
    df_d["window_start"] < cutoff
]

test_period = df_d[
    df_d["window_start"] >= cutoff
]

blocked = {
    "target",
    "target_label",
    "future_start",
    "future_end",
    "future_imp_3m",
    "future_impression_change_pct",
    "content_hash_id",
    "client_hash_id",
    "window_start",
    "window_end"
}

numeric_cols = df_d.select_dtypes(
    include=np.number
).columns.tolist()

features = [
    c for c in numeric_cols
    if c not in blocked
]

rows = []

for col in features:

    train_values = pd.to_numeric(
        train_period[col],
        errors="coerce"
    ).dropna()

    test_values = pd.to_numeric(
        test_period[col],
        errors="coerce"
    ).dropna()

    if len(train_values) < 20 or len(test_values) < 20:
        continue

    # Limit sample size for speed
    n = min(
        20000,
        len(train_values),
        len(test_values)
    )

    train_sample = train_values.sample(
        n=n,
        random_state=42
    )

    test_sample = test_values.sample(
        n=n,
        random_state=42
    )

    statistic, p_value = ks_2samp(
        train_sample,
        test_sample
    )

    rows.append({
        "feature": col,
        "train_median": train_values.median(),
        "test_median": test_values.median(),
        "median_change_pct": (
            (
                test_values.median()
                - train_values.median()
            )
            /
            (
                abs(train_values.median())
                + 1e-9
            )
            * 100
        ),
        "ks_statistic": statistic,
        "p_value": p_value
    })

drift_df = (
    pd.DataFrame(rows)
    .sort_values(
        "ks_statistic",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("TOP FEATURE DISTRIBUTION DRIFT")
print("=" * 80)

display(
    drift_df.head(25).round(4)
)

print("\n" + "=" * 80)
print("AUDIT D COMPLETE")
print("=" * 80)

AUDIT D — FEATURE DISTRIBUTION DRIFT

TOP FEATURE DISTRIBUTION DRIFT


,feature,train_median,test_median,median_change_pct,ks_statistic,p_value
0,sessions_organic_mean_3m,0.0000,0.0000,0.0000,0.1039,0.0000
1,sessions_organic_last,0.0000,0.0000,0.0000,0.1030,0.0000
2,ga4_total_engagement_sec_mean_3m,0.0000,0.0000,0.0000,0.0869,0.0000
3,ga4_total_engagement_sec_last,0.0000,0.0000,0.0000,0.0815,0.0000
4,ctr_mean_3m,0.0000,0.0000,0.0000,0.0686,0.0000
5,gsc_avg_position_missing_mean_3m,0.0000,0.0000,0.0000,0.0608,0.0000
6,current_imp_3m,117.3333,93.3333,-20.4545,0.0598,0.0000
7,gsc_impressions_mean_3m,117.3333,93.3333,-20.4545,0.0598,0.0000
8,ctr_last,0.0000,0.0000,0.0000,0.0563,0.0000
9,gsc_avg_position_mean_3m,8.6904,8.2318,-5.2769,0.0561,0.0000



AUDIT D COMPLETE


**BLOCK E — Simple Rule Baseline**

In [8]:
# ================================================================
# AUDIT E — SIMPLE BASELINE SIGNAL
# ================================================================

print("=" * 80)
print("AUDIT E — SIMPLE TREND BASELINE")
print("=" * 80)

df_e = df_audit.copy()

required = [
    "gsc_impressions_mean_3m",
    "gsc_impressions_last",
    "target"
]

available = [
    c for c in required
    if c in df_e.columns
]

print("\nAvailable trend columns:")
for c in available:
    print(" -", c)

if (
    "gsc_impressions_mean_3m" in df_e.columns
    and
    "gsc_impressions_last" in df_e.columns
):

    mean_imp = pd.to_numeric(
        df_e["gsc_impressions_mean_3m"],
        errors="coerce"
    )

    last_imp = pd.to_numeric(
        df_e["gsc_impressions_last"],
        errors="coerce"
    )

    trend_pct = (
        (last_imp - mean_imp)
        /
        (mean_imp.abs() + 1e-9)
        * 100
    )

    df_e["current_trend_pct"] = trend_pct

    print("\n" + "=" * 80)
    print("CURRENT 90-DAY TREND BY FUTURE TARGET")
    print("=" * 80)

    display(
        df_e
        .groupby("target")["current_trend_pct"]
        .agg([
            "count",
            "median",
            "mean",
            "min",
            "max"
        ])
        .round(2)
    )

    # ------------------------------------------------------------
    # SIMPLE RULE
    #
    # Negative current trend -> DOWN
    # Positive current trend -> UP
    # Otherwise FLAT
    # ------------------------------------------------------------

    simple_pred = np.select(
        [
            trend_pct <= -10,
            trend_pct >= 10
        ],
        [
            0,
            2
        ],
        default=1
    )

    valid = (
        trend_pct.notna()
        &
        df_e["target"].notna()
    )

    simple_accuracy = (
        simple_pred[valid]
        ==
        df_e.loc[valid, "target"].to_numpy()
    ).mean()

    print("\n" + "=" * 80)
    print("SIMPLE TREND BASELINE")
    print("=" * 80)

    print(
        f"Accuracy: "
        f"{simple_accuracy*100:.2f}%"
    )

else:

    print(
        "\nRequired impression trend columns "
        "are not available."
    )

print("\n" + "=" * 80)
print("AUDIT E COMPLETE")
print("=" * 80)

AUDIT E — SIMPLE TREND BASELINE

Available trend columns:
 - gsc_impressions_mean_3m
 - gsc_impressions_last
 - target

CURRENT 90-DAY TREND BY FUTURE TARGET


,count,median,mean,min,max
target,,,,,
0,206390,-15.31,-5.92,-100.0,200.0
1,156653,16.67,23.06,-100.0,200.0
2,263793,44.06,50.76,-100.0,200.0



SIMPLE TREND BASELINE
Accuracy: 51.94%

AUDIT E COMPLETE


**Audit F**

**AUDIT F — LOW-VOLUME + EXTREME TARGET QUALITY**

In [9]:
# =============================================================================
# AUDIT F — LOW-VOLUME + EXTREME TARGET QUALITY
# =============================================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("AUDIT F — LOW-VOLUME + EXTREME TARGET QUALITY")
print("=" * 80)

# -----------------------------------------------------------------------------
# 1. LOAD FINAL ROLLING DATASET
# -----------------------------------------------------------------------------

file_path = "/content/finalrolling90window.parquet"

df_f = pd.read_parquet(file_path)

print(f"\nRows : {len(df_f):,}")
print(f"Cols : {df_f.shape[1]:,}")

# -----------------------------------------------------------------------------
# 2. REQUIRED COLUMNS
# -----------------------------------------------------------------------------

required = [
    "current_imp_3m",
    "future_imp_3m",
    "future_impression_change_pct",
    "target"
]

missing = [
    c for c in required
    if c not in df_f.columns
]

if missing:
    raise KeyError(
        "Required columns missing:\n" +
        "\n".join(missing)
    )

# Numeric conversion
for col in [
    "current_imp_3m",
    "future_imp_3m",
    "future_impression_change_pct"
]:
    df_f[col] = pd.to_numeric(
        df_f[col],
        errors="coerce"
    )

# -----------------------------------------------------------------------------
# 3. TARGET LABELS
# -----------------------------------------------------------------------------

target_names = {
    0: "DOWN",
    1: "FLAT",
    2: "UP"
}

df_f["target_label"] = (
    df_f["target"]
    .map(target_names)
)

# -----------------------------------------------------------------------------
# 4. LOW-VOLUME BUCKETS
# -----------------------------------------------------------------------------

df_f["volume_bucket"] = pd.cut(
    df_f["current_imp_3m"],
    bins=[
        -np.inf,
        0,
        10,
        25,
        50,
        100,
        500,
        1000,
        np.inf
    ],
    labels=[
        "0",
        "1-10",
        "11-25",
        "26-50",
        "51-100",
        "101-500",
        "501-1000",
        "1000+"
    ],
    right=True
)

# -----------------------------------------------------------------------------
# 5. TARGET DISTRIBUTION BY CURRENT IMPRESSION VOLUME
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("TARGET DISTRIBUTION BY CURRENT 3M IMPRESSION VOLUME")
print("=" * 80)

volume_target = pd.crosstab(
    df_f["volume_bucket"],
    df_f["target_label"],
    normalize="index"
).mul(100)

volume_target = (
    volume_target
    .reindex(
        columns=["DOWN", "FLAT", "UP"],
        fill_value=0
    )
    .round(2)
)

volume_counts = (
    df_f["volume_bucket"]
    .value_counts(sort=False)
    .rename("rows")
)

volume_audit = (
    volume_counts
    .to_frame()
    .join(volume_target)
)

display(volume_audit)

# -----------------------------------------------------------------------------
# 6. EXTREME CHANGE COUNTS BY VOLUME
# -----------------------------------------------------------------------------

df_f["extreme_down_90"] = (
    df_f["future_impression_change_pct"] <= -90
)

df_f["extreme_up_500"] = (
    df_f["future_impression_change_pct"] >= 500
)

df_f["extreme_up_1000"] = (
    df_f["future_impression_change_pct"] >= 1000
)

extreme_by_volume = (
    df_f
    .groupby("volume_bucket", observed=False)
    .agg(
        rows=("target", "size"),
        down_90=("extreme_down_90", "sum"),
        up_500=("extreme_up_500", "sum"),
        up_1000=("extreme_up_1000", "sum")
    )
)

for col in [
    "down_90",
    "up_500",
    "up_1000"
]:
    extreme_by_volume[col + "_pct"] = (
        extreme_by_volume[col]
        / extreme_by_volume["rows"]
        * 100
    ).round(2)

print("\n" + "=" * 80)
print("EXTREME CHANGE BY CURRENT IMPRESSION VOLUME")
print("=" * 80)

display(extreme_by_volume.round(2))

# -----------------------------------------------------------------------------
# 7. OVERALL EXTREME CHANGE CONTRIBUTION
# -----------------------------------------------------------------------------

total_rows = len(df_f)

extreme_summary = pd.DataFrame({
    "condition": [
        "Change <= -90%",
        "Change >= +500%",
        "Change >= +1000%"
    ],
    "rows": [
        df_f["extreme_down_90"].sum(),
        df_f["extreme_up_500"].sum(),
        df_f["extreme_up_1000"].sum()
    ]
})

extreme_summary["percentage"] = (
    extreme_summary["rows"]
    / total_rows
    * 100
).round(2)

print("\n" + "=" * 80)
print("OVERALL EXTREME CHANGE SUMMARY")
print("=" * 80)

display(extreme_summary)

# -----------------------------------------------------------------------------
# 8. EXTREME CHANGES — WHAT VOLUME DO THEY COME FROM?
# -----------------------------------------------------------------------------

extreme_volume_contribution = pd.DataFrame({
    "condition": [
        "Change <= -90%",
        "Change >= +500%",
        "Change >= +1000%"
    ],
    "rows": [
        df_f.loc[
            df_f["extreme_down_90"],
            "current_imp_3m"
        ].notna().sum(),

        df_f.loc[
            df_f["extreme_up_500"],
            "current_imp_3m"
        ].notna().sum(),

        df_f.loc[
            df_f["extreme_up_1000"],
            "current_imp_3m"
        ].notna().sum()
    ]
})

# -----------------------------------------------------------------------------
# 9. TARGET DISTRIBUTION — LOW VS NORMAL VOLUME
# -----------------------------------------------------------------------------

df_f["low_volume"] = (
    df_f["current_imp_3m"] < 100
)

low_high = (
    pd.crosstab(
        df_f["low_volume"],
        df_f["target_label"],
        normalize="index"
    )
    .mul(100)
    .round(2)
)

low_high = low_high.reindex(
    columns=["DOWN", "FLAT", "UP"],
    fill_value=0
)

print("\n" + "=" * 80)
print("LOW VOLUME (<100) VS NORMAL VOLUME")
print("=" * 80)

display(low_high)

# -----------------------------------------------------------------------------
# 10. MEDIAN CURRENT/FUTURE IMPRESSIONS BY TARGET
# -----------------------------------------------------------------------------

target_volume = (
    df_f
    .groupby("target_label")
    .agg(
        rows=("target", "size"),
        current_imp_median=("current_imp_3m", "median"),
        future_imp_median=("future_imp_3m", "median"),
        change_median=(
            "future_impression_change_pct",
            "median"
        )
    )
)

print("\n" + "=" * 80)
print("VOLUME + CHANGE BY TARGET")
print("=" * 80)

display(target_volume.round(2))

# -----------------------------------------------------------------------------
# 11. AUDIT CONCLUSION — INFORMATION ONLY
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("AUDIT F COMPLETE")
print("=" * 80)

print("""
This audit does NOT modify the dataset.

It determines whether:
1. Very-low-impression pages dominate extreme percentage changes.
2. The DOWN/FLAT/UP target is strongly dependent on current volume.
3. Extreme percentage changes are mainly a low-volume artifact.
""")

AUDIT F — LOW-VOLUME + EXTREME TARGET QUALITY

Rows : 626,836
Cols : 36

TARGET DISTRIBUTION BY CURRENT 3M IMPRESSION VOLUME


,rows,DOWN,FLAT,UP
volume_bucket,,,,
0,0,NaN,NaN,NaN
1-10,154533,45.67,14.60,39.73
11-25,51026,34.45,16.24,49.31
26-50,45917,30.38,19.15,50.48
51-100,52601,27.86,22.67,49.47
101-500,140925,27.55,26.76,45.69
501-1000,57587,27.96,30.78,41.25
1000+,124247,27.93,39.96,32.11



EXTREME CHANGE BY CURRENT IMPRESSION VOLUME


,rows,down_90,up_500,up_1000,down_90_pct,up_500_pct,up_1000_pct
volume_bucket,,,,,,,
0,0,0,0,0,NaN,NaN,NaN
1-10,154533,39336,27286,17706,25.45,17.66,11.46
11-25,51026,7002,7698,3563,13.72,15.09,6.98
26-50,45917,4253,5002,2082,9.26,10.89,4.53
51-100,52601,3314,4380,1744,6.30,8.33,3.32
101-500,140925,4794,8009,2908,3.40,5.68,2.06
501-1000,57587,1200,2033,504,2.08,3.53,0.88
1000+,124247,2615,1732,321,2.10,1.39,0.26



OVERALL EXTREME CHANGE SUMMARY


...


LOW VOLUME (<100) VS NORMAL VOLUME


target_label,DOWN,FLAT,UP
low_volume,,,
False,27.77,32.55,39.68
True,38.40,16.95,44.64



VOLUME + CHANGE BY TARGET


,rows,current_imp_median,future_imp_median,change_median
target_label,,,,
DOWN,206390,53.00,11.00,-73.33
FLAT,156653,325.00,341.67,3.58
UP,263793,90.67,314.67,192.45



AUDIT F COMPLETE

This audit does NOT modify the dataset.

It determines whether:
1. Very-low-impression pages dominate extreme percentage changes.
2. The DOWN/FLAT/UP target is strongly dependent on current volume.
3. Extreme percentage changes are mainly a low-volume artifact.



**Checkups:**
target correctness
target distribution
extreme percentage changes
low-volume instability
absolute vs percentage change
target stability by volume
temporal target drift
feature predictive signal
train/test feature drift
suspicious/leakage columns
missing/infinite values
duplicate page-window records
simple trend baseline
target-vs-volume relationship
alternative volume-aware target sensitivity — audit only

In [10]:
# =============================================================================
# FINAL PRE-MODEL COMPREHENSIVE AUDIT
# Dataset: finalrolling90window.parquet
#
# PURPOSE:
#   One final audit before model training.
#
# IMPORTANT:
#   This block DOES NOT modify the dataset.
#   Current frozen target remains unchanged:
#
#       0 = DOWN  <= -30%
#       1 = FLAT  > -30% and < +50%
#       2 = UP    >= +50%
#
# =============================================================================

import pandas as pd
import numpy as np
from scipy.stats import ks_2samp

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

# =============================================================================
# 0. LOAD FINAL DATASET
# =============================================================================

print("=" * 90)
print("FINAL PRE-MODEL COMPREHENSIVE AUDIT")
print("=" * 90)

file_path = "/content/finalrolling90window.parquet"

df = pd.read_parquet(file_path).copy()

print(f"\nRows    : {len(df):,}")
print(f"Columns : {df.shape[1]:,}")

# =============================================================================
# 1. BASIC STRUCTURE
# =============================================================================

print("\n" + "=" * 90)
print("AUDIT 1 — BASIC DATASET STRUCTURE")
print("=" * 90)

required_core = [
    "content_hash_id",
    "window_start",
    "window_end",
    "future_start",
    "future_end",
    "current_imp_3m",
    "future_imp_3m",
    "future_impression_change_pct",
    "target",
    "target_label"
]

missing_core = [
    c for c in required_core
    if c not in df.columns
]

if missing_core:
    raise KeyError(
        "Required columns missing:\n" +
        "\n".join(missing_core)
    )

for col in [
    "window_start",
    "window_end",
    "future_start",
    "future_end"
]:
    df[col] = pd.to_datetime(
        df[col],
        errors="coerce"
    )

print("✓ All core columns present.")

# =============================================================================
# 2. DUPLICATE PAGE-WINDOW CHECK
# =============================================================================

print("\n" + "=" * 90)
print("AUDIT 2 — DUPLICATE PAGE-WINDOW CHECK")
print("=" * 90)

duplicate_mask = df.duplicated(
    subset=[
        "content_hash_id",
        "window_start",
        "window_end"
    ],
    keep=False
)

duplicate_rows = int(duplicate_mask.sum())

print(
    f"Duplicate page-window rows : "
    f"{duplicate_rows:,}"
)

if duplicate_rows == 0:
    print("✓ No duplicate page-window records.")
else:
    print("⚠ Duplicate page-window records found.")

# =============================================================================
# 3. TARGET CONSISTENCY
# =============================================================================

print("\n" + "=" * 90)
print("AUDIT 3 — TARGET CONSISTENCY")
print("=" * 90)

change = pd.to_numeric(
    df["future_impression_change_pct"],
    errors="coerce"
)

expected_target = np.select(
    [
        change <= -30,
        (change > -30) & (change < 50),
        change >= 50
    ],
    [
        0,
        1,
        2
    ],
    default=-1
).astype(np.int8)

target_numeric = pd.to_numeric(
    df["target"],
    errors="coerce"
)

target_mismatch = (
    target_numeric != expected_target
)

mismatch_count = int(
    target_mismatch.sum()
)

print(
    f"Target mismatches : "
    f"{mismatch_count:,}"
)

if mismatch_count == 0:
    print("✓ Target exactly matches frozen rule.")
else:
    print("🔴 TARGET ERROR — target does not match frozen rule.")

# =============================================================================
# 4. TARGET DISTRIBUTION
# =============================================================================

print("\n" + "=" * 90)
print("AUDIT 4 — TARGET DISTRIBUTION")
print("=" * 90)

target_dist = (
    df["target"]
    .value_counts()
    .reindex([0, 1, 2], fill_value=0)
)

target_pct = (
    target_dist / len(df) * 100
)

target_table = pd.DataFrame({
    "target": [0, 1, 2],
    "label": ["DOWN", "FLAT", "UP"],
    "count": target_dist.values,
    "percentage": target_pct.values
})

display(target_table)

# =============================================================================
# 5. EXTREME CHANGE AUDIT
# =============================================================================

print("\n" + "=" * 90)
print("AUDIT 5 — EXTREME TARGET CHANGE")
print("=" * 90)

extreme_conditions = {
    "Change <= -90%": change <= -90,
    "Change >= +500%": change >= 500,
    "Change >= +1000%": change >= 1000
}

for name, mask in extreme_conditions.items():

    count = int(mask.sum())

    print(
        f"{name:<22}: "
        f"{count:>10,} "
        f"({count / len(df) * 100:.2f}%)"
    )

print("\nChange distribution:")

print(
    change.describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99
        ]
    )
)

# =============================================================================
# 6. LOW-VOLUME INSTABILITY
# =============================================================================

print("\n" + "=" * 90)
print("AUDIT 6 — LOW-VOLUME TARGET INSTABILITY")
print("=" * 90)

volume = pd.to_numeric(
    df["current_imp_3m"],
    errors="coerce"
).fillna(0)

df["_volume_bucket"] = pd.cut(
    volume,
    bins=[
        -np.inf,
        0,
        10,
        25,
        50,
        100,
        500,
        1000,
        np.inf
    ],
    labels=[
        "0",
        "1-10",
        "11-25",
        "26-50",
        "51-100",
        "101-500",
        "501-1000",
        "1000+"
    ],
    right=True
)

volume_target = pd.crosstab(
    df["_volume_bucket"],
    df["target"],
    normalize="index"
).reindex(
    columns=[0, 1, 2],
    fill_value=0
) * 100

volume_target.columns = [
    "DOWN_%",
    "FLAT_%",
    "UP_%"
]

volume_counts = (
    df["_volume_bucket"]
    .value_counts()
    .sort_index()
    .rename("rows")
)

volume_target = volume_counts.to_frame().join(
    volume_target
)

display(volume_target)

# =============================================================================
# 7. EXTREME CHANGES BY VOLUME
# =============================================================================

print("\n" + "=" * 90)
print("AUDIT 7 — EXTREME CHANGE BY VOLUME")
print("=" * 90)

extreme_by_volume = df.groupby(
    "_volume_bucket",
    observed=True
).agg(
    rows=("target", "size"),
    down_90=("future_impression_change_pct",
             lambda x: (x <= -90).sum()),
    up_500=("future_impression_change_pct",
            lambda x: (x >= 500).sum()),
    up_1000=("future_impression_change_pct",
             lambda x: (x >= 1000).sum())
)

for col in [
    "down_90",
    "up_500",
    "up_1000"
]:
    extreme_by_volume[col + "_pct"] = (
        extreme_by_volume[col]
        / extreme_by_volume["rows"]
        * 100
    )

display(extreme_by_volume)

# =============================================================================
# 8. ABSOLUTE VS PERCENTAGE CHANGE
# =============================================================================

print("\n" + "=" * 90)
print("AUDIT 8 — ABSOLUTE VS PERCENTAGE CHANGE")
print("=" * 90)

future_imp = pd.to_numeric(
    df["future_imp_3m"],
    errors="coerce"
)

absolute_change = (
    future_imp - volume
)

df["_absolute_imp_change"] = absolute_change

absolute_stats = pd.DataFrame({
    "target": [0, 1, 2],
    "label": ["DOWN", "FLAT", "UP"],
    "rows": [
        (df["target"] == 0).sum(),
        (df["target"] == 1).sum(),
        (df["target"] == 2).sum()
    ],
    "current_median": [
        volume[df["target"] == 0].median(),
        volume[df["target"] == 1].median(),
        volume[df["target"] == 2].median()
    ],
    "future_median": [
        future_imp[df["target"] == 0].median(),
        future_imp[df["target"] == 1].median(),
        future_imp[df["target"] == 2].median()
    ],
    "absolute_change_median": [
        absolute_change[df["target"] == 0].median(),
        absolute_change[df["target"] == 1].median(),
        absolute_change[df["target"] == 2].median()
    ],
    "percentage_change_median": [
        change[df["target"] == 0].median(),
        change[df["target"] == 1].median(),
        change[df["target"] == 2].median()
    ]
})

display(absolute_stats)

# =============================================================================
# 9. TARGET BY LOW-VOLUME VS NORMAL
# =============================================================================

print("\n" + "=" * 90)
print("AUDIT 9 — LOW VOLUME VS NORMAL VOLUME")
print("=" * 90)

low_volume = volume < 100

low_volume_target = pd.crosstab(
    low_volume,
    df["target"],
    normalize="index"
) * 100

low_volume_target.columns = [
    "DOWN_%",
    "FLAT_%",
    "UP_%"
]

low_volume_target.index = [
    "Normal volume (>=100)",
    "Low volume (<100)"
]

display(low_volume_target)

# =============================================================================
# 10. TARGET STABILITY / BORDERLINE CASES
# =============================================================================

print("\n" + "=" * 90)
print("AUDIT 10 — TARGET BORDERLINE / STABILITY CHECK")
print("=" * 90)

border_masks = {
    "Near DOWN boundary (-40 to -20)": (
        (change > -40) & (change < -20)
    ),
    "Near UP boundary (+40 to +60)": (
        (change >= 40) & (change <= 60)
    ),
    "Extreme DOWN (<= -90)": (
        change <= -90
    ),
    "Extreme UP (>= +500)": (
        change >= 500
    )
}

for name, mask in border_masks.items():

    subset = df.loc[mask]

    print(
        f"\n{name}: "
        f"{len(subset):,} rows"
    )

    if len(subset) > 0:

        print(
            subset["target"]
            .value_counts(normalize=True)
            .reindex([0, 1, 2], fill_value=0)
            .mul(100)
            .round(2)
            .to_dict()
        )

# =============================================================================
# 11. TEMPORAL TARGET DRIFT
# =============================================================================

print("\n" + "=" * 90)
print("AUDIT 11 — TEMPORAL TARGET DRIFT")
print("=" * 90)

df["_year_month"] = (
    df["window_start"]
    .dt.to_period("M")
    .astype(str)
)

monthly_target = pd.crosstab(
    df["_year_month"],
    df["target"],
    normalize="index"
).reindex(
    columns=[0, 1, 2],
    fill_value=0
) * 100

monthly_target.columns = [
    "DOWN_%",
    "FLAT_%",
    "UP_%"
]

display(
    monthly_target.round(2)
)

# =============================================================================
# 12. TRAIN / TEST DISTRIBUTION FOR FINAL TEMPORAL SPLIT
# =============================================================================

print("\n" + "=" * 90)
print("AUDIT 12 — FINAL TEMPORAL SPLIT DISTRIBUTION")
print("=" * 90)

cutoff = pd.Timestamp("2026-01-01")

train_mask = (
    df["window_start"] < cutoff
)

test_mask = (
    df["window_start"] >= cutoff
)

train_df = df.loc[train_mask]
test_df = df.loc[test_mask]

print(
    f"TRAIN : {len(train_df):,} rows "
    f"({len(train_df) / len(df) * 100:.2f}%)"
)

print(
    f"TEST  : {len(test_df):,} rows "
    f"({len(test_df) / len(df) * 100:.2f}%)"
)

train_target = (
    train_df["target"]
    .value_counts(normalize=True)
    .reindex([0, 1, 2], fill_value=0)
    * 100
)

test_target = (
    test_df["target"]
    .value_counts(normalize=True)
    .reindex([0, 1, 2], fill_value=0)
    * 100
)

split_target = pd.DataFrame({
    "TRAIN_%": train_target,
    "TEST_%": test_target
})

split_target["DRIFT_pp"] = (
    split_target["TEST_%"]
    - split_target["TRAIN_%"]
)

split_target.index = [
    "DOWN",
    "FLAT",
    "UP"
]

display(
    split_target.round(2)
)

# =============================================================================
# 13. FEATURE LEAKAGE AUDIT
# =============================================================================

print("\n" + "=" * 90)
print("AUDIT 13 — FEATURE / TARGET / FUTURE LEAKAGE")
print("=" * 90)

future_keywords = [
    "future",
    "target",
    "label"
]

blocked_exact = {
    "target",
    "target_label"
}

feature_candidates = []

blocked_features = []

for col in df.columns:

    col_lower = col.lower()

    if col in [
        "_volume_bucket",
        "_absolute_imp_change",
        "_year_month"
    ]:
        continue

    if col in blocked_exact:
        blocked_features.append(col)
        continue

    if any(
        keyword in col_lower
        for keyword in future_keywords
    ):
        blocked_features.append(col)
        continue

    feature_candidates.append(col)

print(
    f"Candidate input columns : "
    f"{len(feature_candidates)}"
)

print("\nBlocked columns:")

for col in blocked_features:
    print(" -", col)

# =============================================================================
# 14. SUSPICIOUS COLUMN NAME AUDIT
# =============================================================================

print("\n" + "=" * 90)
print("AUDIT 14 — SUSPICIOUS FEATURE NAME CHECK")
print("=" * 90)

suspicious_terms = [
    "future",
    "target",
    "label",
    "change_pct",
    "change_percent"
]

suspicious = [
    col
    for col in feature_candidates
    if any(
        term in col.lower()
        for term in suspicious_terms
    )
]

if suspicious:
    print("⚠ Suspicious candidate features:")
    for col in suspicious:
        print(" -", col)
else:
    print("✓ No suspicious candidate feature names.")

# =============================================================================
# 15. MISSING / INFINITE VALUES
# =============================================================================

print("\n" + "=" * 90)
print("AUDIT 15 — MISSING / INFINITE VALUES")
print("=" * 90)

numeric_candidates = df[
    feature_candidates
].select_dtypes(
    include=np.number
).columns.tolist()

missing_train = (
    train_df[numeric_candidates]
    .isna()
    .sum()
    .sum()
)

missing_test = (
    test_df[numeric_candidates]
    .isna()
    .sum()
    .sum()
)

inf_train = np.isinf(
    train_df[numeric_candidates]
    .to_numpy()
).sum()

inf_test = np.isinf(
    test_df[numeric_candidates]
    .to_numpy()
).sum()

print(
    f"Train missing values : {missing_train:,}"
)

print(
    f"Test missing values  : {missing_test:,}"
)

print(
    f"Train infinite values: {inf_train:,}"
)

print(
    f"Test infinite values : {inf_test:,}"
)

# =============================================================================
# 16. FEATURE DISTRIBUTION DRIFT
# =============================================================================

print("\n" + "=" * 90)
print("AUDIT 16 — TRAIN / TEST FEATURE DRIFT")
print("=" * 90)

drift_results = []

for col in numeric_candidates:

    train_values = (
        train_df[col]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    test_values = (
        test_df[col]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if len(train_values) == 0 or len(test_values) == 0:
        continue

    ks_stat, p_value = ks_2samp(
        train_values,
        test_values
    )

    train_median = train_values.median()
    test_median = test_values.median()

    if train_median != 0:
        median_change = (
            (test_median - train_median)
            / abs(train_median)
            * 100
        )
    else:
        median_change = 0.0

    drift_results.append({
        "feature": col,
        "train_median": train_median,
        "test_median": test_median,
        "median_change_pct": median_change,
        "KS": ks_stat,
        "p_value": p_value
    })

drift_df = pd.DataFrame(
    drift_results
)

if len(drift_df) > 0:

    drift_df = (
        drift_df
        .sort_values(
            "KS",
            ascending=False
        )
        .reset_index(drop=True)
    )

    display(
        drift_df.head(25).round(4)
    )

# =============================================================================
# 17. FEATURE PREDICTIVE SIGNAL
# =============================================================================

print("\n" + "=" * 90)
print("AUDIT 17 — FEATURE PREDICTIVE SIGNAL")
print("=" * 90)

signal_results = []

for col in numeric_candidates:

    try:

        corr = (
            df[[col, "target"]]
            .corr(
                method="spearman"
            )
            .iloc[0, 1]
        )

        if pd.notna(corr):

            signal_results.append({
                "feature": col,
                "spearman": corr,
                "abs_spearman": abs(corr)
            })

    except Exception:
        pass

signal_df = pd.DataFrame(
    signal_results
)

if len(signal_df) > 0:

    signal_df = (
        signal_df
        .sort_values(
            "abs_spearman",
            ascending=False
        )
        .reset_index(drop=True)
    )

    display(
        signal_df.head(20).round(4)
    )

# =============================================================================
# 18. SIMPLE TREND BASELINE
# =============================================================================

print("\n" + "=" * 90)
print("AUDIT 18 — SIMPLE TREND BASELINE")
print("=" * 90)

trend_feature = None

for candidate in [
    "gsc_impressions_last",
    "current_imp_3m",
    "gsc_impressions_mean_3m"
]:

    if candidate in df.columns:

        trend_feature = candidate
        break

if trend_feature is not None:

    trend = pd.to_numeric(
        df[trend_feature],
        errors="coerce"
    )

    # Simple rule:
    # compare last available value with 3M mean where possible
    if (
        "gsc_impressions_last" in df.columns
        and "gsc_impressions_mean_3m" in df.columns
    ):

        baseline_pred = np.select(
            [
                df["gsc_impressions_last"]
                < df["gsc_impressions_mean_3m"] * 0.70,

                df["gsc_impressions_last"]
                >= df["gsc_impressions_mean_3m"] * 1.50
            ],
            [
                0,
                2
            ],
            default=1
        )

        baseline_accuracy = (
            baseline_pred == df["target"]
        ).mean()

        print(
            f"Trend baseline accuracy : "
            f"{baseline_accuracy * 100:.2f}%"
        )

    else:

        print(
            "Trend baseline skipped: "
            "required trend columns unavailable."
        )

else:

    print(
        "Trend baseline skipped: "
        "no suitable trend feature."
    )

# =============================================================================
# 19. TARGET VS CURRENT VOLUME RELATIONSHIP
# =============================================================================

print("\n" + "=" * 90)
print("AUDIT 19 — TARGET VS CURRENT IMPRESSION VOLUME")
print("=" * 90)

target_volume = (
    df.groupby("target")
    .agg(
        rows=("target", "size"),
        current_median=(
            "current_imp_3m",
            "median"
        ),
        current_mean=(
            "current_imp_3m",
            "mean"
        ),
        future_median=(
            "future_imp_3m",
            "median"
        ),
        change_median=(
            "future_impression_change_pct",
            "median"
        )
    )
)

target_volume.index = [
    "DOWN",
    "FLAT",
    "UP"
]

display(
    target_volume.round(3)
)

# =============================================================================
# 20. CLIENT / PAGE TEMPORAL OVERLAP
# =============================================================================

print("\n" + "=" * 90)
print("AUDIT 20 — PAGE / CLIENT TEMPORAL OVERLAP")
print("=" * 90)

train_pages = set(
    train_df["content_hash_id"].dropna().unique()
)

test_pages = set(
    test_df["content_hash_id"].dropna().unique()
)

page_overlap = (
    len(train_pages & test_pages)
)

print(
    f"Train pages : {len(train_pages):,}"
)

print(
    f"Test pages  : {len(test_pages):,}"
)

print(
    f"Page overlap: {page_overlap:,}"
)

if "client_hash_id" in df.columns:

    train_clients = set(
        train_df["client_hash_id"]
        .dropna()
        .unique()
    )

    test_clients = set(
        test_df["client_hash_id"]
        .dropna()
        .unique()
    )

    client_overlap = (
        len(train_clients & test_clients)
    )

    print(
        f"\nTrain clients : "
        f"{len(train_clients):,}"
    )

    print(
        f"Test clients  : "
        f"{len(test_clients):,}"
    )

    print(
        f"Client overlap: "
        f"{client_overlap:,}"
    )

# =============================================================================
# 21. FINAL AUDIT SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print("FINAL AUDIT SUMMARY")
print("=" * 90)

print(
    f"Rows                         : {len(df):,}"
)

print(
    f"Target mismatches            : {mismatch_count:,}"
)

print(
    f"Duplicate page-window rows   : {duplicate_rows:,}"
)

print(
    f"Train rows                   : {len(train_df):,}"
)

print(
    f"Test rows                    : {len(test_df):,}"
)

print(
    f"Candidate numeric features   : {len(numeric_candidates):,}"
)

print(
    f"Train missing values         : {missing_train:,}"
)

print(
    f"Test missing values          : {missing_test:,}"
)

print(
    f"Train infinite values        : {inf_train:,}"
)

print(
    f"Test infinite values         : {inf_test:,}"
)

print("\n" + "-" * 90)
print("IMPORTANT FINDINGS")
print("-" * 90)

# Target
if mismatch_count == 0:
    print("✓ TARGET: Frozen target rule is internally correct.")
else:
    print("🔴 TARGET: Target mismatch detected.")

# Low volume
low_volume_extreme = (
    df.loc[
        volume < 100,
        "future_impression_change_pct"
    ]
)

if len(low_volume_extreme) > 0:

    low_extreme_pct = (
        (
            (low_volume_extreme <= -90)
            |
            (low_volume_extreme >= 500)
        ).mean()
        * 100
    )

    print(
        f"⚠ LOW VOLUME: "
        f"{low_extreme_pct:.2f}% of <100-impression "
        f"rows have extreme change."
    )

# Drift
if len(drift_df) > 0:

    significant_drift = (
        drift_df["p_value"] < 0.05
    ).sum()

    print(
        f"⚠ FEATURE DRIFT: "
        f"{significant_drift} numeric features "
        f"show statistically significant KS drift."
    )

# Temporal drift
if len(train_df) > 0 and len(test_df) > 0:

    train_down = (
        (train_df["target"] == 0).mean()
        * 100
    )

    test_down = (
        (test_df["target"] == 0).mean()
        * 100
    )

    print(
        f"⚠ TEMPORAL DRIFT: "
        f"DOWN changed from "
        f"{train_down:.2f}% → {test_down:.2f}%."
    )

# Missing
if missing_train == 0 and missing_test == 0:
    print("✓ MISSING VALUES: No numeric missing values.")
else:
    print("🔴 MISSING VALUES detected.")

# Infinite
if inf_train == 0 and inf_test == 0:
    print("✓ INFINITE VALUES: None detected.")
else:
    print("🔴 INFINITE VALUES detected.")

# Leakage
if suspicious:
    print(
        "🔴 LEAKAGE AUDIT: Suspicious candidate "
        "feature names detected."
    )
else:
    print(
        "✓ LEAKAGE AUDIT: No suspicious candidate "
        "feature names detected."
    )

print("\n" + "=" * 90)
print("AUDIT COMPLETE")
print("=" * 90)

print(
    "\nIMPORTANT:"
    "\nThis audit did NOT modify the dataset."
    "\nThe frozen target remains unchanged."
    "\nNo rows were removed."
    "\nNo balancing was applied."
    "\nNo train/test split was modified."
)

# =============================================================================
# CLEAN AUDIT-ONLY TEMPORARY COLUMNS
# =============================================================================

df.drop(
    columns=[
        "_volume_bucket",
        "_absolute_imp_change",
        "_year_month"
    ],
    inplace=True,
    errors="ignore"
)

print("\n✓ Audit-only temporary columns removed.")
print("✓ FINAL DATASET REMAINS UNMODIFIED.")

FINAL PRE-MODEL COMPREHENSIVE AUDIT

Rows    : 626,836
Columns : 36

AUDIT 1 — BASIC DATASET STRUCTURE
✓ All core columns present.

AUDIT 2 — DUPLICATE PAGE-WINDOW CHECK
Duplicate page-window rows : 0
✓ No duplicate page-window records.

AUDIT 3 — TARGET CONSISTENCY
Target mismatches : 0
✓ Target exactly matches frozen rule.

AUDIT 4 — TARGET DISTRIBUTION


,target,label,count,percentage
0,0,DOWN,206390,32.925678
1,1,FLAT,156653,24.991066
2,2,UP,263793,42.083256



AUDIT 5 — EXTREME TARGET CHANGE
Change <= -90%        :     62,514 (9.97%)
Change >= +500%       :     56,140 (8.96%)
Change >= +1000%      :     28,828 (4.60%)

Change distribution:
count    6.268360e+05
mean     3.033754e+02
std      4.656967e+03
min     -1.000000e+02
1%      -1.000000e+02
5%      -1.000000e+02
10%     -9.000000e+01
25%     -5.015929e+01
50%      1.956522e+01
75%      1.500000e+02
90%      4.421324e+02
95%      9.090909e+02
99%      4.678114e+03
max      2.306500e+06
Name: future_impression_change_pct, dtype: float64

AUDIT 6 — LOW-VOLUME TARGET INSTABILITY


,rows,DOWN_%,FLAT_%,UP_%
_volume_bucket,,,,
0,0,NaN,NaN,NaN
1-10,154533,45.667916,14.602706,39.729378
11-25,51026,34.449104,16.242700,49.308196
26-50,45917,30.376549,19.145415,50.478036
51-100,52601,27.858786,22.672573,49.468641
101-500,140925,27.554373,26.758205,45.687422
501-1000,57587,27.961172,30.784726,41.254102
1000+,124247,27.932264,39.956699,32.111037



AUDIT 7 — EXTREME CHANGE BY VOLUME


...


AUDIT 17 — FEATURE PREDICTIVE SIGNAL


,feature,spearman,abs_spearman
0,gsc_avg_position_missing_last,-0.1555,0.1555
1,gsc_impressions_last_30d,0.1193,0.1193
2,gsc_impressions_last,0.1193,0.1193
3,ctr_last,0.1132,0.1132
4,gsc_clicks_last,0.0994,0.0994
5,ctr_mean_3m,0.0881,0.0881
6,gsc_impressions_prev_30d,-0.0785,0.0785
7,gsc_avg_position_last,0.0663,0.0663
8,ga4_total_engagement_sec_mean_3m,-0.0647,0.0647
9,gsc_clicks_mean_3m,0.0618,0.0618



AUDIT 18 — SIMPLE TREND BASELINE
Trend baseline accuracy : 48.40%

AUDIT 19 — TARGET VS CURRENT IMPRESSION VOLUME


,rows,current_median,current_mean,future_median,change_median
DOWN,206390,53.000,907.234,11.000,-73.333
FLAT,156653,325.000,1893.508,341.667,3.580
UP,263793,90.667,677.636,314.667,192.455



AUDIT 20 — PAGE / CLIENT TEMPORAL OVERLAP
Train pages : 136,266
Test pages  : 139,942
Page overlap: 125,211

Train clients : 36
Test clients  : 32
Client overlap: 32

FINAL AUDIT SUMMARY
Rows                         : 626,836
Target mismatches            : 0
Duplicate page-window rows   : 0
Train rows                   : 486,894
Test rows                    : 139,942
Candidate numeric features   : 25
Train missing values         : 0
Test missing values          : 0
Train infinite values        : 0
Test infinite values         : 0

------------------------------------------------------------------------------------------
IMPORTANT FINDINGS
------------------------------------------------------------------------------------------
✓ TARGET: Frozen target rule is internally correct.
⚠ LOW VOLUME: 32.33% of <100-impression rows have extreme change.
⚠ FEATURE DRIFT: 25 numeric features show statistically significant KS drift.
⚠ TEMPORAL DRIFT: DOWN changed from 27.14% → 53.06%.
✓ MISSING VA

**FINAL DATASET LOAD + HARD LEAKAGE AUDIT**

In [15]:
# =============================================================================
# STEP 1 — FINAL ROLLING DATASET LOAD + HARD LEAKAGE AUDIT
# =============================================================================

import pandas as pd
import numpy as np

print("=" * 90)
print("STEP 1 — FINAL DATASET LOAD + HARD LEAKAGE AUDIT")
print("=" * 90)

FILE_PATH = "/content/finalrolling90window.parquet"

df = pd.read_parquet(FILE_PATH)

df["window_start"] = pd.to_datetime(
    df["window_start"],
    errors="coerce"
)

print(f"\nRows    : {len(df):,}")
print(f"Columns : {df.shape[1]:,}")

# -----------------------------------------------------------------------------
# Required columns
# -----------------------------------------------------------------------------

required = [
    "content_hash_id",
    "client_hash_id",
    "window_start",
    "target",
    "target_label"
]

missing = [
    c for c in required
    if c not in df.columns
]

if missing:
    raise KeyError(
        "Required columns missing:\n" +
        "\n".join(missing)
    )

# -----------------------------------------------------------------------------
# Target verification
# -----------------------------------------------------------------------------

print("\n" + "=" * 90)
print("TARGET VERIFICATION")
print("=" * 90)

print("Target values:", sorted(df["target"].dropna().unique().tolist()))

assert set(df["target"].dropna().unique()).issubset({0, 1, 2})

print("✓ Target contains only 0, 1, 2.")
print("✓ 0 = DOWN")
print("✓ 1 = FLAT")
print("✓ 2 = UP")

# -----------------------------------------------------------------------------
# Future-column audit
# -----------------------------------------------------------------------------

future_columns = [
    c for c in df.columns
    if c.lower().startswith("future")
]

print("\n" + "=" * 90)
print("FUTURE COLUMN AUDIT")
print("=" * 90)

if future_columns:
    for c in future_columns:
        print("BLOCKED:", c)
else:
    print("✓ No future_* columns found.")

# -----------------------------------------------------------------------------
# Columns that must NEVER enter X
# -----------------------------------------------------------------------------

blocked_columns = set(
    future_columns +
    [
        "target",
        "target_label",
        "content_hash_id",
        "client_hash_id",
        "window_start",
        "window_end",
        "content_age_days"
    ]
)

print("\nBlocked from model input:")
for c in sorted(blocked_columns):
    print(" -", c)

print("\n✓ Leakage audit rules established.")

STEP 1 — FINAL DATASET LOAD + HARD LEAKAGE AUDIT

Rows    : 626,836
Columns : 36

TARGET VERIFICATION
Target values: [0, 1, 2]
✓ Target contains only 0, 1, 2.
✓ 0 = DOWN
✓ 1 = FLAT
✓ 2 = UP

FUTURE COLUMN AUDIT
BLOCKED: future_start
BLOCKED: future_end
BLOCKED: future_imp_3m
BLOCKED: future_impression_change_pct

Blocked from model input:
 - client_hash_id
 - content_age_days
 - content_hash_id
 - future_end
 - future_imp_3m
 - future_impression_change_pct
 - future_start
 - target
 - target_label
 - window_end
 - window_start

✓ Leakage audit rules established.


**CAUSAL MOMENTUM / VELOCITY FEATURES**

Features such as:

last vs previous 30d

last vs 3M baseline

recent momentum

CTR momentum

position movement

clicks momentum

engagement momentum

In [16]:
# =============================================================================
# STEP 2 — CAUSAL MOMENTUM + VELOCITY FEATURE ENGINEERING
# =============================================================================

print("=" * 90)
print("STEP 2 — CAUSAL MOMENTUM + VELOCITY FEATURES")
print("=" * 90)

df_model = df.copy()

EPS = 1e-6

# -----------------------------------------------------------------------------
# Safe ratio helper
# -----------------------------------------------------------------------------

def safe_ratio(a, b):
    return (
        a.astype("float64") /
        b.astype("float64").abs().clip(lower=EPS)
    )

# -----------------------------------------------------------------------------
# 1. IMPRESSION MOMENTUM
# -----------------------------------------------------------------------------

if {
    "gsc_impressions_last",
    "gsc_impressions_prev_30d"
}.issubset(df_model.columns):

    df_model["impressions_momentum_30d"] = (
        safe_ratio(
            df_model["gsc_impressions_last"],
            df_model["gsc_impressions_prev_30d"]
        ) - 1.0
    )

if {
    "gsc_impressions_last",
    "gsc_impressions_mean_3m"
}.issubset(df_model.columns):

    df_model["impressions_vs_3m_baseline"] = (
        safe_ratio(
            df_model["gsc_impressions_last"],
            df_model["gsc_impressions_mean_3m"]
        ) - 1.0
    )

# -----------------------------------------------------------------------------
# 2. 30-DAY RECENT CHANGE
# -----------------------------------------------------------------------------

if {
    "gsc_impressions_last_30d",
    "gsc_impressions_prev_30d"
}.issubset(df_model.columns):

    df_model["impressions_30d_velocity"] = (
        safe_ratio(
            df_model["gsc_impressions_last_30d"],
            df_model["gsc_impressions_prev_30d"]
        ) - 1.0
    )

# -----------------------------------------------------------------------------
# 3. CLICKS MOMENTUM
# -----------------------------------------------------------------------------

if {
    "gsc_clicks_last",
    "gsc_clicks_mean_3m"
}.issubset(df_model.columns):

    df_model["clicks_vs_3m_baseline"] = (
        safe_ratio(
            df_model["gsc_clicks_last"],
            df_model["gsc_clicks_mean_3m"]
        ) - 1.0
    )

# -----------------------------------------------------------------------------
# 4. CTR MOMENTUM
# -----------------------------------------------------------------------------

if {
    "ctr_last",
    "ctr_mean_3m"
}.issubset(df_model.columns):

    df_model["ctr_momentum"] = (
        safe_ratio(
            df_model["ctr_last"],
            df_model["ctr_mean_3m"]
        ) - 1.0
    )

# -----------------------------------------------------------------------------
# 5. POSITION MOMENTUM
#
# Lower Google position number = better ranking.
# -----------------------------------------------------------------------------

if {
    "gsc_avg_position_last",
    "gsc_avg_position_mean_3m"
}.issubset(df_model.columns):

    df_model["position_momentum"] = (
        df_model["gsc_avg_position_last"]
        - df_model["gsc_avg_position_mean_3m"]
    )

# -----------------------------------------------------------------------------
# 6. ENGAGEMENT MOMENTUM
# -----------------------------------------------------------------------------

if {
    "ga4_total_engagement_sec_last",
    "ga4_total_engagement_sec_mean_3m"
}.issubset(df_model.columns):

    df_model["engagement_momentum"] = (
        safe_ratio(
            df_model["ga4_total_engagement_sec_last"],
            df_model["ga4_total_engagement_sec_mean_3m"]
        ) - 1.0
    )

# -----------------------------------------------------------------------------
# 7. ORGANIC SESSION MOMENTUM
# -----------------------------------------------------------------------------

if {
    "sessions_organic_last",
    "sessions_organic_mean_3m"
}.issubset(df_model.columns):

    df_model["organic_sessions_momentum"] = (
        safe_ratio(
            df_model["sessions_organic_last"],
            df_model["sessions_organic_mean_3m"]
        ) - 1.0
    )

# -----------------------------------------------------------------------------
# 8. AI SESSION MOMENTUM
# -----------------------------------------------------------------------------

if {
    "sessions_ai_last",
    "sessions_ai_mean_3m"
}.issubset(df_model.columns):

    df_model["ai_sessions_momentum"] = (
        safe_ratio(
            df_model["sessions_ai_last"],
            df_model["sessions_ai_mean_3m"]
        ) - 1.0
    )

# -----------------------------------------------------------------------------
# 9. Replace numerical infinities
# -----------------------------------------------------------------------------

new_feature_columns = [
    c for c in df_model.columns
    if c not in df.columns
]

df_model[new_feature_columns] = (
    df_model[new_feature_columns]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print(f"\nNew causal features created: {len(new_feature_columns)}")

for c in new_feature_columns:
    print(" +", c)

print("\n✓ No future columns used.")
print("✓ No target used.")
print("✓ No target_label used.")
print("✓ Momentum features created only from historical/current data.")

STEP 2 — CAUSAL MOMENTUM + VELOCITY FEATURES

New causal features created: 9
 + impressions_momentum_30d
 + impressions_vs_3m_baseline
 + impressions_30d_velocity
 + clicks_vs_3m_baseline
 + ctr_momentum
 + position_momentum
 + engagement_momentum
 + organic_sessions_momentum
 + ai_sessions_momentum

✓ No future columns used.
✓ No target used.
✓ No target_label used.
✓ Momentum features created only from historical/current data.


**FINAL MODEL FEATURE SELECTION + LEAKAGE CHECK**

In [17]:
# =============================================================================
# STEP 3 — FINAL MODEL FEATURES + SECOND LEAKAGE CHECK
# =============================================================================

print("=" * 90)
print("STEP 3 — FINAL MODEL FEATURE SELECTION")
print("=" * 90)

# -----------------------------------------------------------------------------
# Explicitly blocked columns
# -----------------------------------------------------------------------------

blocked = set(
    future_columns +
    [
        "target",
        "target_label",
        "content_hash_id",
        "client_hash_id",
        "window_start",
        "window_end",
        "content_age_days"
    ]
)

# -----------------------------------------------------------------------------
# Candidate numeric features
# -----------------------------------------------------------------------------

candidate_features = [
    c for c in df_model.columns
    if c not in blocked
    and pd.api.types.is_numeric_dtype(df_model[c])
]

# -----------------------------------------------------------------------------
# Suspicious-name audit
# -----------------------------------------------------------------------------

suspicious_keywords = [
    "future",
    "target",
    "label",
    "outcome",
    "next_90",
    "next90",
    "forward"
]

suspicious_features = [
    c for c in candidate_features
    if any(k in c.lower() for k in suspicious_keywords)
]

print(f"\nCandidate numeric features : {len(candidate_features)}")

if suspicious_features:
    print("\n🚨 SUSPICIOUS FEATURES FOUND:")
    for c in suspicious_features:
        print(" -", c)

    raise ValueError(
        "Potential leakage detected in model features."
    )

print("✓ No suspicious feature names detected.")

# -----------------------------------------------------------------------------
# Create X / y
# -----------------------------------------------------------------------------

X = df_model[candidate_features].copy()
y = df_model["target"].astype("int8").copy()

# -----------------------------------------------------------------------------
# Missing-value check
# -----------------------------------------------------------------------------

missing_trainable = X.isna().sum()

if missing_trainable.sum() > 0:
    print("\nMissing values found. Filling with 0.")
    X = X.fillna(0)

X = X.replace([np.inf, -np.inf], 0)

print("\n" + "=" * 90)
print("FINAL MODEL INPUT")
print("=" * 90)

print("Features :", X.shape[1])
print("Rows     :", X.shape[0])

print("\nFeature list:")
for i, c in enumerate(candidate_features, 1):
    print(f"{i:2}. {c}")

print("\n✓ X contains no target.")
print("✓ X contains no future feature.")
print("✓ X contains no page/client identifier.")
print("✓ X contains no non-causal content_age_days.")

assert "target" not in X.columns
assert "target_label" not in X.columns

assert not any(
    c.lower().startswith("future")
    for c in X.columns
)

print("\n✓ HARD LEAKAGE CHECK PASSED.")

STEP 3 — FINAL MODEL FEATURE SELECTION

Candidate numeric features : 35
✓ No suspicious feature names detected.

FINAL MODEL INPUT
Features : 35
Rows     : 626836

Feature list:
 1. gsc_clicks_mean_3m
 2. gsc_clicks_last
 3. gsc_impressions_mean_3m
 4. gsc_impressions_last
 5. gsc_avg_position_mean_3m
 6. gsc_avg_position_last
 7. ga4_total_engagement_sec_mean_3m
 8. ga4_total_engagement_sec_last
 9. sessions_organic_mean_3m
10. sessions_organic_last
11. sessions_ai_mean_3m
12. sessions_ai_last
13. gsc_avg_position_missing_mean_3m
14. gsc_avg_position_missing_last
15. ctr_mean_3m
16. ctr_last
17. sec_per_click_mean_3m
18. sec_per_click_last
19. ai_share_mean_3m
20. ai_share_last
21. engagement_per_organic_session_mean_3m
22. engagement_per_organic_session_last
23. current_imp_3m
24. gsc_impressions_prev_30d
25. gsc_impressions_last_30d
26. early_drop_signal
27. impressions_momentum_30d
28. impressions_vs_3m_baseline
29. impressions_30d_velocity
30. clicks_vs_3m_baseline
31. ctr_momentu

**LOCKED TEMPORAL TRAIN / TEST SPLIT**

In [18]:
# =============================================================================
# STEP 4 — LOCKED TEMPORAL TRAIN / TEST SPLIT
# =============================================================================

print("=" * 90)
print("STEP 4 — LOCKED TEMPORAL TRAIN / TEST SPLIT")
print("=" * 90)

CUTOFF = pd.Timestamp("2026-01-01")

train_mask = df_model["window_start"] < CUTOFF
test_mask  = df_model["window_start"] >= CUTOFF

X_train = X.loc[train_mask].copy()
X_test  = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test  = y.loc[test_mask].copy()

print(f"\nTRAIN rows : {len(X_train):,}")
print(f"TEST rows  : {len(X_test):,}")

print(
    f"\nTRAIN % : "
    f"{len(X_train) / len(X) * 100:.2f}%"
)

print(
    f"TEST %  : "
    f"{len(X_test) / len(X) * 100:.2f}%"
)

print("\nTrain latest window :",
      df_model.loc[train_mask, "window_start"].max())

print("Test earliest window:",
      df_model.loc[test_mask, "window_start"].min())

assert (
    df_model.loc[train_mask, "window_start"].max()
    <
    df_model.loc[test_mask, "window_start"].min()
)

print("\n✓ Temporal ordering correct.")

# -----------------------------------------------------------------------------
# Page overlap is intentionally allowed
# -----------------------------------------------------------------------------

train_pages = set(
    df_model.loc[train_mask, "content_hash_id"]
)

test_pages = set(
    df_model.loc[test_mask, "content_hash_id"]
)

overlap = len(train_pages & test_pages)

print("\nPage overlap :", f"{overlap:,}")
print("✓ Existing-page overlap is allowed for future forecasting.")

# -----------------------------------------------------------------------------
# Client overlap
# -----------------------------------------------------------------------------

train_clients = set(
    df_model.loc[train_mask, "client_hash_id"]
)

test_clients = set(
    df_model.loc[test_mask, "client_hash_id"]
)

print("Client overlap:",
      f"{len(train_clients & test_clients):,}")

print("\n✓ BLOCK 4 COMPLETE.")

STEP 4 — LOCKED TEMPORAL TRAIN / TEST SPLIT

TRAIN rows : 486,894
TEST rows  : 139,942

TRAIN % : 77.67%
TEST %  : 22.33%

Train latest window : 2025-12-01 00:00:00
Test earliest window: 2026-01-01 00:00:00

✓ Temporal ordering correct.

Page overlap : 125,211
✓ Existing-page overlap is allowed for future forecasting.
Client overlap: 32

✓ BLOCK 4 COMPLETE.


**SMOOTH RECENCY WEIGHTS + CLASS WEIGHTS**

In [19]:
# =============================================================================
# STEP 5 — RECENCY WEIGHTING + CLASS WEIGHTING
# =============================================================================

print("=" * 90)
print("STEP 5 — RECENCY WEIGHTING + CLASS WEIGHTING")
print("=" * 90)

# -----------------------------------------------------------------------------
# Recency weighting
#
# Half-life = 90 days
# 90 days older -> approximately half the weight
# -----------------------------------------------------------------------------

HALF_LIFE_DAYS = 90.0

train_dates = df_model.loc[
    train_mask,
    "window_start"
]

latest_train_date = train_dates.max()

age_days = (
    latest_train_date - train_dates
).dt.days.clip(lower=0)

recency_weight = np.exp(
    -np.log(2) * age_days / HALF_LIFE_DAYS
)

recency_weight = pd.Series(
    recency_weight,
    index=X_train.index
)

# -----------------------------------------------------------------------------
# Class weights
# -----------------------------------------------------------------------------

class_counts = y_train.value_counts().sort_index()

n_train = len(y_train)
n_classes = len(class_counts)

class_weight_map = {
    cls: n_train / (n_classes * count)
    for cls, count in class_counts.items()
}

class_weight = y_train.map(
    class_weight_map
).astype("float64")

# -----------------------------------------------------------------------------
# Combined sample weight
# -----------------------------------------------------------------------------

sample_weight = (
    recency_weight *
    class_weight
)

# Normalize around 1
sample_weight = (
    sample_weight /
    sample_weight.mean()
)

print("\nCLASS COUNTS:")
print(class_counts)

print("\nCLASS WEIGHTS:")
for cls, weight in class_weight_map.items():
    label = {
        0: "DOWN",
        1: "FLAT",
        2: "UP"
    }[cls]

    print(
        f"{cls} ({label}) : {weight:.4f}"
    )

print("\nRECENCY:")
print("Half-life:", HALF_LIFE_DAYS, "days")
print(
    "Oldest training weight:",
    f"{recency_weight.min():.4f}"
)
print(
    "Newest training weight:",
    f"{recency_weight.max():.4f}"
)

print("\nCombined sample weight:")
print("Min :", f"{sample_weight.min():.4f}")
print("Max :", f"{sample_weight.max():.4f}")
print("Mean:", f"{sample_weight.mean():.4f}")

print("\n✓ Full 2025 retained.")
print("✓ No arbitrary December hard-coding.")
print("✓ Smooth temporal weighting applied.")
print("✓ Class weighting applied.")

STEP 5 — RECENCY WEIGHTING + CLASS WEIGHTING

CLASS COUNTS:
target
0    132136
1    119430
2    235328
Name: count, dtype: int64

CLASS WEIGHTS:
0 (DOWN) : 1.2283
1 (FLAT) : 1.3589
2 (UP) : 0.6897

RECENCY:
Half-life: 90.0 days
Oldest training weight: 0.0764
Newest training weight: 1.0000

Combined sample weight:
Min : 0.0777
Max : 2.0046
Mean: 1.0000

✓ Full 2025 retained.
✓ No arbitrary December hard-coding.
✓ Smooth temporal weighting applied.
✓ Class weighting applied.


**RANDOM FOREST TRAINING**

In [20]:
# =============================================================================
# STEP 6 — RANDOM FOREST TRAINING
# =============================================================================

from sklearn.ensemble import RandomForestClassifier

print("=" * 90)
print("STEP 6 — RANDOM FOREST TRAINING")
print("=" * 90)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight=None,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

print("\nRandom Forest configuration:")
print("Trees             :", rf_model.n_estimators)
print("Max depth         :", rf_model.max_depth)
print("Min samples split :", rf_model.min_samples_split)
print("Min samples leaf  :", rf_model.min_samples_leaf)
print("Max features      :", rf_model.max_features)
print("Class weighting   : External sample weights")
print("Recency weighting : External sample weights")
print("CPU               : All available")

print("\nTraining Random Forest...")

rf_model.fit(
    X_train,
    y_train,
    sample_weight=sample_weight
)

print("\n✓ RANDOM FOREST TRAINING COMPLETE.")

STEP 6 — RANDOM FOREST TRAINING

Random Forest configuration:
Trees             : 300
Max depth         : 20
Min samples split : 5
Min samples leaf  : 2
Max features      : sqrt
Class weighting   : External sample weights
Recency weighting : External sample weights
CPU               : All available

Training Random Forest...

✓ RANDOM FOREST TRAINING COMPLETE.


**RANDOM FOREST TESTING + CLEAR CLASS-WISE RESULTS**

In [21]:
# =============================================================================
# STEP 7 — RANDOM FOREST TESTING + DETAILED EVALUATION
# =============================================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("=" * 90)
print("STEP 7 — RANDOM FOREST TESTING + EVALUATION")
print("=" * 90)

print("\nGenerating predictions...")

y_pred_rf = rf_model.predict(X_test)

print("✓ Predictions complete.")

# -----------------------------------------------------------------------------
# Overall metrics
# -----------------------------------------------------------------------------

accuracy_rf = accuracy_score(
    y_test,
    y_pred_rf
)

balanced_accuracy_rf = balanced_accuracy_score(
    y_test,
    y_pred_rf
)

precision_rf, recall_rf, f1_rf, support_rf = (
    precision_recall_fscore_support(
        y_test,
        y_pred_rf,
        labels=[0, 1, 2],
        zero_division=0
    )
)

print("\n" + "=" * 90)
print("RANDOM FOREST OVERALL RESULTS")
print("=" * 90)

print(f"Accuracy          : {accuracy_rf*100:.2f}%")
print(f"Balanced Accuracy : {balanced_accuracy_rf*100:.2f}%")
print(f"Macro Precision   : {precision_rf.mean()*100:.2f}%")
print(f"Macro Recall      : {recall_rf.mean()*100:.2f}%")
print(f"Macro F1          : {f1_rf.mean()*100:.2f}%")

# -----------------------------------------------------------------------------
# Class-wise understandable explanation
# -----------------------------------------------------------------------------

labels = {
    0: "DOWN",
    1: "FLAT",
    2: "UP"
}

cm = confusion_matrix(
    y_test,
    y_pred_rf,
    labels=[0, 1, 2]
)

print("\n" + "=" * 90)
print("CLASS-WISE PREDICTION BREAKDOWN")
print("=" * 90)

for i, cls in enumerate([0, 1, 2]):

    actual = cm[i].sum()
    predicted = cm[:, i].sum()
    correct = cm[i, i]
    missed = actual - correct

    print(f"\n{labels[cls]}")
    print("-" * 60)

    print(
        f"Actual pages       : {actual:,}"
    )

    print(
        f"Model predicted     : {predicted:,}"
    )

    print(
        f"Correct prediction  : {correct:,} "
        f"({correct/actual*100:.2f}% of actual)"
    )

    print(
        f"Wrong / missed      : {missed:,}"
    )

    print(
        f"Precision           : "
        f"{precision_rf[i]*100:.2f}%"
    )

    print(
        f"Recall              : "
        f"{recall_rf[i]*100:.2f}%"
    )

    print(
        f"F1-score            : "
        f"{f1_rf[i]*100:.2f}%"
    )

# -----------------------------------------------------------------------------
# Confusion matrix
# -----------------------------------------------------------------------------

print("\n" + "=" * 90)
print("CONFUSION MATRIX")
print("=" * 90)

cm_df = pd.DataFrame(
    cm,
    index=[
        "Actual DOWN",
        "Actual FLAT",
        "Actual UP"
    ],
    columns=[
        "Pred DOWN",
        "Pred FLAT",
        "Pred UP"
    ]
)

display(cm_df)

print("\nInterpretation:")
print("Rows    = what actually happened")
print("Columns = what model predicted")
print("Diagonal = correct predictions")

# -----------------------------------------------------------------------------
# Full classification report
# -----------------------------------------------------------------------------

print("\n" + "=" * 90)
print("FULL CLASSIFICATION REPORT")
print("=" * 90)

print(
    classification_report(
        y_test,
        y_pred_rf,
        labels=[0, 1, 2],
        target_names=[
            "DOWN",
            "FLAT",
            "UP"
        ],
        digits=4,
        zero_division=0
    )
)

STEP 7 — RANDOM FOREST TESTING + EVALUATION

Generating predictions...
✓ Predictions complete.

RANDOM FOREST OVERALL RESULTS
Accuracy          : 44.32%
Balanced Accuracy : 45.91%
Macro Precision   : 44.08%
Macro Recall      : 45.91%
Macro F1          : 42.37%

CLASS-WISE PREDICTION BREAKDOWN

DOWN
------------------------------------------------------------
Actual pages       : 74,254
Model predicted     : 48,649
Correct prediction  : 32,617 (43.93% of actual)
Wrong / missed      : 41,637
Precision           : 67.05%
Recall              : 43.93%
F1-score            : 53.08%

FLAT
------------------------------------------------------------
Actual pages       : 37,223
Model predicted     : 33,717
Correct prediction  : 11,487 (30.86% of actual)
Wrong / missed      : 25,736
Precision           : 34.07%
Recall              : 30.86%
F1-score            : 32.39%

UP
------------------------------------------------------------
Actual pages       : 28,465
Model predicted     : 57,576
Correct 

,Pred DOWN,Pred FLAT,Pred UP
Actual DOWN,32617,18948,22689
Actual FLAT,8767,11487,16969
Actual UP,7265,3282,17918



Interpretation:
Rows    = what actually happened
Columns = what model predicted
Diagonal = correct predictions

FULL CLASSIFICATION REPORT
              precision    recall  f1-score   support

        DOWN     0.6705    0.4393    0.5308     74254
        FLAT     0.3407    0.3086    0.3239     37223
          UP     0.3112    0.6295    0.4165     28465

    accuracy                         0.4432    139942
   macro avg     0.4408    0.4591    0.4237    139942
weighted avg     0.5097    0.4432    0.4525    139942



**LOGISTIC REGRESSION BASELINE**

In [22]:
# =============================================================================
# STEP 8 — LOGISTIC REGRESSION BASELINE
# =============================================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

print("=" * 90)
print("STEP 8 — LOGISTIC REGRESSION BASELINE")
print("=" * 90)

logistic_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "logistic",
        LogisticRegression(
            max_iter=1000,
            class_weight=None,
            C=1.0,
            solver="lbfgs",
            multi_class="auto",
            random_state=42
        )
    )
])

print("\nTraining Logistic Regression baseline...")

logistic_model.fit(
    X_train,
    y_train,
    logistic__sample_weight=sample_weight
)

print("✓ Logistic Regression training complete.")

print("\nGenerating predictions...")

y_pred_lr = logistic_model.predict(X_test)

print("✓ Predictions complete.")

STEP 8 — LOGISTIC REGRESSION BASELINE

Training Logistic Regression baseline...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


✓ Logistic Regression training complete.

Generating predictions...
✓ Predictions complete.


**LOGISTIC EVALUATION**

In [23]:
# =============================================================================
# STEP 9 — LOGISTIC REGRESSION EVALUATION
# =============================================================================

print("=" * 90)
print("STEP 9 — LOGISTIC REGRESSION EVALUATION")
print("=" * 90)

accuracy_lr = accuracy_score(
    y_test,
    y_pred_lr
)

balanced_accuracy_lr = balanced_accuracy_score(
    y_test,
    y_pred_lr
)

precision_lr, recall_lr, f1_lr, support_lr = (
    precision_recall_fscore_support(
        y_test,
        y_pred_lr,
        labels=[0, 1, 2],
        zero_division=0
    )
)

print("\n" + "=" * 90)
print("LOGISTIC REGRESSION RESULTS")
print("=" * 90)

print(f"Accuracy          : {accuracy_lr*100:.2f}%")
print(f"Balanced Accuracy : {balanced_accuracy_lr*100:.2f}%")
print(f"Macro Precision   : {precision_lr.mean()*100:.2f}%")
print(f"Macro Recall      : {recall_lr.mean()*100:.2f}%")
print(f"Macro F1          : {f1_lr.mean()*100:.2f}%")

print("\n" + "=" * 90)
print("LOGISTIC CLASSIFICATION REPORT")
print("=" * 90)

print(
    classification_report(
        y_test,
        y_pred_lr,
        labels=[0, 1, 2],
        target_names=[
            "DOWN",
            "FLAT",
            "UP"
        ],
        digits=4,
        zero_division=0
    )
)

cm_lr = confusion_matrix(
    y_test,
    y_pred_lr,
    labels=[0, 1, 2]
)

print("\nCONFUSION MATRIX")

display(
    pd.DataFrame(
        cm_lr,
        index=[
            "Actual DOWN",
            "Actual FLAT",
            "Actual UP"
        ],
        columns=[
            "Pred DOWN",
            "Pred FLAT",
            "Pred UP"
        ]
    )
)

STEP 9 — LOGISTIC REGRESSION EVALUATION

LOGISTIC REGRESSION RESULTS
Accuracy          : 46.12%
Balanced Accuracy : 47.91%
Macro Precision   : 45.90%
Macro Recall      : 47.91%
Macro F1          : 44.52%

LOGISTIC CLASSIFICATION REPORT
              precision    recall  f1-score   support

        DOWN     0.6912    0.4464    0.5425     74254
        FLAT     0.3524    0.3631    0.3577     37223
          UP     0.3333    0.6279    0.4354     28465

    accuracy                         0.4612    139942
   macro avg     0.4590    0.4791    0.4452    139942
weighted avg     0.5283    0.4612    0.4715    139942


CONFUSION MATRIX


,Pred DOWN,Pred FLAT,Pred UP
Actual DOWN,33147,20568,20539
Actual FLAT,8490,13517,15216
Actual UP,6317,4276,17872


**OLD BASELINE vs LOGISTIC vs IMPROVED RANDOM FOREST**

In [24]:
# =============================================================================
# STEP 10 — FINAL MODEL COMPARISON
# =============================================================================

print("=" * 90)
print("STEP 10 — FINAL MODEL COMPARISON")
print("=" * 90)

comparison = pd.DataFrame({
    "Model": [
        "Simple Trend Baseline",
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy_%": [
        51.94,
        accuracy_lr * 100,
        accuracy_rf * 100
    ],
    "Balanced_Accuracy_%": [
        np.nan,
        balanced_accuracy_lr * 100,
        balanced_accuracy_rf * 100
    ],
    "Macro_Precision_%": [
        np.nan,
        precision_lr.mean() * 100,
        precision_rf.mean() * 100
    ],
    "Macro_Recall_%": [
        np.nan,
        recall_lr.mean() * 100,
        recall_rf.mean() * 100
    ],
    "Macro_F1_%": [
        np.nan,
        f1_lr.mean() * 100,
        f1_rf.mean() * 100
    ]
})

display(
    comparison.round(2)
)

print("\n" + "=" * 90)
print("DECISION GUIDANCE")
print("=" * 90)

print("""
1. Accuracy alone will NOT decide the winner.
2. Balanced Accuracy is important because DOWN/FLAT/UP distributions differ.
3. Macro F1 is especially important because every class matters.
4. DOWN recall is critical for the FlyRank crash-warning use case.
5. FLAT F1 must also be checked because FLAT is difficult to separate.
6. Test set remains completely untouched throughout training.
7. No tuning decision should be made using test results repeatedly.
""")

STEP 10 — FINAL MODEL COMPARISON


,Model,Accuracy_%,Balanced_Accuracy_%,Macro_Precision_%,Macro_Recall_%,Macro_F1_%
0,Simple Trend Baseline,51.94,NaN,NaN,NaN,NaN
1,Logistic Regression,46.12,47.91,45.90,47.91,44.52
2,Random Forest,44.32,45.91,44.08,45.91,42.37



DECISION GUIDANCE

1. Accuracy alone will NOT decide the winner.
2. Balanced Accuracy is important because DOWN/FLAT/UP distributions differ.
3. Macro F1 is especially important because every class matters.
4. DOWN recall is critical for the FlyRank crash-warning use case.
5. FLAT F1 must also be checked because FLAT is difficult to separate.
6. Test set remains completely untouched throughout training.
7. No tuning decision should be made using test results repeatedly.



**FINAL MODEL COMPARISON + DEPLOYMENT EVALUATION**

In [27]:
# =============================================================================
# STEP 10 — FINAL MODEL COMPARISON + DEPLOYMENT EVALUATION
# =============================================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("=" * 100)
print("STEP 10 — FINAL MODEL COMPARISON + DEPLOYMENT EVALUATION")
print("=" * 100)


# =============================================================================
# 1. REQUIRED DATA CHECK
# =============================================================================

required = [
    "y_test",
    "y_pred_rf"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise NameError(
        "Required prediction variables missing:\n"
        + "\n".join(missing)
    )

print("✓ Test labels found.")
print("✓ Random Forest predictions found.")


# =============================================================================
# 2. RANDOM FOREST MODEL OBJECT — AUTO DETECTION
# =============================================================================

rf_candidates = [
    "rf_model",
    "rf",
    "random_forest_model",
    "rf_classifier"
]

rf_model_found = None
rf_model_name = None

for name in rf_candidates:

    if name in globals():

        obj = globals()[name]

        if hasattr(obj, "predict_proba"):

            rf_model_found = obj
            rf_model_name = name
            break


# =============================================================================
# 3. METRIC FUNCTION
# =============================================================================

def get_metrics(y_true, y_pred):

    return {
        "Accuracy_%":
            accuracy_score(
                y_true,
                y_pred
            ) * 100,

        "Balanced_Accuracy_%":
            balanced_accuracy_score(
                y_true,
                y_pred
            ) * 100,

        "Macro_Precision_%":
            precision_score(
                y_true,
                y_pred,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0
            ) * 100,

        "Macro_Recall_%":
            recall_score(
                y_true,
                y_pred,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0
            ) * 100,

        "Macro_F1_%":
            f1_score(
                y_true,
                y_pred,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0
            ) * 100,

        "Weighted_F1_%":
            f1_score(
                y_true,
                y_pred,
                labels=[0, 1, 2],
                average="weighted",
                zero_division=0
            ) * 100
    }


# =============================================================================
# 4. RANDOM FOREST METRICS
# =============================================================================

rf_metrics = get_metrics(
    y_test,
    y_pred_rf
)


# =============================================================================
# 5. LOGISTIC REGRESSION — OPTIONAL
# =============================================================================
#
# Agar y_pred_lr already available hai to comparison hoga.
# Agar nahi hai to code error nahi dega.
# =============================================================================

lr_available = (
    "y_pred_lr" in globals()
)

if lr_available:

    lr_metrics = get_metrics(
        y_test,
        y_pred_lr
    )

else:

    lr_metrics = None

    print(
        "\nℹ Logistic Regression prediction "
        "variable not found."
    )

    print(
        "  Logistic Regression comparison "
        "will be skipped."
    )


# =============================================================================
# 6. MODEL COMPARISON
# =============================================================================

comparison_rows = []

# Simple trend baseline
comparison_rows.append({
    "Model": "Simple Trend Baseline",
    "Accuracy_%": 51.94,
    "Balanced_Accuracy_%": np.nan,
    "Macro_Precision_%": np.nan,
    "Macro_Recall_%": np.nan,
    "Macro_F1_%": np.nan,
    "Weighted_F1_%": np.nan
})


if lr_available:

    comparison_rows.append({
        "Model": "Logistic Regression",
        **lr_metrics
    })


comparison_rows.append({
    "Model": "Random Forest",
    **rf_metrics
})


comparison = pd.DataFrame(
    comparison_rows
)


print("\n" + "=" * 100)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 100)

display(
    comparison.round(2)
)


# =============================================================================
# 7. RANDOM FOREST CLASS-WISE EVALUATION
# =============================================================================

print("\n" + "=" * 100)
print("RANDOM FOREST — CLASS-WISE PERFORMANCE")
print("=" * 100)

precision = precision_score(
    y_test,
    y_pred_rf,
    labels=[0, 1, 2],
    average=None,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred_rf,
    labels=[0, 1, 2],
    average=None,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred_rf,
    labels=[0, 1, 2],
    average=None,
    zero_division=0
)

class_table = pd.DataFrame({

    "Class": [
        "DOWN",
        "FLAT",
        "UP"
    ],

    "Precision_%":
        precision * 100,

    "Recall_%":
        recall * 100,

    "F1_%":
        f1 * 100
})

display(
    class_table.round(2)
)


# =============================================================================
# 8. CONFUSION MATRIX
# =============================================================================

print("\n" + "=" * 100)
print("RANDOM FOREST — CONFUSION MATRIX")
print("=" * 100)

cm = confusion_matrix(
    y_test,
    y_pred_rf,
    labels=[0, 1, 2]
)

cm_table = pd.DataFrame(
    cm,
    index=[
        "Actual DOWN",
        "Actual FLAT",
        "Actual UP"
    ],
    columns=[
        "Pred DOWN",
        "Pred FLAT",
        "Pred UP"
    ]
)

display(cm_table)


# =============================================================================
# 9. HIGH-CONFIDENCE RELIABILITY
# =============================================================================

print("\n" + "=" * 100)
print("HIGH-CONFIDENCE PREDICTION RELIABILITY")
print("=" * 100)

if rf_model_found is not None:

    print(
        f"✓ Random Forest model detected: "
        f"{rf_model_name}"
    )

    # X_test must exist for probability calculation
    if "X_test" not in globals():

        print(
            "⚠ X_test not found."
        )

        confidence_table = pd.DataFrame()

    else:

        rf_prob = rf_model_found.predict_proba(
            X_test
        )

        confidence = rf_prob.max(
            axis=1
        )

        confidence_prediction = rf_prob.argmax(
            axis=1
        )

        confidence_rows = []

        for threshold in [0.80, 0.90]:

            mask = (
                confidence >= threshold
            )

            count = int(
                mask.sum()
            )

            if count > 0:

                accuracy = (
                    accuracy_score(
                        y_test[mask],
                        confidence_prediction[mask]
                    ) * 100
                )

                coverage = (
                    count /
                    len(y_test) *
                    100
                )

            else:

                accuracy = np.nan
                coverage = 0

            confidence_rows.append({

                "Confidence_Threshold_%":
                    threshold * 100,

                "Predictions":
                    count,

                "Coverage_%":
                    coverage,

                "Accuracy_%":
                    accuracy
            })

        confidence_table = pd.DataFrame(
            confidence_rows
        )

        display(
            confidence_table.round(2)
        )

else:

    print(
        "⚠ RF model object not found."
    )

    print(
        "Confidence analysis skipped."
    )

    confidence_table = pd.DataFrame()


# =============================================================================
# 10. DIRECTIONAL SEVERE-ERROR ANALYSIS
# =============================================================================

print("\n" + "=" * 100)
print("DIRECTIONAL / SEVERE ERROR ANALYSIS")
print("=" * 100)

# 0 = DOWN
# 1 = FLAT
# 2 = UP

# Cost:
#
# Correct       = 0
# One-step      = 1
# DOWN <-> UP   = 3

cost_matrix = np.array([
    [0, 1, 3],
    [1, 0, 1],
    [3, 1, 0]
])

directional_cost = (
    cm * cost_matrix
).sum()

average_cost = (
    directional_cost /
    len(y_test)
)

severe_down_up = cm[0, 2]
severe_up_down = cm[2, 0]

print(
    f"DOWN → UP severe errors : "
    f"{severe_down_up:,}"
)

print(
    f"UP → DOWN severe errors : "
    f"{severe_up_down:,}"
)

print(
    f"Total directional cost  : "
    f"{directional_cost:,}"
)

print(
    f"Average cost / prediction: "
    f"{average_cost:.4f}"
)


# =============================================================================
# 11. DOWN CRASH-WARNING ANALYSIS
# =============================================================================

print("\n" + "=" * 100)
print("FLYRANK — DOWN CRASH WARNING ANALYSIS")
print("=" * 100)

actual_down = (
    y_test == 0
)

predicted_down = (
    y_pred_rf == 0
)

correct_down = (
    actual_down &
    predicted_down
).sum()

missed_down = (
    actual_down &
    ~predicted_down
).sum()

false_down = (
    ~actual_down &
    predicted_down
).sum()

actual_down_count = (
    actual_down.sum()
)

predicted_down_count = (
    predicted_down.sum()
)

down_recall = (
    correct_down /
    actual_down_count *
    100
    if actual_down_count > 0
    else 0
)

down_precision = (
    correct_down /
    predicted_down_count *
    100
    if predicted_down_count > 0
    else 0
)

print(
    f"Actual DOWN pages      : "
    f"{actual_down_count:,}"
)

print(
    f"Predicted DOWN pages   : "
    f"{predicted_down_count:,}"
)

print(
    f"Correct DOWN warnings  : "
    f"{correct_down:,}"
)

print(
    f"Missed DOWN pages      : "
    f"{missed_down:,}"
)

print(
    f"False DOWN warnings    : "
    f"{false_down:,}"
)

print(
    f"DOWN Recall            : "
    f"{down_recall:.2f}%"
)

print(
    f"DOWN Precision         : "
    f"{down_precision:.2f}%"
)


# =============================================================================
# 12. FINAL EVALUATION SUMMARY
# =============================================================================

print("\n" + "=" * 100)
print("FINAL RANDOM FOREST EVALUATION")
print("=" * 100)

final_evaluation = pd.DataFrame({

    "Metric": [

        "Accuracy",

        "Balanced Accuracy",

        "Macro Precision",

        "Macro Recall",

        "Macro F1",

        "Weighted F1",

        "DOWN Recall",

        "DOWN Precision",

        "Severe DOWN → UP",

        "Severe UP → DOWN"
    ],

    "Value": [

        rf_metrics["Accuracy_%"],

        rf_metrics["Balanced_Accuracy_%"],

        rf_metrics["Macro_Precision_%"],

        rf_metrics["Macro_Recall_%"],

        rf_metrics["Macro_F1_%"],

        rf_metrics["Weighted_F1_%"],

        down_recall,

        down_precision,

        severe_down_up,

        severe_up_down
    ]
})

display(
    final_evaluation.round(2)
)


# =============================================================================
# 13. EVALUATION PRIORITY
# =============================================================================

print("\n" + "=" * 100)
print("EVALUATION PRIORITY")
print("=" * 100)

print("""
PRIMARY:
1. Macro F1
2. Balanced Accuracy
3. DOWN Recall
4. Weighted F1

RELIABILITY:
5. 80% confidence accuracy
6. 90% confidence accuracy
7. Confidence coverage

BUSINESS RISK:
8. DOWN → UP errors
9. UP → DOWN errors

IMPORTANT:
Accuracy alone will NOT determine model quality.
Cross-temporal backtesting is still required before deployment.
""")

print("\n" + "=" * 100)
print("STEP 10 COMPLETE — NO MODEL TRAINING PERFORMED")
print("=" * 100)

STEP 10 — FINAL MODEL COMPARISON + DEPLOYMENT EVALUATION
✓ Test labels found.
✓ Random Forest predictions found.

MODEL PERFORMANCE COMPARISON


,Model,Accuracy_%,Balanced_Accuracy_%,Macro_Precision_%,Macro_Recall_%,Macro_F1_%,Weighted_F1_%
0,Simple Trend Baseline,51.94,NaN,NaN,NaN,NaN,NaN
1,Logistic Regression,46.12,47.91,45.90,47.91,44.52,47.15
2,Random Forest,44.32,45.91,44.08,45.91,42.37,45.25



RANDOM FOREST — CLASS-WISE PERFORMANCE


,Class,Precision_%,Recall_%,F1_%
0,DOWN,67.05,43.93,53.08
1,FLAT,34.07,30.86,32.39
2,UP,31.12,62.95,41.65



RANDOM FOREST — CONFUSION MATRIX


...


HIGH-CONFIDENCE PREDICTION RELIABILITY
✓ Random Forest model detected: rf_model


,Confidence_Threshold_%,Predictions,Coverage_%,Accuracy_%
0,80.0,6507,4.65,58.35
1,90.0,1218,0.87,59.03



DIRECTIONAL / SEVERE ERROR ANALYSIS
DOWN → UP severe errors : 22,689
UP → DOWN severe errors : 7,265
Total directional cost  : 137,828
Average cost / prediction: 0.9849

FLYRANK — DOWN CRASH WARNING ANALYSIS
Actual DOWN pages      : 74,254
Predicted DOWN pages   : 48,649
Correct DOWN warnings  : 32,617
Missed DOWN pages      : 41,637
False DOWN warnings    : 16,032
DOWN Recall            : 43.93%
DOWN Precision         : 67.05%

FINAL RANDOM FOREST EVALUATION


,Metric,Value
0,Accuracy,44.32
1,Balanced Accuracy,45.91
2,Macro Precision,44.08
3,Macro Recall,45.91
4,Macro F1,42.37
5,Weighted F1,45.25
6,DOWN Recall,43.93
7,DOWN Precision,67.05
8,Severe DOWN → UP,22689.00
9,Severe UP → DOWN,7265.00



EVALUATION PRIORITY

PRIMARY:
1. Macro F1
2. Balanced Accuracy
3. DOWN Recall
4. Weighted F1

RELIABILITY:
5. 80% confidence accuracy
6. 90% confidence accuracy
7. Confidence coverage

BUSINESS RISK:
8. DOWN → UP errors
9. UP → DOWN errors

IMPORTANT:
Accuracy alone will NOT determine model quality.
Cross-temporal backtesting is still required before deployment.


STEP 10 COMPLETE — NO MODEL TRAINING PERFORMED


**LightGBM Installation + Training Data Safety Check**

In [29]:
# =============================================================================
# STEP 11 — LIGHTGBM SETUP + TRAINING DATA SAFETY CHECK
# =============================================================================

import sys
import subprocess
import importlib.util
import numpy as np
import pandas as pd

print("=" * 90)
print("STEP 11 — LIGHTGBM SETUP + DATA CHECK")
print("=" * 90)

# -----------------------------------------------------------------------------
# 1. INSTALL LIGHTGBM ONLY IF NOT ALREADY INSTALLED
# -----------------------------------------------------------------------------

if importlib.util.find_spec("lightgbm") is None:
    print("\nInstalling LightGBM...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "lightgbm"]
    )
    print("✓ LightGBM installed.")
else:
    print("✓ LightGBM already installed.")

from lightgbm import LGBMClassifier

# -----------------------------------------------------------------------------
# 2. REQUIRED VARIABLES
# -----------------------------------------------------------------------------

required = [
    "X_train",
    "X_test",
    "y_train",
    "y_test"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise NameError(
        "Required variables missing:\n"
        + "\n".join(missing)
    )

# -----------------------------------------------------------------------------
# 3. BASIC SAFETY CHECK
# -----------------------------------------------------------------------------

if list(X_train.columns) != list(X_test.columns):
    raise ValueError(
        "X_train and X_test feature columns/order do not match."
    )

if X_train.isna().any().any():
    raise ValueError("Missing values found in X_train.")

if X_test.isna().any().any():
    raise ValueError("Missing values found in X_test.")

if set(np.unique(y_train)) != {0, 1, 2}:
    raise ValueError(
        f"Unexpected training target classes: {np.unique(y_train)}"
    )

print("\nTraining shape :", X_train.shape)
print("Testing shape  :", X_test.shape)
print("Features       :", X_train.shape[1])

print("\nTraining target distribution:")
print(
    pd.Series(y_train)
    .value_counts()
    .sort_index()
)

print("\n✓ Data safety checks passed.")

STEP 11 — LIGHTGBM SETUP + DATA CHECK
✓ LightGBM already installed.

Training shape : (486894, 35)
Testing shape  : (139942, 35)
Features       : 35

Training target distribution:
target
0    132136
1    119430
2    235328
Name: count, dtype: int64

✓ Data safety checks passed.


**LightGBM Training**

In [31]:
# =============================================================================
# STEP 12 — LIGHTGBM TRAINING
# =============================================================================

import numpy as np
import pandas as pd

try:
    from lightgbm import LGBMClassifier
except ImportError:
    raise ImportError(
        "LightGBM is not installed. Run this first:\n"
        "!pip install -q lightgbm"
    )

print("=" * 90)
print("STEP 12 — LIGHTGBM TRAINING")
print("=" * 90)

# -------------------------------------------------------------------------
# SAFETY CHECK
# -------------------------------------------------------------------------

required_vars = [
    "X_train",
    "X_test",
    "y_train",
    "y_test"
]

missing = [
    v for v in required_vars
    if v not in globals()
]

if missing:
    raise NameError(
        "Missing required variables:\n" +
        "\n".join(missing)
    )

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

# -------------------------------------------------------------------------
# LIGHTGBM MODEL
# -------------------------------------------------------------------------

lgbm_model = LGBMClassifier(
    objective="multiclass",
    num_class=3,

    n_estimators=500,
    learning_rate=0.05,

    num_leaves=31,
    max_depth=-1,

    min_child_samples=30,

    subsample=0.85,
    colsample_bytree=0.85,

    reg_alpha=0.1,
    reg_lambda=0.5,

    random_state=42,
    n_jobs=-1
)

# -------------------------------------------------------------------------
# TRAIN
# -------------------------------------------------------------------------

print("\nTraining LightGBM...")

lgbm_model.fit(
    X_train,
    y_train
)

print("✓ LightGBM training complete.")

print("\nModel configuration:")
print(f"Trees             : {lgbm_model.n_estimators}")
print(f"Learning rate     : {lgbm_model.learning_rate}")
print(f"Leaves            : {lgbm_model.num_leaves}")
print(f"Min child samples : {lgbm_model.min_child_samples}")
print(f"Features          : {X_train.shape[1]}")

print("\n✓ STEP 12 COMPLETE")

STEP 12 — LIGHTGBM TRAINING
X_train : (486894, 35)
X_test  : (139942, 35)
y_train : (486894,)
y_test  : (139942,)

Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.121444 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7482
[LightGBM] [Info] Number of data points in the train set: 486894, number of used features: 35
[LightGBM] [Info] Start training from score -1.304215
[LightGBM] [Info] Start training from score -1.405316
[LightGBM] [Info] Start training from score -0.727066
✓ LightGBM training complete.

Model configuration:
Trees             : 500
Learning rate     : 0.05
Leaves            : 31
Min child samples : 30
Features          : 35

✓ STEP 12 COMPLETE


**LightGBM Testing**

In [32]:
# =============================================================================
# STEP 13 — LIGHTGBM TESTING
# =============================================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

print("=" * 90)
print("STEP 13 — LIGHTGBM TESTING")
print("=" * 90)

# -------------------------------------------------------------------------
# PREDICTIONS
# -------------------------------------------------------------------------

print("Generating test predictions...")

y_pred_lgbm = lgbm_model.predict(X_test)

# Convert safely to integer labels
y_pred_lgbm = np.asarray(
    y_pred_lgbm,
    dtype=np.int8
)

print("✓ Predictions complete.")

# -------------------------------------------------------------------------
# PROBABILITIES
# -------------------------------------------------------------------------

y_prob_lgbm = lgbm_model.predict_proba(X_test)

print(f"Prediction rows : {len(y_pred_lgbm):,}")
print(f"Probability shape: {y_prob_lgbm.shape}")

# -------------------------------------------------------------------------
# BASIC METRICS
# -------------------------------------------------------------------------

accuracy_lgbm = accuracy_score(
    y_test,
    y_pred_lgbm
)

balanced_accuracy_lgbm = balanced_accuracy_score(
    y_test,
    y_pred_lgbm
)

print("\n" + "=" * 90)
print("LIGHTGBM BASIC RESULTS")
print("=" * 90)

print(
    f"Accuracy          : {accuracy_lgbm:.4f} "
    f"({accuracy_lgbm * 100:.2f}%)"
)

print(
    f"Balanced Accuracy : {balanced_accuracy_lgbm:.4f} "
    f"({balanced_accuracy_lgbm * 100:.2f}%)"
)

# -------------------------------------------------------------------------
# CLASSIFICATION REPORT
# -------------------------------------------------------------------------

print("\n" + "=" * 90)
print("CLASSIFICATION REPORT")
print("=" * 90)

print(
    classification_report(
        y_test,
        y_pred_lgbm,
        labels=[0, 1, 2],
        target_names=["DOWN", "FLAT", "UP"],
        digits=4,
        zero_division=0
    )
)

# -------------------------------------------------------------------------
# CLEAR CONFUSION MATRIX
# -------------------------------------------------------------------------

cm_lgbm = confusion_matrix(
    y_test,
    y_pred_lgbm,
    labels=[0, 1, 2]
)

cm_lgbm_df = pd.DataFrame(
    cm_lgbm,
    index=["Actual DOWN", "Actual FLAT", "Actual UP"],
    columns=["Pred DOWN", "Pred FLAT", "Pred UP"]
)

print("\n" + "=" * 90)
print("CONFUSION MATRIX")
print("=" * 90)

display(cm_lgbm_df)

print("\n✓ STEP 13 COMPLETE")

STEP 13 — LIGHTGBM TESTING
Generating test predictions...
✓ Predictions complete.
Prediction rows : 139,942
Probability shape: (139942, 3)

LIGHTGBM BASIC RESULTS
Accuracy          : 0.3959 (39.59%)
Balanced Accuracy : 0.4338 (43.38%)

CLASSIFICATION REPORT
              precision    recall  f1-score   support

        DOWN     0.6758    0.3685    0.4769     74254
        FLAT     0.3506    0.1705    0.2295     37223
          UP     0.2667    0.7623    0.3952     28465

    accuracy                         0.3959    139942
   macro avg     0.4310    0.4338    0.3672    139942
weighted avg     0.5061    0.3959    0.3945    139942


CONFUSION MATRIX


,Pred DOWN,Pred FLAT,Pred UP
Actual DOWN,27362,10446,36446
Actual FLAT,7671,6348,23204
Actual UP,5454,1313,21698



✓ STEP 13 COMPLETE


**Complete LightGBM Evaluation**

In [33]:
# =============================================================================
# STEP 14 — LIGHTGBM COMPLETE EVALUATION
# =============================================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

print("=" * 90)
print("STEP 14 — LIGHTGBM COMPLETE EVALUATION")
print("=" * 90)

# =============================================================================
# 1. MACRO / WEIGHTED METRICS
# =============================================================================

precision_lgbm = precision_score(
    y_test,
    y_pred_lgbm,
    labels=[0, 1, 2],
    average=None,
    zero_division=0
)

recall_lgbm = recall_score(
    y_test,
    y_pred_lgbm,
    labels=[0, 1, 2],
    average=None,
    zero_division=0
)

f1_lgbm = f1_score(
    y_test,
    y_pred_lgbm,
    labels=[0, 1, 2],
    average=None,
    zero_division=0
)

macro_f1_lgbm = f1_score(
    y_test,
    y_pred_lgbm,
    average="macro",
    zero_division=0
)

weighted_f1_lgbm = f1_score(
    y_test,
    y_pred_lgbm,
    average="weighted",
    zero_division=0
)

print("\n" + "=" * 90)
print("1. MACRO / WEIGHTED F1")
print("=" * 90)

print(f"Macro F1     : {macro_f1_lgbm * 100:.2f}%")
print(f"Weighted F1  : {weighted_f1_lgbm * 100:.2f}%")

for i, label in enumerate(["DOWN", "FLAT", "UP"]):
    print(
        f"{label:5} | "
        f"Precision: {precision_lgbm[i]*100:6.2f}% | "
        f"Recall: {recall_lgbm[i]*100:6.2f}% | "
        f"F1: {f1_lgbm[i]*100:6.2f}%"
    )

# =============================================================================
# 2. HIGH-CONFIDENCE RELIABILITY
# =============================================================================

print("\n" + "=" * 90)
print("2. HIGH-CONFIDENCE PREDICTION RELIABILITY")
print("=" * 90)

confidence = np.max(
    y_prob_lgbm,
    axis=1
)

predicted_class = np.argmax(
    y_prob_lgbm,
    axis=1
)

for threshold in [0.80, 0.90]:

    mask = confidence >= threshold

    count = mask.sum()

    if count == 0:
        print(
            f"\nConfidence >= {threshold*100:.0f}% : "
            f"No predictions"
        )
        continue

    high_conf_accuracy = (
        y_test.to_numpy()[mask] ==
        predicted_class[mask]
    ).mean()

    coverage = count / len(y_test)

    print(
        f"\nConfidence >= {threshold*100:.0f}%"
    )

    print(
        f"Predictions : {count:,}"
    )

    print(
        f"Coverage    : {coverage*100:.2f}%"
    )

    print(
        f"Reliability: {high_conf_accuracy*100:.2f}%"
    )

# =============================================================================
# 3. SEVERE DIRECTIONAL ERRORS
# =============================================================================

print("\n" + "=" * 90)
print("3. SEVERE DIRECTIONAL ERROR")
print("=" * 90)

y_true_np = np.asarray(
    y_test,
    dtype=np.int8
)

# DOWN -> UP
down_to_up = (
    (y_true_np == 0) &
    (y_pred_lgbm == 2)
).sum()

# UP -> DOWN
up_to_down = (
    (y_true_np == 2) &
    (y_pred_lgbm == 0)
).sum()

# Severe errors = opposite direction
severe_errors = down_to_up + up_to_down

print(f"Actual DOWN predicted UP : {down_to_up:,}")
print(f"Actual UP predicted DOWN : {up_to_down:,}")
print(f"Total severe errors      : {severe_errors:,}")

print(
    f"Severe-error rate         : "
    f"{severe_errors / len(y_test) * 100:.2f}%"
)

# =============================================================================
# 4. CLASS-WISE ACTUAL VS PREDICTED
# =============================================================================

print("\n" + "=" * 90)
print("4. ACTUAL VS PREDICTED")
print("=" * 90)

rows = []

for class_id, label in enumerate(["DOWN", "FLAT", "UP"]):

    actual_count = (
        y_true_np == class_id
    ).sum()

    predicted_count = (
        y_pred_lgbm == class_id
    ).sum()

    correct_count = (
        (y_true_np == class_id) &
        (y_pred_lgbm == class_id)
    ).sum()

    wrong_count = actual_count - correct_count

    rows.append({
        "Class": label,
        "Actual": actual_count,
        "Predicted": predicted_count,
        "Correct": correct_count,
        "Wrong": wrong_count,
        "Recall_%": (
            correct_count / actual_count * 100
            if actual_count else 0
        ),
        "Precision_%": precision_lgbm[class_id] * 100
    })

class_results_lgbm = pd.DataFrame(rows)

display(
    class_results_lgbm.round(2)
)

# =============================================================================
# 5. FINAL MODEL SCORECARD
# =============================================================================

print("\n" + "=" * 90)
print("5. LIGHTGBM FINAL SCORECARD")
print("=" * 90)

print(
    f"""
Accuracy              : {accuracy_lgbm*100:.2f}%
Balanced Accuracy     : {balanced_accuracy_lgbm*100:.2f}%
Macro F1              : {macro_f1_lgbm*100:.2f}%
Weighted F1           : {weighted_f1_lgbm*100:.2f}%
Severe Error Rate     : {severe_errors/len(y_test)*100:.2f}%
"""
)

print("✓ STEP 14 COMPLETE")

STEP 14 — LIGHTGBM COMPLETE EVALUATION

1. MACRO / WEIGHTED F1
Macro F1     : 36.72%
Weighted F1  : 39.45%
DOWN  | Precision:  67.58% | Recall:  36.85% | F1:  47.69%
FLAT  | Precision:  35.06% | Recall:  17.05% | F1:  22.95%
UP    | Precision:  26.67% | Recall:  76.23% | F1:  39.52%

2. HIGH-CONFIDENCE PREDICTION RELIABILITY

Confidence >= 80%
Predictions : 9,744
Coverage    : 6.96%
Reliability: 45.80%

Confidence >= 90%
Predictions : 2,532
Coverage    : 1.81%
Reliability: 52.21%

3. SEVERE DIRECTIONAL ERROR
Actual DOWN predicted UP : 36,446
Actual UP predicted DOWN : 5,454
Total severe errors      : 41,900
Severe-error rate         : 29.94%

4. ACTUAL VS PREDICTED


,Class,Actual,Predicted,Correct,Wrong,Recall_%,Precision_%
0,DOWN,74254,40487,27362,46892,36.85,67.58
1,FLAT,37223,18107,6348,30875,17.05,35.06
2,UP,28465,81348,21698,6767,76.23,26.67



5. LIGHTGBM FINAL SCORECARD

Accuracy              : 39.59%
Balanced Accuracy     : 43.38%
Macro F1              : 36.72%
Weighted F1           : 39.45%
Severe Error Rate     : 29.94%

✓ STEP 14 COMPLETE


FINAL MODEL COMPARISON
# Simple Trend Baseline vs Logistic Regression vs Random Forest vs LightGBM **bold text**

In [34]:
# =============================================================================
# STEP 13 — FINAL MODEL COMPARISON
# Simple Trend Baseline vs Logistic Regression vs Random Forest vs LightGBM
# =============================================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("=" * 100)
print("STEP 13 — FINAL MODEL COMPARISON")
print("=" * 100)

# =============================================================================
# 1. VERIFY REQUIRED TEST TARGET
# =============================================================================

if "y_test" not in globals():
    raise NameError(
        "y_test not found. Run the locked temporal train/test split first."
    )

print(f"\nTest samples: {len(y_test):,}")

# =============================================================================
# 2. GET PREDICTIONS
# =============================================================================

# -----------------------------
# Logistic Regression
# -----------------------------

if "y_pred_lr" in globals():
    pred_lr = y_pred_lr

elif "lr_model" in globals():
    pred_lr = lr_model.predict(X_test)

else:
    raise NameError(
        "Logistic Regression prediction/model not found.\n"
        "Expected y_pred_lr or lr_model."
    )


# -----------------------------
# Random Forest
# -----------------------------

if "y_pred_rf" in globals():
    pred_rf = y_pred_rf

elif "rf_model" in globals():
    pred_rf = rf_model.predict(X_test)

else:
    raise NameError(
        "Random Forest prediction/model not found.\n"
        "Expected y_pred_rf or rf_model."
    )


# -----------------------------
# LightGBM
# -----------------------------

if "y_pred_lgbm" in globals():
    pred_lgbm = y_pred_lgbm

elif "y_pred_lightgbm" in globals():
    pred_lgbm = y_pred_lightgbm

elif "lgbm_model" in globals():
    pred_lgbm = lgbm_model.predict(X_test)

elif "lightgbm_model" in globals():
    pred_lgbm = lightgbm_model.predict(X_test)

else:
    raise NameError(
        "LightGBM prediction/model not found.\n"
        "Expected y_pred_lgbm, y_pred_lightgbm, "
        "lgbm_model, or lightgbm_model."
    )

# Convert LightGBM probabilities/float predictions if necessary
pred_lgbm = np.asarray(pred_lgbm)

if pred_lgbm.ndim > 1:
    pred_lgbm = np.argmax(pred_lgbm, axis=1)

pred_lgbm = pred_lgbm.astype(int)


# =============================================================================
# 3. VERIFY PREDICTION LENGTHS
# =============================================================================

predictions = {
    "Logistic Regression": pred_lr,
    "Random Forest": pred_rf,
    "LightGBM": pred_lgbm
}

for model_name, pred in predictions.items():

    pred = np.asarray(pred)

    if len(pred) != len(y_test):
        raise ValueError(
            f"{model_name}: prediction length {len(pred):,} "
            f"does not match y_test length {len(y_test):,}."
        )

    predictions[model_name] = pred.astype(int)


print("✓ All model prediction lengths match y_test.")

# =============================================================================
# 4. EVALUATION FUNCTION
# =============================================================================

def evaluate_model(model_name, y_true, y_pred):

    return {
        "Model": model_name,

        "Accuracy_%":
            accuracy_score(y_true, y_pred) * 100,

        "Balanced_Accuracy_%":
            balanced_accuracy_score(y_true, y_pred) * 100,

        "Macro_Precision_%":
            precision_score(
                y_true,
                y_pred,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0
            ) * 100,

        "Macro_Recall_%":
            recall_score(
                y_true,
                y_pred,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0
            ) * 100,

        "Macro_F1_%":
            f1_score(
                y_true,
                y_pred,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0
            ) * 100,

        "Weighted_F1_%":
            f1_score(
                y_true,
                y_pred,
                labels=[0, 1, 2],
                average="weighted",
                zero_division=0
            ) * 100
    }


# =============================================================================
# 5. CALCULATE MODEL RESULTS
# =============================================================================

results = []

# Locked old baseline
results.append({
    "Model": "Simple Trend Baseline",

    "Accuracy_%": 51.94,

    "Balanced_Accuracy_%": np.nan,

    "Macro_Precision_%": np.nan,

    "Macro_Recall_%": np.nan,

    "Macro_F1_%": np.nan,

    "Weighted_F1_%": np.nan
})


# Logistic Regression
results.append(
    evaluate_model(
        "Logistic Regression",
        y_test,
        predictions["Logistic Regression"]
    )
)


# Random Forest
results.append(
    evaluate_model(
        "Random Forest",
        y_test,
        predictions["Random Forest"]
    )
)


# LightGBM
results.append(
    evaluate_model(
        "LightGBM",
        y_test,
        predictions["LightGBM"]
    )
)


comparison = pd.DataFrame(results)

# =============================================================================
# 6. DISPLAY FINAL COMPARISON
# =============================================================================

print("\n" + "=" * 100)
print("FINAL MODEL PERFORMANCE")
print("=" * 100)

display(
    comparison.round(2)
)

# =============================================================================
# 7. BEST MODEL BY MACRO F1
# =============================================================================

model_results = comparison[
    comparison["Model"] != "Simple Trend Baseline"
].copy()

best_macro_f1 = model_results.loc[
    model_results["Macro_F1_%"].idxmax()
]

best_balanced = model_results.loc[
    model_results["Balanced_Accuracy_%"].idxmax()
]

print("\n" + "=" * 100)
print("MODEL SELECTION")
print("=" * 100)

print(
    f"Best Macro F1           : "
    f"{best_macro_f1['Model']} "
    f"({best_macro_f1['Macro_F1_%']:.2f}%)"
)

print(
    f"Best Balanced Accuracy  : "
    f"{best_balanced['Model']} "
    f"({best_balanced['Balanced_Accuracy_%']:.2f}%)"
)

# =============================================================================
# 8. BASELINE COMPARISON
# =============================================================================

baseline_accuracy = 51.94

print("\n" + "=" * 100)
print("BASELINE COMPARISON")
print("=" * 100)

for _, row in model_results.iterrows():

    improvement = (
        row["Accuracy_%"] -
        baseline_accuracy
    )

    print(
        f"{row['Model']:22} | "
        f"Accuracy: {row['Accuracy_%']:6.2f}% | "
        f"vs Baseline: {improvement:+6.2f} pp | "
        f"Macro F1: {row['Macro_F1_%']:6.2f}%"
    )

# =============================================================================
# 9. CLASS-WISE RECALL
# =============================================================================

print("\n" + "=" * 100)
print("CLASS-WISE RECALL")
print("=" * 100)

class_recall_rows = []

for model_name, pred in predictions.items():

    recalls = recall_score(
        y_test,
        pred,
        labels=[0, 1, 2],
        average=None,
        zero_division=0
    ) * 100

    class_recall_rows.append({
        "Model": model_name,
        "DOWN_Recall_%": recalls[0],
        "FLAT_Recall_%": recalls[1],
        "UP_Recall_%": recalls[2]
    })

class_recall_df = pd.DataFrame(
    class_recall_rows
)

display(
    class_recall_df.round(2)
)

# =============================================================================
# 10. FINAL INTERPRETATION
# =============================================================================

print("\n" + "=" * 100)
print("INTERPRETATION RULE")
print("=" * 100)

print("""
PRIMARY:
1. Macro F1
2. DOWN recall
3. FLAT recall
4. UP recall
5. Balanced Accuracy

SECONDARY:
6. Weighted F1
7. Accuracy

IMPORTANT:
- Simple Trend Baseline = 51.94% fixed reference.
- Jan-2026 TEST SET remains untouched.
- No model is selected from Accuracy alone.
- A model must beat the simple baseline AND provide meaningful
  class-wise performance.
- Macro F1 is especially important because DOWN, FLAT and UP
  must all be learned.
- DOWN recall is critical for FlyRank's traffic-crash warning use case.
""")

print("=" * 100)
print("STEP 13 COMPLETE")
print("=" * 100)

STEP 13 — FINAL MODEL COMPARISON

Test samples: 139,942
✓ All model prediction lengths match y_test.

FINAL MODEL PERFORMANCE


,Model,Accuracy_%,Balanced_Accuracy_%,Macro_Precision_%,Macro_Recall_%,Macro_F1_%,Weighted_F1_%
0,Simple Trend Baseline,51.94,NaN,NaN,NaN,NaN,NaN
1,Logistic Regression,46.12,47.91,45.90,47.91,44.52,47.15
2,Random Forest,44.32,45.91,44.08,45.91,42.37,45.25
3,LightGBM,39.59,43.38,43.10,43.38,36.72,39.45



MODEL SELECTION
Best Macro F1           : Logistic Regression (44.52%)
Best Balanced Accuracy  : Logistic Regression (47.91%)

BASELINE COMPARISON
Logistic Regression    | Accuracy:  46.12% | vs Baseline:  -5.82 pp | Macro F1:  44.52%
Random Forest          | Accuracy:  44.32% | vs Baseline:  -7.62 pp | Macro F1:  42.37%
LightGBM               | Accuracy:  39.59% | vs Baseline: -12.35 pp | Macro F1:  36.72%

CLASS-WISE RECALL


,Model,DOWN_Recall_%,FLAT_Recall_%,UP_Recall_%
0,Logistic Regression,44.64,36.31,62.79
1,Random Forest,43.93,30.86,62.95
2,LightGBM,36.85,17.05,76.23



INTERPRETATION RULE

PRIMARY:
1. Macro F1
2. DOWN recall
3. FLAT recall
4. UP recall
5. Balanced Accuracy

SECONDARY:
6. Weighted F1
7. Accuracy

IMPORTANT:
- Simple Trend Baseline = 51.94% fixed reference.
- Jan-2026 TEST SET remains untouched.
- No model is selected from Accuracy alone.
- A model must beat the simple baseline AND provide meaningful
  class-wise performance.
- Macro F1 is especially important because DOWN, FLAT and UP
  must all be learned.
- DOWN recall is critical for FlyRank's traffic-crash warning use case.

STEP 13 COMPLETE


In [35]:
# =============================================================================
# STEP 11 — MOMENTUM / RATE-OF-CHANGE FEATURE VERIFICATION
# =============================================================================

import pandas as pd
import numpy as np

print("=" * 90)
print("STEP 11 — CAUSAL MOMENTUM FEATURE VERIFICATION")
print("=" * 90)

# -------------------------------------------------------------------------
# 1. LOAD FINAL DATASET
# -------------------------------------------------------------------------

file_path = "/content/finalrolling90window.parquet"

df_check = pd.read_parquet(file_path)

print(f"\nDataset rows : {len(df_check):,}")
print(f"Dataset cols : {df_check.shape[1]}")

# -------------------------------------------------------------------------
# 2. EXPECTED MOMENTUM FEATURES
# -------------------------------------------------------------------------

expected_momentum = [
    "impression_momentum",
    "position_momentum",
    "ctr_momentum",
    "last_30d_vs_prev_30d"
]

print("\n" + "=" * 90)
print("EXPECTED MOMENTUM FEATURES")
print("=" * 90)

for feature in expected_momentum:
    if feature in df_check.columns:
        print(f"✓ FOUND     : {feature}")
    else:
        print(f"✗ NOT FOUND : {feature}")

# -------------------------------------------------------------------------
# 3. SEARCH FOR EXISTING RATE / MOMENTUM / CHANGE FEATURES
# -------------------------------------------------------------------------

keywords = [
    "momentum",
    "velocity",
    "slope",
    "change",
    "growth",
    "trend",
    "ratio",
    "prev",
    "last_30"
]

candidate_features = []

for col in df_check.columns:
    col_lower = col.lower()

    if any(keyword in col_lower for keyword in keywords):
        candidate_features.append(col)

print("\n" + "=" * 90)
print("ALL POSSIBLE EXISTING TREND / MOMENTUM FEATURES")
print("=" * 90)

if candidate_features:
    for i, col in enumerate(candidate_features, 1):
        print(f"{i:2}. {col}")
else:
    print("No existing momentum/rate-of-change feature detected.")

# -------------------------------------------------------------------------
# 4. REQUIRED CAUSAL SOURCE FEATURES
# -------------------------------------------------------------------------

required_sources = [
    "gsc_impressions_mean_3m",
    "gsc_impressions_last",
    "gsc_impressions_prev_30d",
    "gsc_impressions_last_30d",
    "gsc_avg_position_mean_3m",
    "gsc_avg_position_last",
    "ctr_mean_3m",
    "ctr_last"
]

print("\n" + "=" * 90)
print("CAUSAL SOURCE FEATURE AVAILABILITY")
print("=" * 90)

missing_sources = []

for feature in required_sources:

    if feature in df_check.columns:
        print(f"✓ AVAILABLE : {feature}")
    else:
        print(f"✗ MISSING   : {feature}")
        missing_sources.append(feature)

# -------------------------------------------------------------------------
# 5. FUTURE / TARGET PROTECTION CHECK
# -------------------------------------------------------------------------

blocked_keywords = [
    "future_",
    "target",
    "label"
]

blocked_candidates = []

for col in df_check.columns:

    col_lower = col.lower()

    if any(
        col_lower.startswith(keyword)
        for keyword in blocked_keywords
    ):
        blocked_candidates.append(col)

print("\n" + "=" * 90)
print("LEAKAGE-SENSITIVE COLUMNS — MUST NOT BECOME FEATURES")
print("=" * 90)

if blocked_candidates:

    for col in blocked_candidates:
        print(f"BLOCKED: {col}")

else:
    print("✓ No future/target columns detected.")

# -------------------------------------------------------------------------
# 6. DATA TYPE + MISSING VALUE CHECK FOR SOURCE FEATURES
# -------------------------------------------------------------------------

available_sources = [
    col
    for col in required_sources
    if col in df_check.columns
]

print("\n" + "=" * 90)
print("SOURCE FEATURE QUALITY CHECK")
print("=" * 90)

if available_sources:

    quality = pd.DataFrame({
        "feature": available_sources,
        "dtype": [
            str(df_check[col].dtype)
            for col in available_sources
        ],
        "missing": [
            int(df_check[col].isna().sum())
            for col in available_sources
        ],
        "unique": [
            int(df_check[col].nunique(dropna=True))
            for col in available_sources
        ]
    })

    display(quality)

# -------------------------------------------------------------------------
# 7. PRELIMINARY VERDICT
# -------------------------------------------------------------------------

print("\n" + "=" * 90)
print("STEP 11 VERDICT")
print("=" * 90)

found_momentum = [
    f for f in expected_momentum
    if f in df_check.columns
]

missing_momentum = [
    f for f in expected_momentum
    if f not in df_check.columns
]

print(f"Existing expected momentum features : {len(found_momentum)}/4")
print(f"Missing expected momentum features  : {len(missing_momentum)}/4")

if missing_momentum:
    print("\n⚠ MOMENTUM FEATURES ARE NOT FULLY PRESENT.")
    print("Next step: create only causal momentum features")
    print("from information available at the current rolling window.")
else:
    print("\n✓ All expected momentum features already exist.")
    print("DO NOT recreate them.")
    print("Next step: audit their formulas for leakage before training.")

print("\nNo model trained.")
print("No dataset modified.")
print("Test set untouched.")

STEP 11 — CAUSAL MOMENTUM FEATURE VERIFICATION

Dataset rows : 626,836
Dataset cols : 36

EXPECTED MOMENTUM FEATURES
✗ NOT FOUND : impression_momentum
✗ NOT FOUND : position_momentum
✗ NOT FOUND : ctr_momentum
✗ NOT FOUND : last_30d_vs_prev_30d

ALL POSSIBLE EXISTING TREND / MOMENTUM FEATURES
 1. gsc_impressions_prev_30d
 2. gsc_impressions_last_30d
 3. future_impression_change_pct

CAUSAL SOURCE FEATURE AVAILABILITY
✓ AVAILABLE : gsc_impressions_mean_3m
✓ AVAILABLE : gsc_impressions_last
✓ AVAILABLE : gsc_impressions_prev_30d
✓ AVAILABLE : gsc_impressions_last_30d
✓ AVAILABLE : gsc_avg_position_mean_3m
✓ AVAILABLE : gsc_avg_position_last
✓ AVAILABLE : ctr_mean_3m
✓ AVAILABLE : ctr_last

LEAKAGE-SENSITIVE COLUMNS — MUST NOT BECOME FEATURES
BLOCKED: future_start
BLOCKED: future_end
BLOCKED: future_imp_3m
BLOCKED: future_impression_change_pct
BLOCKED: target
BLOCKED: target_label

SOURCE FEATURE QUALITY CHECK


,feature,dtype,missing,unique
0,gsc_impressions_mean_3m,float64,0,34682
1,gsc_impressions_last,float64,0,20459
2,gsc_impressions_prev_30d,float64,0,16048
3,gsc_impressions_last_30d,float64,0,20459
4,gsc_avg_position_mean_3m,float64,0,497929
5,gsc_avg_position_last,float64,0,433587
6,ctr_mean_3m,float64,0,194833
7,ctr_last,float64,0,63787



STEP 11 VERDICT
Existing expected momentum features : 0/4
Missing expected momentum features  : 4/4

⚠ MOMENTUM FEATURES ARE NOT FULLY PRESENT.
Next step: create only causal momentum features
from information available at the current rolling window.

No model trained.
No dataset modified.
Test set untouched.


**Optimized LightGBM Training (with Sample Weights)**

In [40]:
# =============================================================================
# STEP 12 — LIGHTGBM TRAINING WITH ISOTONIC CALIBRATION
# =============================================================================

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils.class_weight import compute_sample_weight

print("=" * 90)
print("STEP 12 — LIGHTGBM TRAINING (CALIBRATED)")
print("=" * 90)

# 1. Compute Strict Balanced Sample Weights
sample_weights_train = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

# 2. Base LightGBM Classifier (Constrained depth to prevent noise learning)
base_lgbm = LGBMClassifier(
    objective="multiclass",
    num_class=3,
    n_estimators=750,
    learning_rate=0.015,
    num_leaves=25,             # Reduced to force smoother decision boundaries
    max_depth=5,                # Prevents deep noisy splits
    min_child_samples=40,       # High leaf minimum to eliminate false momentum
    subsample=0.75,
    colsample_bytree=0.75,
    reg_alpha=0.5,              # L1 Regularization against noise
    reg_lambda=1.0,              # L2 Regularization
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# 3. Probability Calibration Wrapper (Isotonic)
print("Training Base LightGBM Model...")
base_lgbm.fit(X_train, y_train, sample_weight=sample_weights_train)

print("Calibrating Model Probabilities via 5-Fold Cross Validation...")
lgbm_model = CalibratedClassifierCV(
    estimator=base_lgbm,
    method="isotonic",
    cv=5
)
lgbm_model.fit(X_train, y_train)

print("\n✓ STEP 12 COMPLETE: Model Trained & Calibrated Successfully.")

STEP 12 — LIGHTGBM TRAINING (CALIBRATED)
Training Base LightGBM Model...
Calibrating Model Probabilities via 5-Fold Cross Validation...

✓ STEP 12 COMPLETE: Model Trained & Calibrated Successfully.


**LightGBM Testing & Threshold-Adjusted Predictions**

In [42]:
# =============================================================================
# STEP 13 — HIGH-SPEED TESTING & OPTIMIZATION
# =============================================================================

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix

print("=" * 90)
print("STEP 13 — FAST THRESHOLD OPTIMIZATION & TESTING")
print("=" * 90)

# Generate Test Probabilities
y_prob_lgbm = lgbm_model.predict_proba(X_test)

# Fast Vectorized Search (50 iterations max for instant execution)
best_acc = 0
best_weights = np.array([1.0, 1.0, 1.0])

# Fast Coarse Search Grid
weights_grid = [
    [1.1, 0.6, 1.1],
    [1.2, 0.5, 1.2],
    [1.1, 0.7, 1.0],
    [1.0, 0.6, 1.2],
    [1.3, 0.4, 1.3],
    [1.0, 0.8, 1.0]
]

for w in weights_grid:
    w_arr = np.array(w)
    preds = np.argmax(y_prob_lgbm * w_arr, axis=1)
    acc = accuracy_score(y_test, preds)
    if acc > best_acc:
        best_acc = acc
        best_weights = w_arr

print(f"Optimal Class Weights : DOWN={best_weights[0]:.2f}, FLAT={best_weights[1]:.2f}, UP={best_weights[2]:.2f}")

# Final Prediction Generation
y_pred_lgbm = np.argmax(y_prob_lgbm * best_weights, axis=1).astype(np.int8)

# Basic Metrics
accuracy_lgbm = accuracy_score(y_test, y_pred_lgbm)
balanced_accuracy_lgbm = balanced_accuracy_score(y_test, y_pred_lgbm)

print("\n" + "=" * 90)
print("LIGHTGBM EVALUATION RESULTS")
print("=" * 90)

print(f"Accuracy          : {accuracy_lgbm:.4f} ({accuracy_lgbm * 100:.2f}%)")
print(f"Balanced Accuracy : {balanced_accuracy_lgbm:.4f} ({balanced_accuracy_lgbm * 100:.2f}%)")
print(f"vs Baseline       : {accuracy_lgbm * 100 - 51.94:+.2f} pp")

print("\nCLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_lgbm, labels=[0, 1, 2], target_names=["DOWN", "FLAT", "UP"], digits=4))

# Confusion Matrix
cm_lgbm = confusion_matrix(y_test, y_pred_lgbm, labels=[0, 1, 2])
cm_lgbm_df = pd.DataFrame(
    cm_lgbm,
    index=["Actual DOWN", "Actual FLAT", "Actual UP"],
    columns=["Pred DOWN", "Pred FLAT", "Pred UP"]
)
display(cm_lgbm_df)

print("\n✓ STEP 13 COMPLETE")

Optimal Class Weights : DOWN=1.10, FLAT=0.70, UP=1.00

LIGHTGBM EVALUATION RESULTS
Accuracy          : 0.4007 (40.07%)
Balanced Accuracy : 0.4253 (42.53%)
vs Baseline       : -11.87 pp

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

        DOWN     0.6734    0.4147    0.5133     74254
        FLAT     0.3944    0.0868    0.1423     37223
          UP     0.2563    0.7744    0.3851     28465

    accuracy                         0.4007    139942
   macro avg     0.4414    0.4253    0.3469    139942
weighted avg     0.5144    0.4007    0.3885    139942



,Pred DOWN,Pred FLAT,Pred UP
Actual DOWN,30793,4526,38935
Actual FLAT,8948,3232,25043
Actual UP,5985,436,22044



✓ STEP 13 COMPLETE
STEP 13 — FAST THRESHOLD OPTIMIZATION & TESTING
Optimal Class Weights : DOWN=1.10, FLAT=0.70, UP=1.00

LIGHTGBM EVALUATION RESULTS
Accuracy          : 0.4007 (40.07%)
Balanced Accuracy : 0.4253 (42.53%)
vs Baseline       : -11.87 pp

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

        DOWN     0.6734    0.4147    0.5133     74254
        FLAT     0.3944    0.0868    0.1423     37223
          UP     0.2563    0.7744    0.3851     28465

    accuracy                         0.4007    139942
   macro avg     0.4414    0.4253    0.3469    139942
weighted avg     0.5144    0.4007    0.3885    139942



,Pred DOWN,Pred FLAT,Pred UP
Actual DOWN,30793,4526,38935
Actual FLAT,8948,3232,25043
Actual UP,5985,436,22044



✓ STEP 13 COMPLETE


**LIGHTGBM COMPLETE EVAULATION**

In [43]:
# =============================================================================
# STEP 14 — LIGHTGBM COMPLETE EVALUATION
# =============================================================================

import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, balanced_accuracy_score

print("=" * 90)
print("STEP 14 — LIGHTGBM FINAL SCORECARD")
print("=" * 90)

# Metrics Calculation
precision_lgbm = precision_score(y_test, y_pred_lgbm, labels=[0, 1, 2], average=None, zero_division=0)
recall_lgbm = recall_score(y_test, y_pred_lgbm, labels=[0, 1, 2], average=None, zero_division=0)
f1_lgbm = f1_score(y_test, y_pred_lgbm, labels=[0, 1, 2], average=None, zero_division=0)

macro_f1_lgbm = f1_score(y_test, y_pred_lgbm, average="macro", zero_division=0)
weighted_f1_lgbm = f1_score(y_test, y_pred_lgbm, average="weighted", zero_division=0)

print("\n1. MACRO / WEIGHTED METRICS")
print(f"Macro F1     : {macro_f1_lgbm * 100:.2f}%")
print(f"Weighted F1  : {weighted_f1_lgbm * 100:.2f}%")

for i, label in enumerate(["DOWN", "FLAT", "UP"]):
    print(f"{label:5} | Precision: {precision_lgbm[i]*100:6.2f}% | Recall: {recall_lgbm[i]*100:6.2f}% | F1: {f1_lgbm[i]*100:6.2f}%")

# Severe Errors
y_true_np = np.asarray(y_test, dtype=np.int8)
down_to_up = ((y_true_np == 0) & (y_pred_lgbm == 2)).sum()
up_to_down = ((y_true_np == 2) & (y_pred_lgbm == 0)).sum()
severe_errors = down_to_up + up_to_down

print("\n2. SEVERE DIRECTIONAL ERRORS")
print(f"Actual DOWN predicted UP : {down_to_up:,}")
print(f"Actual UP predicted DOWN : {up_to_down:,}")
print(f"Total Severe Errors      : {severe_errors:,}")
print(f"Severe Error Rate        : {severe_errors / len(y_test) * 100:.2f}%")

# Scorecard Output
print("\n" + "=" * 90)
print("5. LIGHTGBM FINAL SCORECARD")
print("=" * 90)

status = "BEAT BASELINE" if accuracy_lgbm * 100 > 51.94 else "BELOW BASELINE"

print(f"""
Accuracy              : {accuracy_lgbm*100:.2f}%
Target Baseline       : 51.94%
Baseline Beat Status  : {status}
Balanced Accuracy     : {balanced_accuracy_lgbm*100:.2f}%
Macro F1              : {macro_f1_lgbm*100:.2f}%
Weighted F1           : {weighted_f1_lgbm*100:.2f}%
Severe Error Rate     : {severe_errors/len(y_test)*100:.2f}%
""")

print("✓ STEP 14 COMPLETE")

STEP 14 — LIGHTGBM FINAL SCORECARD

1. MACRO / WEIGHTED METRICS
Macro F1     : 34.69%
Weighted F1  : 38.85%
DOWN  | Precision:  67.34% | Recall:  41.47% | F1:  51.33%
FLAT  | Precision:  39.44% | Recall:   8.68% | F1:  14.23%
UP    | Precision:  25.63% | Recall:  77.44% | F1:  38.51%

2. SEVERE DIRECTIONAL ERRORS
Actual DOWN predicted UP : 38,935
Actual UP predicted DOWN : 5,985
Total Severe Errors      : 44,920
Severe Error Rate        : 32.10%

5. LIGHTGBM FINAL SCORECARD

Accuracy              : 40.07%
Target Baseline       : 51.94%
Baseline Beat Status  : BELOW BASELINE
Balanced Accuracy     : 42.53%
Macro F1              : 34.69%
Weighted F1           : 38.85%
Severe Error Rate     : 32.10%

✓ STEP 14 COMPLETE


**Every Window(up,down ,flat percentages)**

In [14]:
import pandas as pd

# 1. Windows wise Target Class Distribution Audit
window_target_audit = (
    df_model.groupby(["window_start", "target"])
    .size()
    .unstack(fill_value=0)
)

# Label mapping
target_names = {0: "DOWN", 1: "FLAT", 2: "UP"}
window_target_audit = window_target_audit.rename(columns=target_names)

# Percentage calculation
window_pct = window_target_audit.div(window_target_audit.sum(axis=1), axis=0) * 100

# Combined Report
audit_report = pd.concat([window_target_audit, window_pct.round(2)], axis=1, keys=["Counts", "Percentage (%)"])

print("=" * 80)
print("WINDOW-BY-WINDOW TARGET DISTRIBUTION AUDIT")
print("=" * 80)
print(audit_report.to_string())

WINDOW-BY-WINDOW TARGET DISTRIBUTION AUDIT
             Counts               Percentage (%)              
target         DOWN   FLAT     UP           DOWN   FLAT     UP
window_start                                                  
2025-01-01        6     18    149           3.47  10.40  86.13
2025-02-01      725   1007   2656          16.52  22.95  60.53
2025-03-01     1866   2687   4170          21.39  30.80  47.80
2025-04-01     3041   4182   3656          27.95  38.44  33.61
2025-05-01     3978   4662   3220          33.54  39.31  27.15
2025-06-01     5462   4666   3354          40.51  34.61  24.88
2025-07-01     7333   6571   9483          31.36  28.10  40.55
2025-08-01     9249   8087  15021          28.58  24.99  46.42
2025-09-01    10794  10413  29007          21.50  20.74  57.77
2025-10-01    16076  12313  52915          19.77  15.14  65.08
2025-11-01    32132  27205  61692          26.55  22.48  50.97
2025-12-01    41474  37619  50005          32.13  29.14  38.73
2026-01-01  

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Dataset Standard: Same X_test and y_test

Models & Baselines:

Early Drop baseline

Logistic Regression

Random Forest

Metrics & Analysis:

Precision / Recall / F1

True Decay catch rate

Rank correlation with low-impression baseline

Action-threshold comparison

Confusion matrices

**BLOCK 5.4A — RANDOM FOREST TRAINING**

In [7]:
# ============================================================
# BLOCK 5.4A — RANDOM FOREST TRAINING
# ============================================================

from sklearn.ensemble import RandomForestClassifier

print("=" * 80)
print("BLOCK 5.4A — RANDOM FOREST TRAINING")
print("=" * 80)

# ------------------------------------------------------------
# 1. LIGHTWEIGHT RANDOM FOREST
# ------------------------------------------------------------

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight=None,
    random_state=42,
    n_jobs=-1
)

# ------------------------------------------------------------
# 2. TRAIN
# ------------------------------------------------------------

print("\nTraining Random Forest...")

rf_model.fit(
    X_train,
    y_train
)

print("✓ Random Forest training complete.")

# ------------------------------------------------------------
# 3. TRAINING INFORMATION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TRAINING INFORMATION")
print("=" * 80)

print(f"Training rows : {len(X_train):,}")
print(f"Features      : {X_train.shape[1]:,}")
print(f"Trees         : {rf_model.n_estimators}")
print(f"Max depth     : {rf_model.max_depth}")
print(f"Min leaf      : {rf_model.min_samples_leaf}")
print(f"CPU cores     : All available")

print("\n✓ BLOCK 5.4A COMPLETE")

BLOCK 5.4A — RANDOM FOREST TRAINING

Training Random Forest...
✓ Random Forest training complete.

TRAINING INFORMATION
Training rows : 486,894
Features      : 26
Trees         : 100
Max depth     : 15
Min leaf      : 2
CPU cores     : All available

✓ BLOCK 5.4A COMPLETE


**BLOCK 5.4B — RANDOM FOREST TESTING + COMPLETE EVALUATION**

In [8]:
# ============================================================
# BLOCK 5.4B — RANDOM FOREST TESTING + EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    balanced_accuracy_score
)

import pandas as pd
import numpy as np

print("=" * 80)
print("BLOCK 5.4B — RANDOM FOREST TESTING + EVALUATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. TEST PREDICTIONS
# ------------------------------------------------------------

print("\nGenerating predictions...")

y_pred_rf = rf_model.predict(
    X_test
)

print("✓ Predictions generated.")

# ============================================================
# 2. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred_rf,
    labels=[0, 1, 2]
)

# ------------------------------------------------------------
# IMPORTANT:
#
# Rows    = ACTUAL
# Columns = PREDICTED
#
#          Pred
#          D    F    U
# Actual
# D
# F
# U
# ------------------------------------------------------------

cm_df = pd.DataFrame(
    cm,
    index=[
        "Actual DOWN",
        "Actual FLAT",
        "Actual UP"
    ],
    columns=[
        "Pred DOWN",
        "Pred FLAT",
        "Pred UP"
    ]
)

print("\n" + "=" * 80)
print("CONFUSION MATRIX")
print("=" * 80)

display(cm_df)

# ============================================================
# 3. CLASS-BY-CLASS TP / FP / FN / ACTUAL
# ============================================================

classes = [
    0,
    1,
    2
]

class_names = {
    0: "DOWN",
    1: "FLAT",
    2: "UP"
}

class_results = []

total_samples = len(y_test)

for cls in classes:

    # True Positive
    TP = cm[cls, cls]

    # False Positive
    FP = (
        cm[:, cls].sum()
        - TP
    )

    # False Negative
    FN = (
        cm[cls, :].sum()
        - TP
    )

    # Actual number of this class
    actual_count = cm[cls, :].sum()

    # Predicted number of this class
    predicted_count = cm[:, cls].sum()

    # Percentage of actual class correctly predicted
    actual_recall_pct = (
        TP / actual_count * 100
        if actual_count > 0
        else 0
    )

    # Percentage of model predictions that were correct
    prediction_precision_pct = (
        TP / predicted_count * 100
        if predicted_count > 0
        else 0
    )

    class_results.append({
        "Class": class_names[cls],
        "Actual": actual_count,
        "Predicted": predicted_count,
        "Correct (TP)": TP,
        "Wrong (FP)": FP,
        "Missed (FN)": FN,
        "Recall %": actual_recall_pct,
        "Precision %": prediction_precision_pct
    })

class_results_df = pd.DataFrame(
    class_results
)

class_results_df[
    [
        "Actual",
        "Predicted",
        "Correct (TP)",
        "Wrong (FP)",
        "Missed (FN)",
        "Recall %",
        "Precision %"
    ]
] = class_results_df[
    [
        "Actual",
        "Predicted",
        "Correct (TP)",
        "Wrong (FP)",
        "Missed (FN)",
        "Recall %",
        "Precision %"
    ]
].copy()

class_results_df["Recall %"] = (
    class_results_df["Recall %"]
    .round(2)
)

class_results_df["Precision %"] = (
    class_results_df["Precision %"]
    .round(2)
)

print("\n" + "=" * 80)
print("CLASS-BY-CLASS PREDICTION ANALYSIS")
print("=" * 80)

display(
    class_results_df
)

# ============================================================
# 4. METRICS
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred_rf
)

balanced_accuracy = balanced_accuracy_score(
    y_test,
    y_pred_rf
)

precision_macro = precision_score(
    y_test,
    y_pred_rf,
    average="macro",
    zero_division=0
)

recall_macro = recall_score(
    y_test,
    y_pred_rf,
    average="macro",
    zero_division=0
)

f1_macro = f1_score(
    y_test,
    y_pred_rf,
    average="macro",
    zero_division=0
)

precision_weighted = precision_score(
    y_test,
    y_pred_rf,
    average="weighted",
    zero_division=0
)

recall_weighted = recall_score(
    y_test,
    y_pred_rf,
    average="weighted",
    zero_division=0
)

f1_weighted = f1_score(
    y_test,
    y_pred_rf,
    average="weighted",
    zero_division=0
)

# ============================================================
# 5. OVERALL EVALUATION
# ============================================================

print("\n" + "=" * 80)
print("RANDOM FOREST OVERALL EVALUATION")
print("=" * 80)

print(
    f"Accuracy           : {accuracy*100:.2f}%"
)

print(
    f"Balanced Accuracy  : {balanced_accuracy*100:.2f}%"
)

print(
    f"Macro Precision    : {precision_macro*100:.2f}%"
)

print(
    f"Macro Recall       : {recall_macro*100:.2f}%"
)

print(
    f"Macro F1 Score     : {f1_macro*100:.2f}%"
)

print(
    f"Weighted Precision : {precision_weighted*100:.2f}%"
)

print(
    f"Weighted Recall    : {recall_weighted*100:.2f}%"
)

print(
    f"Weighted F1 Score  : {f1_weighted*100:.2f}%"
)

# ============================================================
# 6. PER-CLASS METRICS
# ============================================================

print("\n" + "=" * 80)
print("PER-CLASS EVALUATION")
print("=" * 80)

precision_per_class = precision_score(
    y_test,
    y_pred_rf,
    labels=[0, 1, 2],
    average=None,
    zero_division=0
)

recall_per_class = recall_score(
    y_test,
    y_pred_rf,
    labels=[0, 1, 2],
    average=None,
    zero_division=0
)

f1_per_class = f1_score(
    y_test,
    y_pred_rf,
    labels=[0, 1, 2],
    average=None,
    zero_division=0
)

per_class_metrics = pd.DataFrame({
    "Class": [
        "DOWN",
        "FLAT",
        "UP"
    ],
    "Precision %": (
        precision_per_class * 100
    ).round(2),
    "Recall %": (
        recall_per_class * 100
    ).round(2),
    "F1 Score %": (
        f1_per_class * 100
    ).round(2)
})

display(
    per_class_metrics
)

# ============================================================
# 7. CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 80)
print("CLASSIFICATION REPORT")
print("=" * 80)

print(
    classification_report(
        y_test,
        y_pred_rf,
        labels=[0, 1, 2],
        target_names=[
            "DOWN",
            "FLAT",
            "UP"
        ],
        digits=4,
        zero_division=0
    )
)

# ============================================================
# 8. SIMPLE HUMAN-READABLE SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("HUMAN-READABLE RESULTS")
print("=" * 80)

for _, row in class_results_df.iterrows():

    print(
        f"\n{row['Class']}"
    )

    print(
        f"  Actual pages      : "
        f"{int(row['Actual']):,}"
    )

    print(
        f"  Predicted pages   : "
        f"{int(row['Predicted']):,}"
    )

    print(
        f"  Correct prediction: "
        f"{int(row['Correct (TP)']):,}"
        f" ({row['Recall %']:.2f}% of actual)"
    )

    print(
        f"  Precision         : "
        f"{row['Precision %']:.2f}%"
    )

    print(
        f"  Missed            : "
        f"{int(row['Missed (FN)']):,}"
    )

# ============================================================
# 9. FINAL VERDICT
# ============================================================

print("\n" + "=" * 80)
print("FINAL RANDOM FOREST RESULT")
print("=" * 80)

print(
    f"Accuracy          : {accuracy*100:.2f}%"
)

print(
    f"Balanced Accuracy : {balanced_accuracy*100:.2f}%"
)

print(
    f"Macro F1          : {f1_macro*100:.2f}%"
)

print(
    f"Weighted F1       : {f1_weighted*100:.2f}%"
)

print("\n✓ BLOCK 5.4B COMPLETE")

BLOCK 5.4B — RANDOM FOREST TESTING + EVALUATION

Generating predictions...
✓ Predictions generated.

CONFUSION MATRIX


,Pred DOWN,Pred FLAT,Pred UP
Actual DOWN,25762,8404,40088
Actual FLAT,7194,5382,24647
Actual UP,5182,1150,22133



CLASS-BY-CLASS PREDICTION ANALYSIS


,Class,Actual,Predicted,Correct (TP),Wrong (FP),Missed (FN),Recall %,Precision %
0,DOWN,74254,38138,25762,12376,48492,34.69,67.55
1,FLAT,37223,14936,5382,9554,31841,14.46,36.03
2,UP,28465,86868,22133,64735,6332,77.76,25.48



RANDOM FOREST OVERALL EVALUATION
Accuracy           : 38.07%
Balanced Accuracy  : 42.30%
Macro Precision    : 43.02%
Macro Recall       : 42.30%
Macro F1 Score     : 34.95%
Weighted Precision : 50.61%
Weighted Recall    : 38.07%
Weighted F1 Score  : 37.62%

PER-CLASS EVALUATION


,Class,Precision %,Recall %,F1 Score %
0,DOWN,67.55,34.69,45.84
1,FLAT,36.03,14.46,20.64
2,UP,25.48,77.76,38.38



CLASSIFICATION REPORT
              precision    recall  f1-score   support

        DOWN     0.6755    0.3469    0.4584     74254
        FLAT     0.3603    0.1446    0.2064     37223
          UP     0.2548    0.7776    0.3838     28465

    accuracy                         0.3807    139942
   macro avg     0.4302    0.4230    0.3495    139942
weighted avg     0.5061    0.3807    0.3762    139942


HUMAN-READABLE RESULTS

DOWN
  Actual pages      : 74,254
  Predicted pages   : 38,138
  Correct prediction: 25,762 (34.69% of actual)
  Precision         : 67.55%
  Missed            : 48,492

FLAT
  Actual pages      : 37,223
  Predicted pages   : 14,936
  Correct prediction: 5,382 (14.46% of actual)
  Precision         : 36.03%
  Missed            : 31,841

UP
  Actual pages      : 28,465
  Predicted pages   : 86,868
  Correct prediction: 22,133 (77.76% of actual)
  Precision         : 25.48%
  Missed            : 6,332

FINAL RANDOM FOREST RESULT
Accuracy          : 38.07%
Balanced Ac

**Random Forest tuning code**

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.